In [4]:
import pandas as pd
import numpy as np
import json
import os
import math
import calendar
from datetime import datetime, date



PLANNING_DATE = date(2026, 3, 20)   # ← change daily
INDENT_MONTH  = date(2026, 3,  1)   # ← change when month rolls over

# =============================================================
# SECTION 2 — PARAMETERS
# =============================================================

AVAILABLE_HOURS      = 22
MIN_RUN_HOURS        = 4
TARGET_DAYS_INV      = 3
MACHINE_STATE_FILE   = "machine_state.json"

MIN_DAILY_INDENT     = 150
MIN_INDENT_HOURS     = 4.0



SAFETY_DAYS          = 2     # extra days beyond buffer for safety cushion
FULL_TARGET_DAYS     = TARGET_DAYS_INV + SAFETY_DAYS   # = 5 days

OPD_SCENARIO_0 = float(FULL_TARGET_DAYS)   # 5.0 — critical: build to full target
OPD_SCENARIO_1 = float(FULL_TARGET_DAYS)   # 5.0 — mixed:    build to full target
OPD_SCENARIO_2 = float(FULL_TARGET_DAYS)   # 5.0 — below:    build to full target
OPD_SCENARIO_3 = 1.0                        # healthy: replace today's consumption only

W_URGENCY  = 0.55
W_CATEGORY = 0.25
W_INDENT   = 0.20

UTIL_TARGET_PCT = 98.0



book_path       = "C:/Users/Ex0164/Book1.xlsx"
matrix_path     = "C:/Users/Ex0164/Important codes/compatibility_matrix.xlsx"
changeover_path = "D:/Tushar/TOOL FIX/Unique_Machines_vt.xlsx"
output_path     = f"Smart_APS_V8_Plan_{PLANNING_DATE.strftime('%Y%m%d')}.xlsx"



def compute_working_days(ref_date):
    year  = ref_date.year
    month = ref_date.month
    total = calendar.monthrange(year, month)[1]
    sundays = sum(
        1 for d in range(1, total + 1)
        if date(year, month, d).weekday() == 6
    )
    return total - sundays, total, sundays

WORKING_DAYS, TOTAL_DAYS, SUNDAY_COUNT = compute_working_days(INDENT_MONTH)

print(f"\n{'='*65}")
print(f"  Smart APS V8  —  Tool-Aware Multi-Machine Scheduling")
print(f"  Planning date : {PLANNING_DATE}")
print(f"  Indent month  : {INDENT_MONTH.strftime('%B %Y')}")
print(f"  Working days  : {WORKING_DAYS}  ({TOTAL_DAYS} days − {SUNDAY_COUNT} Sundays)")
print(f"{'='*65}\n")

# =============================================================
# SECTION 5 — LOAD DATA
# =============================================================

print("Loading data...")
vt_parts_raw = pd.read_excel(book_path,       sheet_name="VT")
vt_matrix    = pd.read_excel(matrix_path,     sheet_name="VT_Matrix")
vt_co_raw         = pd.read_excel(changeover_path, sheet_name="VT_Changeover")
vt_machine_count_raw = pd.read_excel(matrix_path, sheet_name="VT_Machine_Part_Count")



def find_col(df, name, sheet):
    match = next((c for c in df.columns
                  if str(c).strip().lower() == name.lower()), None)
    if match is None:
        raise ValueError(
            f"Column '{name}' not found in sheet '{sheet}'.\n"
            f"Available columns: {list(df.columns)}"
        )
    return match

vt_col_part      = find_col(vt_parts_raw, "Part",       "VT")
vt_col_cycletime = find_col(vt_parts_raw, "Cycle time", "VT")
vt_col_cavity    = find_col(vt_parts_raw, "Cavity",     "VT")
vt_col_inventory = find_col(vt_parts_raw, "Inventory",  "VT")
vt_col_indent    = find_col(vt_parts_raw, "Indent",     "VT")
vt_col_tools     = find_col(vt_parts_raw, "Tools",      "VT")
vt_col_color     = find_col(vt_parts_raw, "Color",      "VT")

data = vt_parts_raw[
    vt_parts_raw[vt_col_part].notna() &
    (vt_parts_raw[vt_col_part].astype(str).str.strip() != "")
].copy()
data = data.drop_duplicates(subset=vt_col_part).copy()
data["Material"] = data[vt_col_part].astype(str).str.strip()

# Rate = 3600 / cycle_time  (cavities cancel — full cavity operation assumed)
data["_ct"] = pd.to_numeric(data[vt_col_cycletime], errors="coerce").replace(0, np.nan)
data["Rate"] = 3600 / data["_ct"]

data_valid     = data[data["Rate"].notna()].copy()
data_zero_rate = data[data["Rate"].isna()].copy()

print(f"  VT parts in sheet       : {len(data)}")
color_groups = {}
for p, c in part_color.items():
    color_groups.setdefault(c, []).append(p)
print(f"  Distinct colours        : {len(color_groups)}")
for col, parts in sorted(color_groups.items()):
    print(f"    {col:<20} → {len(parts)} part(s)")
print(f"  Parts with valid rate   : {len(data_valid)}")

# =============================================================
# SECTION 7 — LOOKUP DICTIONARIES
# =============================================================

def safe_dict(df, key_col, val_col, default=0.0):
    return {
        str(k).strip(): (default if pd.isna(v) else float(v))
        for k, v in zip(df[key_col], df[val_col])
        if pd.notna(k) and str(k).strip() != ""
    }

inventory      = safe_dict(data,       "Material", vt_col_inventory)
rate           = safe_dict(data_valid, "Material", "Rate")
indent_monthly = safe_dict(data,       "Material", vt_col_indent)

# Tools available per part (default 1 if missing)
tools_available = {}
part_color      = {}   # {part: color_string}
for _, row in data.iterrows():
    p = str(row["Material"]).strip()
    v = row[vt_col_tools]
    tools_available[p] = max(1, int(float(v))) if pd.notna(v) and str(v).strip() != "" else 1
    c = row[vt_col_color]
    part_color[p] = str(c).strip().upper() if pd.notna(c) and str(c).strip() != "" else "UNKNOWN"

indent_daily = {
    p: round(qty / WORKING_DAYS, 4)
    for p, qty in indent_monthly.items()
}

today_target_qty = {
    p: max(0.0, indent_daily.get(p, 0.0) - inventory.get(p, 0.0))
    for p in indent_monthly
}

# =============================================================
# SECTION 7A — SKIP RULES
# =============================================================

def should_skip(part):
    daily   = indent_daily.get(part, 0.0)
    monthly = indent_monthly.get(part, 0.0)
    r       = rate.get(part, 1.0)

    if daily <= MIN_DAILY_INDENT:
        return True, f"Daily indent {daily:.2f} ≤ {MIN_DAILY_INDENT} threshold"

    indent_hrs = monthly / r if r > 0 else 0.0
    if indent_hrs <= MIN_INDENT_HOURS:
        return True, f"Whole monthly indent = {indent_hrs:.2f}h ≤ {MIN_INDENT_HOURS}h threshold"

    return False, ""

# =============================================================
# SECTION 7B — CHANGEOVER TIMES
# =============================================================

def build_changeover_dict(co_df):
    co_dict = {}
    for _, row in co_df.iterrows():
        machine = str(row["Unique Machines"]).strip()
        minutes = row["Changeover time"]
        if pd.notna(minutes) and machine:
            co_dict[machine] = float(minutes) / 60.0
    return co_dict

vt_changeover          = build_changeover_dict(vt_co_raw)
DEFAULT_CHANGEOVER_HRS = 40 / 60.0



def build_machine_part_count(df):
    """
    Reads VT_Machine_Part_Count sheet.
    Columns: Machine | Part_Count
    Returns dict {machine_name: part_count}.
    """
    mpc = {}
    machine_col = next((c for c in df.columns
                        if str(c).strip().lower() == "machine"), None)
    count_col   = next((c for c in df.columns
                        if str(c).strip().lower() == "part_count"), None)
    if machine_col is None or count_col is None:
        print(f"  WARNING: VT_Machine_Part_Count sheet missing "
              f"'Machine' or 'Part_Count' column — using equal ranking")
        return {}
    for _, row in df.iterrows():
        m = str(row[machine_col]).strip()
        v = row[count_col]
        if m and pd.notna(v):
            try:
                mpc[m] = int(float(v))
            except (ValueError, TypeError):
                pass
    return mpc

machine_part_count = build_machine_part_count(vt_machine_count_raw)
max_part_count     = max(machine_part_count.values(), default=1) or 1

print(f"  Machine part counts loaded: {len(machine_part_count)} machines")
if machine_part_count:
    # Show most specialized machines
    specialized = sorted(machine_part_count.items(), key=lambda x: x[1])[:5]
    for m, c in specialized:
        print(f"    {m:<25} → {c} part(s)  "
              f"{'← MOST SPECIALIZED' if c <= 3 else ''}")

# =============================================================
# SECTION 7C — PART CATEGORY
# =============================================================

def build_category(df):
    cat      = {}
    part_col = next((c for c in df.columns if str(c).strip().lower() == "part"), None)
    cat_col  = next((c for c in df.columns if str(c).strip().lower() == "category"), None)
    if part_col is None:
        return cat
    for _, row in df.iterrows():
        part = row[part_col]
        if pd.isna(part) or str(part).strip() == "":
            continue
        val = "Stranger"
        if cat_col and pd.notna(row[cat_col]):
            val = str(row[cat_col]).strip().capitalize()
            if val not in ("Runner", "Repeater", "Stranger"):
                val = "Stranger"
        cat[str(part).strip()] = val
    return cat

part_category  = build_category(vt_parts_raw)
CATEGORY_SCORE = {"Runner": 100, "Repeater": 60, "Stranger": 20}

# =============================================================
# SECTION 8 — MACHINE STATE
# =============================================================

def load_machine_state():
    if os.path.exists(MACHINE_STATE_FILE):
        try:
            with open(MACHINE_STATE_FILE) as f:
                content = f.read().strip()
            if not content:
                print(f"  Machine state : file empty — treating as first run")
                os.remove(MACHINE_STATE_FILE)
                return {}
            state = json.loads(content)
            if not isinstance(state, dict):
                print(f"  Machine state : file corrupt — treating as first run")
                os.remove(MACHINE_STATE_FILE)
                return {}
            print(f"  Machine state loaded  ({len(state)} machines with history)")
            return state
        except json.JSONDecodeError as e:
            print(f"  Machine state : JSON error ({e}) — treating as first run")
            os.remove(MACHINE_STATE_FILE)
            return {}
        except Exception as e:
            print(f"  Machine state : read error ({e}) — treating as first run")
            os.remove(MACHINE_STATE_FILE)
            return {}
    print(f"  Machine state : FIRST RUN — no changeover today")
    return {}

def save_machine_state(state):
    combined = {m: p for m, p in state.items() if p is not None}
    with open(MACHINE_STATE_FILE, "w") as f:
        json.dump(combined, f, indent=2)
    print(f"\n  Machine state saved → '{MACHINE_STATE_FILE}'")

machine_state = load_machine_state()

# =============================================================
# SECTION 9 — COMPATIBILITY MATRIX
# =============================================================

def build_compatibility(matrix):
    machines = matrix.columns[1:].tolist()
    compat   = {}
    for _, row in matrix.iterrows():
        part = row["Part"]
        for m in machines:
            if row[m] == 1:
                compat.setdefault(part, []).append(m)
    return compat, machines

vt_compat, vt_machines = build_compatibility(vt_matrix)

# =============================================================
# SECTION 10 — SCENARIO CLASSIFIER
# =============================================================

def classify_scenario(parts):
    """
    Classify the day based on inventory health.

    Thresholds:
      critical = inventory < 1 day           → must produce today
      below    = 1 day ≤ inv < FULL_TARGET_DAYS (5 days)
      healthy  = inventory ≥ FULL_TARGET_DAYS (5 days)

    S0: all active parts critical
    S1: some critical, some have stock
    S2: none critical but below 5-day target
    S3: all parts at or above 5-day target → produce 1 day only
    """
    coverage = []
    for p in parts:
        inv   = inventory.get(p, 0)
        daily = indent_daily.get(p, 0)
        skip, _ = should_skip(p)
        if skip or daily == 0:
            continue
        coverage.append(inv / daily)

    if not coverage:
        return 3, "SCENARIO 3 — No active parts today"

    n        = len(coverage)
    critical = sum(1 for c in coverage if c < 1)
    low      = sum(1 for c in coverage if c < FULL_TARGET_DAYS)

    if critical == n:
        return 0, (f"SCENARIO 0 — ALL {n} parts critical (inv < 1 day) "
                   f"| target = {FULL_TARGET_DAYS} days")
    elif critical > 0:
        return 1, (f"SCENARIO 1 — {critical}/{n} parts critical "
                   f"| {n-critical} have stock | target = {FULL_TARGET_DAYS} days")
    elif low > 0:
        return 2, (f"SCENARIO 2 — {low}/{n} parts below {FULL_TARGET_DAYS}-day target "
                   f"| {n-low} already healthy")
    else:
        return 3, (f"SCENARIO 3 — All {n} parts healthy "
                   f"(≥{FULL_TARGET_DAYS} days) | produce 1 day only")

# =============================================================
# SECTION 11 — OPD CAP BY SCENARIO
# =============================================================

def opd_cap(scenario_id):
    """
    Returns the production target as a multiple of daily_indent.
    S0/S1/S2 → 5.0 (build to full 5-day target: 3 buffer + 2 safety)
    S3       → 1.0 (inventory already healthy — replace 1 day only)
    """
    return {0: OPD_SCENARIO_0, 1: OPD_SCENARIO_1,
            2: OPD_SCENARIO_2, 3: OPD_SCENARIO_3}.get(scenario_id, OPD_SCENARIO_2)

# =============================================================
# SECTION 12 — PRIORITY SCORING
# =============================================================

def compute_priority_scores(active_parts):
    rows = []
    for p in active_parts:
        inv      = inventory.get(p, 0)
        daily    = indent_daily.get(p, 0)
        cat      = part_category.get(p, "Stranger")
        days_cov = inv / daily if daily > 0 else 999.0
        gap_days = min(1.0, max(0.0, TARGET_DAYS_INV - days_cov) / TARGET_DAYS_INV)
        rows.append({"part": p, "inv": inv, "daily": daily,
                     "days_cov": days_cov, "cat": cat, "urgency_raw": gap_days})

    if not rows:
        return {}, []

    max_daily = max(r["daily"] for r in rows) or 1.0
    scores, score_rows = {}, []

    for r in rows:
        p              = r["part"]
        urgency_score  = r["urgency_raw"] * 100
        category_score = CATEGORY_SCORE.get(r["cat"], 20)
        indent_score   = (r["daily"] / max_daily) * 100
        final_score    = (W_URGENCY * urgency_score +
                          W_CATEGORY * category_score +
                          W_INDENT   * indent_score)
        scores[p] = round(final_score, 2)
        score_rows.append({
            "Part":           p,
            "Category":       r["cat"],
            "Tools":          tools_available.get(p, 1),
            "Inventory_Now":  round(r["inv"], 0),
            "Daily_Indent":   round(r["daily"], 2),
            "Days_Coverage":  round(r["days_cov"], 2),
            "Urgency_Score":  round(urgency_score, 1),
            "Category_Score": category_score,
            "Indent_Score":   round(indent_score, 1),
            "Final_Score":    round(final_score, 2),
        })

    return scores, score_rows

# =============================================================
# SECTION 13 — MACHINE RANKER
# =============================================================

def rank_machines(part, machines_to_try, machine_hours,
                  machine_last_part, inv_days):
    """
    Ranks compatible machines for a part assignment.

    PRIMARY criterion — Part_Count (from VT_Machine_Part_Count sheet):
      The machine that can run the FEWEST parts is preferred first.
      Rationale: a machine running only 2 parts is far more constrained
      than one running 15. Assigning the part to the constrained machine
      first keeps flexible machines free for parts that have no other option.

      part_count_score = part_count / max_part_count  (0..1, lower = better)

    SECONDARY criteria (tie-break within same part_count):
      • Same part ran last → no changeover (strong bonus)
      • Lower current utilization → more hours available

    Runner lock: a Runner with inv ≤ 1 day must stay on its last machine.
    """
    category    = part_category.get(part, "Stranger")
    runner_lock = (category == "Runner" and inv_days <= 1.0)

    ranked = []
    for m in machines_to_try:
        used = machine_hours.get(m, 0)
        free = round(AVAILABLE_HOURS - used, 4)
        if free < MIN_RUN_HOURS:
            continue
        last = machine_last_part.get(m)
        if runner_lock and last != part:
            continue
        co_hrs = 0.0 if (last is None or last == part) else \
                 vt_changeover.get(m, DEFAULT_CHANGEOVER_HRS)
        effective_free = round(free - co_hrs, 4)
        if effective_free < MIN_RUN_HOURS:
            continue

        # PRIMARY: prefer machine with lowest part count
        # (most specialized / least flexible machine first)
        part_count      = machine_part_count.get(m, max_part_count)
        count_score     = part_count / max_part_count   # 0..1, lower = better

        # SECONDARY: changeover, colour match, utilization
        co_penalty      = (co_hrs / AVAILABLE_HOURS) * 0.3
        util_penalty    = (used  / AVAILABLE_HOURS) * 0.2
        same_part_bonus = -0.15 if (last == part) else 0.0

        # Same-colour bonus: prefer machines where last part
        # shares the same colour — avoids the 10-min purge.
        # Applies only when a changeover is needed (last != part).
        if last is not None and last != part:
            last_color = part_color.get(last, "UNKNOWN")
            new_color  = part_color.get(part,  "UNKNOWN")
            same_color = (last_color == new_color
                          and last_color != "UNKNOWN")
            color_bonus = -0.08 if same_color else 0.0
        else:
            color_bonus = 0.0

        # Combined cost — part_count_score dominates (weight 1.0)
        cost = count_score + co_penalty + util_penalty + same_part_bonus + color_bonus

        ranked.append((m, co_hrs, effective_free, cost))

    ranked.sort(key=lambda x: x[3])
    return ranked, runner_lock



def assign_part(part, scenario_id, machine_hours, machine_last_part,
                current_inventory, plan, already_planned,
                priority_scores):
    """
    Assigns a part using minimum-tools principle.
    Returns list of new plan rows (empty = not planned).
    """
    daily    = indent_daily.get(part, 0)
    monthly  = indent_monthly.get(part, 0)
    r_val    = rate.get(part, 1)
    inv_now  = current_inventory.get(part, 0)
    category = part_category.get(part, "Stranger")
    tools    = tools_available.get(part, 1)
    score    = priority_scores.get(part, 0)
    inv_days = inv_now / daily if daily > 0 else 999
    compatible = vt_compat.get(part, [])

    if not compatible:
        return []

   
    #
    cap_days        = opd_cap(scenario_id)            # 5.0 or 1.0
    target_inv      = cap_days * daily                # desired inventory level
    total_shortfall = max(0.0, target_inv - inv_now)  # gap to fill
    # Floor: always produce at least MIN_RUN_HOURS worth
    hrs_for_full    = total_shortfall / r_val if r_val > 0 else MIN_RUN_HOURS
    hrs_for_full    = max(MIN_RUN_HOURS, hrs_for_full)

    new_rows        = []
    produced_so_far = 0.0
    tools_used      = 0
    used_machines   = set()

    # ── PHASE 1: Primary machine — try to fit FULL indent on one machine
    ranked, runner_lock = rank_machines(
        part, compatible, machine_hours, machine_last_part, inv_days)

    if not ranked:
        return []

    m1, co1, eff1, _ = ranked[0]

    # Can machine 1 cover the full shortfall?
    # run1 = min(what machine can give, what we need for full indent)
    run1 = min(eff1, hrs_for_full)
    run1 = max(run1, MIN_RUN_HOURS)
    qty1 = round(run1 * r_val, 0)

    machine_hours[m1]       = round(machine_hours.get(m1, 0) + co1 + run1, 4)
    current_inventory[part] = round(current_inventory.get(part, 0) + qty1, 0)
    machine_last_part[m1]   = part
    produced_so_far        += qty1
    tools_used             += 1
    used_machines.add(m1)
    already_planned.add(part)

    # Did machine 1 cover the full indent?
    indent_met_on_primary = (produced_so_far >= total_shortfall - 0.5)

    new_rows.append({
        "Part":             part,
        "Category":         category,
        "Machine":          m1,
        "Run_Hours":        round(run1, 3),
        "Changeover_Hrs":   round(co1, 3),
        "Total_Hrs_Used":   round(co1 + run1, 3),
        "Rate_Per_Hour":    round(r_val, 2),
        "Production_Qty":   qty1,
        "Monthly_Indent":   round(monthly, 0),
        "Daily_Indent":     round(daily, 2),
        "Today_Target":     round(today_target_qty.get(part, 0), 0),
        "Changeover":       "No" if co1 == 0 else "Yes",
        "Type":             "Primary" + (" [ZERO-INV]" if inv_now == 0 else ""),
        "Role":             "Primary",
        "Tools_Available":  tools,
        "Tools_Used":       1,
        "Runner_Lock":      "YES" if runner_lock else "No",
        "Priority_Score":   score,
        "Phase":            1,
        "Indent_Met":       "YES" if indent_met_on_primary else "NO — shortfall remains",
        "Color":            part_color.get(part, "UNKNOWN"),
        "Stagger_Adjusted": "No",
    })

    if indent_met_on_primary:
        # Full indent covered on primary machine — minimum tools = 1
        # Skip Phase 2 entirely
        pass

    else:

        is_critical   = (inv_now == 0)
        tool_hard_cap = tools if is_critical else min(2, tools)

        while produced_so_far < (total_shortfall - 0.5) and tools_used < tool_hard_cap:
            shortfall_now = total_shortfall - produced_so_far
            hrs_needed    = max(MIN_RUN_HOURS,
                                shortfall_now / r_val if r_val > 0 else MIN_RUN_HOURS)

            remaining_machines = [m for m in compatible if m not in used_machines]
            ranked_next, _ = rank_machines(
                part, remaining_machines, machine_hours,
                machine_last_part, inv_days)

            if not ranked_next:
                break

            mx, cox, effx, _ = ranked_next[0]

            # Run only as much as needed to cover remaining shortfall
            # Never run more than the machine has available
            run_x = min(effx, hrs_needed)
            run_x = max(run_x, MIN_RUN_HOURS)
            qty_x = round(run_x * r_val, 0)

            machine_hours[mx]       = round(machine_hours.get(mx, 0) + cox + run_x, 4)
            current_inventory[part] = round(current_inventory.get(part, 0) + qty_x, 0)
            machine_last_part[mx]   = part
            produced_so_far        += qty_x
            tools_used             += 1
            used_machines.add(mx)

            indent_met_here = (produced_so_far >= total_shortfall - 0.5)

            new_rows.append({
                "Part":             part,
                "Category":         category,
                "Machine":          mx,
                "Run_Hours":        round(run_x, 3),
                "Changeover_Hrs":   round(cox, 3),
                "Total_Hrs_Used":   round(cox + run_x, 3),
                "Rate_Per_Hour":    round(r_val, 2),
                "Production_Qty":   qty_x,
                "Monthly_Indent":   round(monthly, 0),
                "Daily_Indent":     round(daily, 2),
                "Today_Target":     round(today_target_qty.get(part, 0), 0),
                "Changeover":       "No" if cox == 0 else "Yes",
                "Type":             "Tool-Expansion",
                "Role":             f"Tool-Expansion (tool {tools_used}/{tools})",
                "Tools_Available":  tools,
                "Tools_Used":       tools_used,
                "Runner_Lock":      "No",
                "Priority_Score":   score,
                "Phase":            2,
                "Indent_Met":       "YES" if indent_met_here else "NO — shortfall remains",
                "Stagger_Adjusted": "No",
            })

            crisis_tag = " [CRISIS — 3rd+ tool]" if tools_used >= 3 else ""
            print(f"      ↳ TOOL-EXP {part:26s} tool {tools_used}/{tool_hard_cap} → "
                  f"{mx:15s}  {run_x:.2f}h  qty={qty_x:.0f}  "
                  f"shortfall was {shortfall_now:.0f}  "
                  f"{'COVERED ✓' if indent_met_here else 'still short'}"
                  f"{crisis_tag}")

    # ── PHASE 3: Inventory build on same machines ─────────────
    # Extend existing run rows for this part up to OPD cap.
    # No new machines, no new tools.
    cap_days     = opd_cap(scenario_id)
    inv_after    = current_inventory.get(part, 0)
    cap_qty      = cap_days * daily
    headroom_qty = max(0.0, cap_qty - inv_after)

    if headroom_qty > 0:
        for row in new_rows:
            if headroom_qty <= 0:
                break
            m       = row["Machine"]
            free_m  = round(AVAILABLE_HOURS - machine_hours.get(m, 0), 4)
            if free_m < 0.05:
                continue
            extend_hrs = min(free_m, headroom_qty / r_val if r_val > 0 else 0)
            if extend_hrs < 0.05:
                continue
            extra_qty = round(extend_hrs * r_val, 0)

            row["Run_Hours"]      = round(float(row["Run_Hours"]) + extend_hrs, 3)
            row["Total_Hrs_Used"] = round(float(row["Changeover_Hrs"]) + float(row["Run_Hours"]), 3)
            row["Production_Qty"] = round(float(row["Production_Qty"]) + extra_qty, 0)
            row["Type"]           = str(row["Type"]) + "+InvBuild"

            machine_hours[m]        = round(machine_hours.get(m, 0) + extend_hrs, 4)
            current_inventory[part] = round(current_inventory.get(part, 0) + extra_qty, 0)
            headroom_qty           -= extra_qty

            print(f"      ↳ INV-BUILD {part:25s} on {m:15s}  "
                  f"+{extend_hrs:.2f}h  qty+={extra_qty:.0f}")

    # Update Tools_Used on all rows for this part
    for row in new_rows:
        row["Tools_Used"] = tools_used

    return new_rows

# =============================================================
# SECTION 15 — CSP TOOL-CHANGER  (unchanged from V7)
# =============================================================

def _fmt_h(h):
    try:
        total_min = int(round(float(h) * 60))
        return f"{total_min // 60:02d}:{total_min % 60:02d}"
    except Exception:
        return "??"

def _collect_co_events(plan, machines):
    events = []
    for m in machines:
        m_rows = [r for r in plan if r["Machine"] == m]
        if len(m_rows) < 2:
            continue
        cursor = 0.0
        for i, row in enumerate(m_rows):
            co_h  = float(row.get("Changeover_Hrs") or 0.0)
            run_h = float(row.get("Run_Hours")       or 0.0)
            if co_h > 0 and i > 0:
                events.append({
                    "machine":       m,
                    "part_before":   m_rows[i-1]["Part"],
                    "part_after":    row["Part"],
                    "co_duration":   co_h,
                    "natural_start": cursor,
                    "row_before":    m_rows[i-1],
                    "row_after":     row,
                    "actual_start":  None,
                    "wait_hrs":      0.0,
                })
            cursor += co_h + run_h
    return events

def _recompute_natural_start(ev, plan):
    m, target = ev["machine"], ev["row_after"]
    cursor = 0.0
    for r in [r for r in plan if r["Machine"] == m]:
        if r is target:
            break
        cursor += float(r.get("Changeover_Hrs") or 0) + float(r.get("Run_Hours") or 0)
    return cursor

def _machine_spare(m, plan):
    used = sum(
        float(r.get("Changeover_Hrs") or 0) + float(r.get("Run_Hours") or 0)
        for r in plan if r["Machine"] == m
    )
    return max(0.0, AVAILABLE_HOURS - used)

def _extend_row_before(ev, wait_hrs, plan):
    spare     = _machine_spare(ev["machine"], plan)
    extend_by = min(wait_hrs, spare)
    if extend_by <= 0:
        return 0.0, 0
    rb    = ev["row_before"]
    r_val = rate.get(rb["Part"], 1.0)
    extra = round(extend_by * r_val, 0)
    rb["Run_Hours"]      = round(float(rb.get("Run_Hours") or 0) + extend_by, 3)
    rb["Production_Qty"] = round(float(rb.get("Production_Qty") or 0) + extra, 0)
    rb["Total_Hrs_Used"] = round(
        float(rb.get("Changeover_Hrs") or 0) + float(rb["Run_Hours"]), 3)
    rb["Stagger_Adjusted"] = f"CSP: extended +{round(extend_by*60,1)}min to fill TC wait"
    return extend_by, extra



MIN_CO_GAP_HRS = 20 / 60.0   


def _finish_time_of_co(ev, plan):
    """
    Finish time = cumulative run hours of all plan rows
    on this machine UP TO (but not including) the CO row.
    This is when the machine will call for the tool changer.
    """
    m      = ev["machine"]
    target = ev["row_after"]   # the row AFTER the CO
    total  = 0.0
    for row in plan:
        if row["Machine"] != m:
            continue
        if row is target:
            break
        total += float(row.get("Run_Hours") or 0)
        total += float(row.get("Changeover_Hrs") or 0)
    return round(total, 4)


def _last_row_before_co(ev, plan):
    """Return the plan row immediately before this CO event."""
    return ev["row_before"]


def stagger_co_by_quantity(plan, machines, scenario_id, current_inventory):
    """
    Adjusts production quantities to stagger changeover finish times
    so no two machines call for the tool changer within MIN_CO_GAP_HRS.
    Modifies plan rows in-place. Returns count of adjustments made.
    """
    events = _collect_co_events(plan, machines)
    if len(events) < 2:
        return 0   # nothing to stagger

    adjustments = 0
    max_passes  = len(events) * 2   # safety limit on iterations

    for _ in range(max_passes):
        # Recompute finish times from live plan each pass
        for ev in events:
            ev["_ft"] = _finish_time_of_co(ev, plan)

        # Sort by finish_time
        events.sort(key=lambda e: e["_ft"])

        # Find first pair violating the required gap.
        # required_gap = duration of the earlier CO + 20-min post-finish buffer
        # e.g. M1 CO = 40 min → next CO cannot start until minute 60.
        conflict = None
        for i in range(len(events) - 1):
            ft_early   = events[i]["_ft"]
            ft_late    = events[i+1]["_ft"]
            co_dur_e   = events[i]["co_duration"]
            required   = co_dur_e + MIN_CO_GAP_HRS
            gap        = ft_late - ft_early
            if gap < required - 0.001:
                conflict = (events[i], events[i+1], gap, required)
                break

        if conflict is None:
            break   # all gaps satisfied

        ev_early, ev_late, gap, required_gap = conflict
        shortfall_hrs = required_gap - gap

        # ── Try PUSH: add qty to the later machine's part before CO ──
        row_late  = _last_row_before_co(ev_late, plan)
        p_late    = row_late["Part"]
        r_late    = rate.get(p_late, 1)
        m_late    = ev_late["machine"]
        daily_l   = indent_daily.get(p_late, 0)
        inv_l     = current_inventory.get(p_late, 0)
        cap_qty   = opd_cap(scenario_id) * daily_l
        produced_l = float(row_late.get("Production_Qty") or 0)
        headroom  = max(0.0, cap_qty - inv_l)

        push_hrs  = shortfall_hrs
        push_qty  = round(push_hrs * r_late, 0)

        # Check free hours on that machine
        used_m    = sum(
            float(r.get("Changeover_Hrs") or 0) + float(r.get("Run_Hours") or 0)
            for r in plan if r["Machine"] == m_late
        )
        free_m    = max(0.0, AVAILABLE_HOURS - used_m)

        can_push = (
            push_qty <= headroom
            and push_hrs <= free_m + 0.001
            and r_late > 0
        )

        if can_push:
            row_late["Run_Hours"]      = round(float(row_late.get("Run_Hours") or 0) + push_hrs, 3)
            row_late["Production_Qty"] = round(produced_l + push_qty, 0)
            row_late["Total_Hrs_Used"] = round(
                float(row_late.get("Changeover_Hrs") or 0) + float(row_late["Run_Hours"]), 3)
            row_late["Stagger_Adjusted"] = (
                f"CO-stagger PUSH +{round(push_hrs*60,1)}min "
                f"gap={round(gap*60,1)}min need={round(required_gap*60,0):.0f}min "
                f"({round(ev_early['co_duration']*60,0):.0f}min CO + "
                f"{round(MIN_CO_GAP_HRS*60,0):.0f}min buffer)")
            current_inventory[p_late] = round(
                current_inventory.get(p_late, 0) + push_qty, 0)
            adjustments += 1
            print(f"    [CO-STAGGER PUSH]  {p_late:28s} on {m_late:15s}  "
                  f"+{round(push_hrs*60,1)}min  qty+={push_qty:.0f}  "
                  f"gap={round(gap*60,1)}min  "
                  f"need={round(required_gap*60,0):.0f}min "
                  f"({round(ev_early['co_duration']*60,0):.0f}min CO + "
                  f"{round(MIN_CO_GAP_HRS*60,0):.0f}min buffer)")
            continue

        # ── Try PULL: reduce qty of the earlier machine's part ──
        row_early = _last_row_before_co(ev_early, plan)
        p_early   = row_early["Part"]
        r_early   = rate.get(p_early, 1)
        m_early   = ev_early["machine"]
        daily_e   = indent_daily.get(p_early, 0)
        inv_e     = current_inventory.get(p_early, 0)
        produced_e = float(row_early.get("Production_Qty") or 0)

        # Must not pull below daily_indent coverage
        min_qty_e  = max(0.0, daily_e - inv_e)
        max_pull_qty = max(0.0, produced_e - min_qty_e)
        pull_hrs   = shortfall_hrs
        pull_qty   = round(pull_hrs * r_early, 0)

        can_pull = (
            pull_qty <= max_pull_qty
            and r_early > 0
            and produced_e - pull_qty >= MIN_RUN_HOURS * r_early
        )

        if can_pull:
            row_early["Run_Hours"]      = round(
                float(row_early.get("Run_Hours") or 0) - pull_hrs, 3)
            row_early["Production_Qty"] = round(produced_e - pull_qty, 0)
            row_early["Total_Hrs_Used"] = round(
                float(row_early.get("Changeover_Hrs") or 0) + float(row_early["Run_Hours"]), 3)
            row_early["Stagger_Adjusted"] = (
                f"CO-stagger PULL -{round(pull_hrs*60,1)}min "
                f"gap={round(gap*60,1)}min need={round(required_gap*60,0):.0f}min "
                f"({round(ev_early['co_duration']*60,0):.0f}min CO + "
                f"{round(MIN_CO_GAP_HRS*60,0):.0f}min buffer)")
            current_inventory[p_early] = round(
                current_inventory.get(p_early, 0) - pull_qty, 0)
            adjustments += 1
            print(f"    [CO-STAGGER PULL]  {p_early:28s} on {m_early:15s}  "
                  f"-{round(pull_hrs*60,1)}min  qty-={pull_qty:.0f}  "
                  f"gap={round(gap*60,1)}min  "
                  f"need={round(required_gap*60,0):.0f}min "
                  f"({round(ev_early['co_duration']*60,0):.0f}min CO + "
                  f"{round(MIN_CO_GAP_HRS*60,0):.0f}min buffer)")
            continue

        # Cannot resolve this conflict — log and move on
        print(f"    [CO-STAGGER SKIP]  {ev_early['machine']} / {ev_late['machine']}  "
              f"gap={round(gap*60,1)}min  need={round(required_gap*60,0):.0f}min  "
              f"push blocked (OPD cap) + pull blocked (daily indent) — unresolvable")
        # Artificially mark resolved to avoid infinite loop
        ev_late["_ft"] = ev_early["_ft"] + MIN_CO_GAP_HRS
        break

    return adjustments


def stagger_changeovers_serial_queue(plan, machines):
    """
    STRICT SERIAL QUEUE — guaranteed zero overlap, always.
    ═══════════════════════════════════════════════════════
    Mechanism (not optimisation):
      The tool changer is a single resource. It processes
      one changeover at a time, in order. The next CO cannot
      physically start until the previous one is finished.
      This is enforced by construction:

        tool_changer_free_at tracks when the tool changer
        becomes available again.

        For each CO event:
          actual_start = max(natural_start, tool_changer_free_at)
          tool_changer_free_at = actual_start + co_duration

      Since tool_changer_free_at only ever moves forward,
      and actual_start >= tool_changer_free_at always,
      two COs cannot overlap by construction — no solver,
      no constraint, no chance of failure.

    Ordering (greedy — earliest natural_start first):
      Serves the machine that needs its changeover soonest.
      This minimises total machine waiting time across the floor.

    Wait filling:
      If a machine must wait (actual_start > natural_start),
      the previous part's run is extended to fill the gap.
      The machine keeps producing — zero idle time.

    Why not CSP:
      CSP with getSolutions() enumerates all permutations and
      hangs the system with 20+ events. getSolution() finds one
      valid solution but cannot guarantee quality. The greedy
      serial queue gives the same no-overlap guarantee with
      O(n log n) complexity — runs in microseconds for any n.
    """
    print(f"\n  Tool-Changer Serial Queue Scheduler")
    print(f"  Guarantee: strict serial — no two COs can overlap by construction")

    events = _collect_co_events(plan, machines)

    if not events:
        print(f"  No changeovers in plan — tool changer idle  ✓")
        return

    # Sort by natural_start ascending — earliest-needed CO first
    events.sort(key=lambda e: e["natural_start"])

    print(f"  {len(events)} CO events queued across "
          f"{len({e['machine'] for e in events})} machines")
    print()
    print(f"  {'#':<4} {'Machine':<18} {'Part Before':<22} {'Part After':<22} "
          f"{'Dur':>5} {'Natural':>8} {'Actual':>8} {'Wait':>7} {'Fill':>6}")
    print(f"  {'─'*4} {'─'*18} {'─'*22} {'─'*22} "
          f"{'─'*5} {'─'*8} {'─'*8} {'─'*7} {'─'*6}")

    tool_changer_free_at = 0.0
    total_extra_pcs      = 0
    total_wait_min       = 0.0

    for idx, ev in enumerate(events, 1):
        # Recompute natural_start from live plan
        # (previous extensions may have shifted it)
        natural_start = _recompute_natural_start(ev, plan)
        co_h          = ev["co_duration"]

        # CORE GUARANTEE: actual_start cannot be before tool changer is free
        actual_start = max(natural_start, tool_changer_free_at)
        wait_hrs     = round(actual_start - natural_start, 4)

        # Advance the tool changer cursor — next CO starts no earlier than this
        tool_changer_free_at = actual_start + co_h

        # Fill machine wait by extending the previous part's run
        extra_pcs = 0
        if wait_hrs > 0.001:
            _, extra_pcs = _extend_row_before(ev, wait_hrs, plan)
            total_extra_pcs += extra_pcs
            total_wait_min  += wait_hrs * 60

        ev["actual_start"] = actual_start
        ev["wait_hrs"]     = wait_hrs

        wait_str  = f"+{round(wait_hrs*60,1)}m" if wait_hrs > 0.001 else "none"
        fill_str  = f"+{extra_pcs:.0f}" if extra_pcs > 0 else "—"

        print(f"  {idx:<4} {ev['machine']:<18} {ev['part_before']:<22} "
              f"{ev['part_after']:<22} "
              f"{round(co_h*60,1):>4.0f}m "
              f"{_fmt_h(natural_start):>8} "
              f"{_fmt_h(actual_start):>8} "
              f"{wait_str:>7} "
              f"{fill_str:>6}")

    n_waited = sum(1 for e in events if e.get("wait_hrs", 0) > 0.001)
    print(f"\n  Queue complete. Tool changer free at: {_fmt_h(tool_changer_free_at)}")
    print(f"  Events that had to wait : {n_waited} / {len(events)}")
    print(f"  Total wait time filled  : {round(total_wait_min, 1)} min "
          f"(converted to production, zero idle)")
    print(f"  Extra pieces produced   : {total_extra_pcs:,.0f}")
    print(f"  Overlap guarantee       : ABSOLUTE — serial queue, structurally impossible to overlap")


def stagger_changeovers(plan, machines):
    """Alias — calls the strict serial queue scheduler."""
    stagger_changeovers_serial_queue(plan, machines)



def _add_part_to_machine(p, m, run_hrs, co_hrs, machine_hours,
                          machine_last_part, current_inventory,
                          already_planned, plan, priority_scores,
                          type_label, role_label):
    """Helper: add a plan row and update all tracking dicts."""
    r_val = rate.get(p, 1)
    qty   = round(run_hrs * r_val, 0)

    machine_hours[m]     = round(machine_hours.get(m, 0) + co_hrs + run_hrs, 4)
    current_inventory[p] = round(current_inventory.get(p, 0) + qty, 0)
    machine_last_part[m] = p
    already_planned.add(p)

    plan.append({
        "Part":             p,
        "Category":         part_category.get(p, "Stranger"),
        "Machine":          m,
        "Run_Hours":        round(run_hrs, 3),
        "Changeover_Hrs":   round(co_hrs, 3),
        "Total_Hrs_Used":   round(co_hrs + run_hrs, 3),
        "Rate_Per_Hour":    round(r_val, 2),
        "Production_Qty":   qty,
        "Monthly_Indent":   round(indent_monthly.get(p, 0), 0),
        "Daily_Indent":     round(indent_daily.get(p, 0), 2),
        "Today_Target":     round(today_target_qty.get(p, 0), 0),
        "Changeover":       "No" if co_hrs == 0 else "Yes",
        "Type":             type_label,
        "Role":             role_label,
        "Tools_Available":  tools_available.get(p, 1),
        "Tools_Used":       1,
        "Runner_Lock":      "No",
        "Priority_Score":   round(priority_scores.get(p, 0), 2),
        "Phase":            1,
        "Stagger_Adjusted": "No",
    })
    return qty


def _co_hrs_for(p, m, machine_last_part):
    """
    Changeover hours for adding part p to machine m.
    Same part   -> 0 hrs (mould already loaded).
    Same colour -> normal changeover time (no purge needed).
    Diff colour -> changeover time + 10 min colour purge.
    """
    last = machine_last_part.get(m)
    if last is None or last == p:
        return 0.0
    base_co    = vt_changeover.get(m, DEFAULT_CHANGEOVER_HRS)
    last_color = part_color.get(last, "UNKNOWN")
    new_color  = part_color.get(p,    "UNKNOWN")
    purge      = (COLOR_PURGE_HRS
                  if last_color != new_color
                  and last_color != "UNKNOWN"
                  and new_color  != "UNKNOWN"
                  else 0.0)
    return base_co + purge

def _run_hours_for(p, m, remaining, scenario_id, current_inventory):
    """
    Compute viable run hours for part p on machine m given remaining
    capacity. Returns (run_hrs, co_hrs) or (0, 0) if not viable.
    """
    co_hrs   = _co_hrs_for(p, m, machine_last_part_ref[m] if hasattr(_run_hours_for, '_dummy') else {})
    eff_free = round(remaining - co_hrs, 4)
    if eff_free < MIN_RUN_HOURS:
        return 0.0, 0.0
    daily_p  = indent_daily.get(p, 0)
    r_val    = rate.get(p, 1)
    inv_now  = current_inventory.get(p, 0)
    cap_qty  = opd_cap(scenario_id) * daily_p
    headroom = max(0.0, cap_qty - inv_now)
    if headroom <= 0:
        return 0.0, 0.0
    shortfall    = max(0.0, daily_p - inv_now)
    min_run      = max(MIN_RUN_HOURS, shortfall / r_val if r_val > 0 else MIN_RUN_HOURS)
    run_hrs      = min(eff_free, max(min_run, headroom / r_val if r_val > 0 else eff_free))
    run_hrs      = max(MIN_RUN_HOURS, min(run_hrs, eff_free))
    return run_hrs, co_hrs


def utilization_enforcer(plan, machine_hours, machine_last_part,
                          all_parts, already_planned,
                          current_inventory, scenario_id,
                          priority_scores):

    print(f"\n  22H UTILIZATION ENFORCER  (target ≥{UTIL_TARGET_PCT}%)")
    print(f"  Steps: 1=extend existing  2=unplanned parts  "
          f"3=re-run best planned  4=skipped parts")
    micro_idle_log = []

    # All skipped parts (for Step 4)
    all_skipped = [
        p for p in all_parts
        if should_skip(p)[0]
        and rate.get(p, 0) > 0
        and indent_monthly.get(p, 0) > 0
    ]

    # Process machines least-utilized first
    machines_by_util = sorted(vt_machines, key=lambda m: machine_hours.get(m, 0))

    for m in machines_by_util:
        remaining = round(AVAILABLE_HOURS - machine_hours.get(m, 0), 4)
        if remaining < 0.05:
            continue

        # ── STEP 1: Extend existing parts on this machine ────
        parts_on_machine = list({row["Part"] for row in plan if row["Machine"] == m})
        for p in sorted(parts_on_machine,
                        key=lambda x: priority_scores.get(x, 0), reverse=True):
            if remaining < 0.05:
                break
            daily_p  = indent_daily.get(p, 0)
            r_val    = rate.get(p, 1)
            inv_now  = current_inventory.get(p, 0)
            # Cap: in S3 (healthy) extend only up to 1×daily.
            # In S0/S1/S2 extend up to 3×daily buffer target.
            headroom = max(0.0, opd_cap(scenario_id) * daily_p - inv_now)
            ext_hrs  = min(remaining, headroom / r_val if r_val > 0 else 0)
            if ext_hrs < 0.05:
                continue
            extra_qty = round(ext_hrs * r_val, 0)
            for row in plan:
                if row["Part"] == p and row["Machine"] == m:
                    row["Run_Hours"]      = round(float(row.get("Run_Hours", 0)) + ext_hrs, 3)
                    row["Production_Qty"] = round(float(row.get("Production_Qty", 0)) + extra_qty, 0)
                    row["Total_Hrs_Used"] = round(
                        float(row.get("Changeover_Hrs", 0)) + float(row["Run_Hours"]), 3)
                    row["Type"] = str(row.get("Type", "Primary")) + "+Extended"
                    break
            machine_hours[m]     = round(machine_hours.get(m, 0) + ext_hrs, 4)
            current_inventory[p] = round(current_inventory.get(p, 0) + extra_qty, 0)
            remaining            = round(remaining - ext_hrs, 4)
            print(f"    [S1-EXTEND]  {p:28s} on {m:15s}  +{ext_hrs:.2f}h  qty+={extra_qty:.0f}")

        if remaining < MIN_RUN_HOURS:
            continue

       
        unplanned = [
            p for p in all_parts
            if p not in already_planned
            and m in vt_compat.get(p, [])
            and not should_skip(p)[0]
            and indent_monthly.get(p, 0) > 0
            and rate.get(p, 0) > 0
            # Only include if still below the scenario cap
            and current_inventory.get(p, 0) < opd_cap(scenario_id) * indent_daily.get(p, 0)
        ]

        last_on_m = machine_last_part.get(m)   # part currently loaded on m

        def _co_sort_key(p):
            needs_co  = 0 if (last_on_m is None or last_on_m == p) else 1
            inv_now   = current_inventory.get(p, 0)
            is_critical = 1 if inv_now == 0 else 0   # critical = zero inventory
            score     = priority_scores.get(p, 0)
            # Within tier 0 (no CO): critical first, then score desc
            # Within tier 1 (CO needed): critical first, then score desc
            return (needs_co, -is_critical, -score)

        unplanned.sort(key=_co_sort_key)

        for p in unplanned:
            if remaining < MIN_RUN_HOURS:
                break
            co_hrs  = _co_hrs_for(p, m, machine_last_part)
            eff_free = round(remaining - co_hrs, 4)
            if eff_free < MIN_RUN_HOURS:
                continue
            daily_p  = indent_daily.get(p, 0)
            r_val    = rate.get(p, 1)
            inv_now  = current_inventory.get(p, 0)
            cap_qty  = opd_cap(scenario_id) * daily_p
            headroom = max(0.0, cap_qty - inv_now)
            if headroom <= 0:
                continue
            shortfall = max(0.0, daily_p - inv_now)
            min_run   = max(MIN_RUN_HOURS, shortfall / r_val if r_val > 0 else MIN_RUN_HOURS)
            run_hrs   = min(eff_free, max(min_run, headroom / r_val if r_val > 0 else eff_free))
            run_hrs   = max(MIN_RUN_HOURS, min(run_hrs, eff_free))

            co_tag = "No CO — same part loaded" if co_hrs == 0 else "CO"
            qty = _add_part_to_machine(
                p, m, run_hrs, co_hrs,
                machine_hours, machine_last_part, current_inventory,
                already_planned, plan, priority_scores,
                "Filler-Unplanned", "Primary")
            remaining = round(remaining - co_hrs - run_hrs, 4)
            print(f"    [S2-UNPLAN]  {p:28s} → {m:15s}  {run_hrs:.2f}h  "
                  f"qty={qty:.0f}  [{co_tag}]")

        if remaining < MIN_RUN_HOURS:
            continue

        # ── STEP 3: Re-run best planned part (largest indent, lowest inv)
        # Only if the part has spare tools not yet used on this machine
        planned_parts_elsewhere = [
            p for p in already_planned
            if m in vt_compat.get(p, [])
            and m not in [row["Machine"] for row in plan if row["Part"] == p]
        ]
        # Filter: must have a spare tool available
        planned_parts_elsewhere = [
            p for p in planned_parts_elsewhere
            if tools_available.get(p, 1) > len({
                row["Machine"] for row in plan if row["Part"] == p
            })
        ]
        # Sort: no-CO parts first (last ran on this machine),
        # then critical (zero inv), then priority score desc
        planned_parts_elsewhere.sort(key=lambda p: (
            0 if (machine_last_part.get(m) is None or machine_last_part.get(m) == p) else 1,
            1 if current_inventory.get(p, 0) == 0 else 0,   # critical first (negate)
            -priority_scores.get(p, 0)
        ))

        for p in planned_parts_elsewhere:
            if remaining < MIN_RUN_HOURS:
                break
            co_hrs   = _co_hrs_for(p, m, machine_last_part)
            eff_free = round(remaining - co_hrs, 4)
            if eff_free < MIN_RUN_HOURS:
                continue
            daily_p  = indent_daily.get(p, 0)
            r_val    = rate.get(p, 1)
            inv_now  = current_inventory.get(p, 0)
            cap_qty  = opd_cap(scenario_id) * daily_p
            headroom = max(0.0, cap_qty - inv_now)
            if headroom <= 0:
                continue
            shortfall = max(0.0, daily_p - inv_now)
            min_run   = max(MIN_RUN_HOURS, shortfall / r_val if r_val > 0 else MIN_RUN_HOURS)
            run_hrs   = min(eff_free, max(min_run, headroom / r_val if r_val > 0 else eff_free))
            run_hrs   = max(MIN_RUN_HOURS, min(run_hrs, eff_free))

            # Don’t add to already_planned (part is already there)
            # Just append the extra run row directly
            r_val2  = rate.get(p, 1)
            qty     = round(run_hrs * r_val2, 0)
            machine_hours[m]     = round(machine_hours.get(m, 0) + co_hrs + run_hrs, 4)
            current_inventory[p] = round(current_inventory.get(p, 0) + qty, 0)
            machine_last_part[m] = p
            remaining            = round(remaining - co_hrs - run_hrs, 4)

            tools_used_now = len({row["Machine"] for row in plan if row["Part"] == p}) + 1
            plan.append({
                "Part":             p,
                "Category":         part_category.get(p, "Stranger"),
                "Machine":          m,
                "Run_Hours":        round(run_hrs, 3),
                "Changeover_Hrs":   round(co_hrs, 3),
                "Total_Hrs_Used":   round(co_hrs + run_hrs, 3),
                "Rate_Per_Hour":    round(r_val2, 2),
                "Production_Qty":   qty,
                "Monthly_Indent":   round(indent_monthly.get(p, 0), 0),
                "Daily_Indent":     round(daily_p, 2),
                "Today_Target":     round(today_target_qty.get(p, 0), 0),
                "Changeover":       "No" if co_hrs == 0 else "Yes",
                "Type":             "Re-run (spare tool)",
                "Role":             f"Tool-Expansion (tool {tools_used_now})",
                "Tools_Available":  tools_available.get(p, 1),
                "Tools_Used":       tools_used_now,
                "Runner_Lock":      "No",
                "Priority_Score":   round(priority_scores.get(p, 0), 2),
                "Phase":            3,
                "Stagger_Adjusted": "No",
            })
            print(f"    [S3-RERUN]   {p:28s} → {m:15s}  {run_hrs:.2f}h  "
                  f"qty={qty:.0f}  tool {tools_used_now}/{tools_available.get(p,1)}")
            break   # one re-run per machine per pass

        if remaining < MIN_RUN_HOURS:
            continue

        # ── STEP 4: Assign a skipped part (last resort) ───────
        # Among compatible skipped parts: largest indent, lowest inv
        skipped_candidates = [
            p for p in all_skipped
            if m in vt_compat.get(p, [])
            and rate.get(p, 0) > 0
        ]
        skipped_candidates.sort(key=lambda p: (
            0 if (machine_last_part.get(m) is None or machine_last_part.get(m) == p) else 1,
            1 if current_inventory.get(p, 0) == 0 else 0,
            -indent_daily.get(p, 0)
        ))

        for p in skipped_candidates:
            if remaining < MIN_RUN_HOURS:
                break
            co_hrs   = _co_hrs_for(p, m, machine_last_part)
            eff_free = round(remaining - co_hrs, 4)
            if eff_free < MIN_RUN_HOURS:
                continue
            r_val   = rate.get(p, 1)
            inv_now = current_inventory.get(p, 0)
            daily_p = indent_daily.get(p, 0)
            # For skipped parts use remaining hours directly (no OPD cap)
            run_hrs = min(eff_free, max(MIN_RUN_HOURS,
                          indent_monthly.get(p, 0) / r_val if r_val > 0 else eff_free))
            run_hrs = max(MIN_RUN_HOURS, min(run_hrs, eff_free))

            qty = _add_part_to_machine(
                p, m, run_hrs, co_hrs,
                machine_hours, machine_last_part, current_inventory,
                already_planned, plan, priority_scores,
                "Filler-Skipped (last resort)", "Primary")
            remaining = round(remaining - co_hrs - run_hrs, 4)
            skip_reason = should_skip(p)[1]
            print(f"    [S4-SKIPPED] {p:28s} → {m:15s}  {run_hrs:.2f}h  "
                  f"qty={qty:.0f}  [{skip_reason[:35]}]")
            break   # one skipped part per machine per pass

        # ── STEP 5: Log micro-idle ────────────────────────────
        final_remaining = round(AVAILABLE_HOURS - machine_hours.get(m, 0), 4)
        if final_remaining >= 0.25:
            util_final = round((1 - final_remaining / AVAILABLE_HOURS) * 100, 1)
            if util_final < UTIL_TARGET_PCT:
                micro_idle_log.append({
                    "Machine":         m,
                    "Idle_Hrs":        round(final_remaining, 3),
                    "Utilization_Pct": util_final,
                    "Note":            ("No unplanned/skipped compatible part available "
                                        "and all planned parts at OPD cap"),
                })
                print(f"    [⚠ IDLE]     {m:15s}  {final_remaining:.2f}h idle "
                      f"({util_final}%) — all options exhausted")

    return micro_idle_log

# =============================================================
# SECTION 17 — NEW OUTPUT: MULTI-MACHINE PARTS VIEW
# =============================================================

def build_multi_machine_view(plan):
    """
    Returns a DataFrame showing only parts that were assigned
    to more than one machine, with one row per machine assignment.
    Includes a summary row per part showing totals.
    """
    if not plan:
        return pd.DataFrame()

    # Group plan rows by part
    from collections import defaultdict
    part_rows = defaultdict(list)
    for row in plan:
        part_rows[row["Part"]].append(row)

    # Keep only parts with multiple machine assignments
    multi = {p: rows for p, rows in part_rows.items() if len(rows) > 1}

    if not multi:
        return pd.DataFrame()

    output_rows = []
    for part, rows in sorted(multi.items(),
                              key=lambda x: -sum(r["Production_Qty"] for r in x[1])):
        daily  = indent_daily.get(part, 0)
        total_qty = sum(float(r["Production_Qty"]) for r in rows)

        for row in rows:
            qty_this   = float(row["Production_Qty"])
            run_h      = float(row["Run_Hours"])
            output_rows.append({
                "Part":                  part,
                "Category":              part_category.get(part, "Stranger"),
                "Tools_Available":       tools_available.get(part, 1),
                "Machines_Used":         len(rows),
                "Machine":               row["Machine"],
                "Role":                  row.get("Role", "Primary"),
                "Run_Hours":             round(run_h, 2),
                "Changeover_Hrs":        round(float(row.get("Changeover_Hrs", 0)), 2),
                "Production_Qty":        round(qty_this, 0),
                "Daily_Indent":          round(daily, 2),
                "Total_Qty_All_Machines":round(total_qty, 0),
                "Type":                  row.get("Type", "—"),
            })

        # Summary row for this part
        output_rows.append({
            "Part":                  f"  ↳ TOTAL — {part}",
            "Category":              "—",
            "Tools_Available":       tools_available.get(part, 1),
            "Machines_Used":         len(rows),
            "Machine":               f"{len(rows)} machines",
            "Role":                  "TOTAL",
            "Run_Hours":             round(sum(float(r["Run_Hours"]) for r in rows), 2),
            "Changeover_Hrs":        round(sum(float(r.get("Changeover_Hrs",0)) for r in rows), 2),
            "Production_Qty":        round(total_qty, 0),
            "Daily_Indent":          round(daily, 2),
            "Total_Qty_All_Machines":round(total_qty, 0),
            "Type":                  "—",
        })
        # Blank spacer
        output_rows.append({k: "" for k in output_rows[-1].keys()})

    return pd.DataFrame(output_rows)

# =============================================================
# SECTION 18 — NEW OUTPUT: PRODUCTION VS INDENT VIEW
# =============================================================

def build_production_vs_indent(plan, all_parts):
    """
    Every planned part: one row showing total production across
    all machines vs daily indent, gap direction, and extra days stock.
    """
    if not plan:
        return pd.DataFrame()

    from collections import defaultdict
    part_qty     = defaultdict(float)
    part_machines = defaultdict(list)

    for row in plan:
        p = row["Part"]
        part_qty[p]      += float(row.get("Production_Qty", 0))
        part_machines[p].append(row["Machine"])

    rows = []
    for p in sorted(part_qty.keys()):
        daily    = indent_daily.get(p, 0)
        monthly  = indent_monthly.get(p, 0)
        inv_b    = inventory.get(p, 0)
        produced = round(part_qty[p], 0)
        inv_after = round(inv_b + produced, 0)
        gap      = round(produced - daily, 0)   # +ve = over, -ve = under
        gap_dir  = "OVER" if gap > 0 else ("UNDER" if gap < 0 else "MET")

        # Extra days stock = how many days the produced qty covers
        # beyond the daily indent
        extra_days = round(gap / daily, 2) if daily > 0 and gap > 0 else 0.0

        machines_str = ", ".join(dict.fromkeys(part_machines[p]))  # unique, ordered

        rows.append({
            "Part":                  p,
            "Category":              part_category.get(p, "Stranger"),
            "Tools_Available":       tools_available.get(p, 1),
            "Machines":              machines_str,
            "Machines_Count":        len(set(part_machines[p])),
            "Total_Qty_Produced":    produced,
            "Daily_Indent":          round(daily, 2),
            "Monthly_Indent":        round(monthly, 0),
            "Gap_vs_Daily":          gap,      # +ve = over, -ve = under
            "Gap_Direction":         gap_dir,
            "Extra_Days_Stock":      extra_days,
            "Inventory_Before":      round(inv_b, 0),
            "Inventory_After":       inv_after,
            "Days_Coverage_After":   round(inv_after / daily, 2) if daily > 0 else 0,
        })

    # Sort: UNDER first (most urgent), then MET, then OVER
    order_map = {"UNDER": 0, "MET": 1, "OVER": 2}
    df = pd.DataFrame(rows)
    if not df.empty:
        df["_sort"] = df["Gap_Direction"].map(order_map)
        df = df.sort_values(["_sort", "Gap_vs_Daily"]).drop(columns=["_sort"])
        df = df.reset_index(drop=True)
    return df

# =============================================================
# SECTION 19 — INDENT HORIZON TABLE
# =============================================================

def compute_indent_horizon(parts):
    rows = []
    for p in parts:
        inv     = inventory.get(p, 0.0)
        monthly = indent_monthly.get(p, 0.0)
        daily   = indent_daily.get(p, 0.0)
        target  = today_target_qty.get(p, 0.0)
        r       = rate.get(p, 1.0)
        skip, skip_reason = should_skip(p)
        indent_hrs = monthly / r if r > 0 else 0.0

        if inv == 0.0 and monthly > 0:
            status = "ZERO INV — FORCED"
        elif skip:
            status = "SKIPPED"
        elif target > 0:
            status = "PRODUCTION NEEDED"
        elif monthly == 0:
            status = "NO INDENT"
        else:
            status = "INV SUFFICIENT"

        rows.append({
            "Part":             p,
            "Tools":            tools_available.get(p, 1),
            "Monthly_Indent":   round(monthly, 0),
            "Indent_Hrs_Total": round(indent_hrs, 2),
            "Working_Days":     WORKING_DAYS,
            "Daily_Indent":     round(daily, 2),
            "Inventory_Now":    round(inv, 0),
            "Today_Target_Qty": round(target, 0),
            "Today_Target_Hrs": round(target / r if r > 0 else 0, 2),
            "Rate_Per_Hour":    round(r, 2),
            "Indent_Status":    status,
            "Skip_Reason":      skip_reason,
        })
    return pd.DataFrame(rows)


SPECIALIZED_MACHINE_THRESHOLD = 3   # ≤ this many parts → shown as specialized

def detect_specialized_machines(all_parts):
    """Audit helper — returns specialized machines for console display."""
    machine_parts = {}
    for m in vt_machines:
        compatible_parts = [
            p for p in all_parts
            if m in vt_compat.get(p, [])
            and not should_skip(p)[0]
            and indent_monthly.get(p, 0) > 0
        ]
        machine_parts[m] = compatible_parts

    specialized_machines = {
        m for m, mparts in machine_parts.items()
        if 0 < len(mparts) <= SPECIALIZED_MACHINE_THRESHOLD
    }
    spec_machine_parts = {m: machine_parts[m] for m in specialized_machines}
    return specialized_machines, spec_machine_parts, machine_parts


# =============================================================
# SECTION 21 — MAIN SCHEDULER
# =============================================================

def schedule(parts, label=""):

    print(f"\n{'─'*65}")
    print(f"  {label}  |  {len(parts)} parts  |  {len(vt_machines)} machines")
    print(f"{'─'*65}")

    scenario_id, scenario_desc = classify_scenario(parts)
    print(f"  {scenario_desc}")

    horizon_df = compute_indent_horizon(parts)

    
    cap = opd_cap(scenario_id)
    active_parts = [
        p for p in parts
        if not should_skip(p)[0]
        and indent_monthly.get(p, 0) > 0
        and inventory.get(p, 0) < cap * indent_daily.get(p, 0)
    ]

    priority_scores, score_rows = compute_priority_scores(active_parts)
    score_df = pd.DataFrame(score_rows) if score_rows else pd.DataFrame()

    machine_hours     = {m: 0.0 for m in vt_machines}
    machine_last_part = {m: machine_state.get(m) for m in vt_machines}
    current_inventory = inventory.copy()
    plan              = []
    already_planned   = set()
    not_planned       = []
    deferred          = []

    # ── Show specialized machine summary (audit only) ────────
    # The actual specialized-machine preference is built into
    # rank_machines() via Part_Count — no separate pass needed.
    specialized_machines, spec_machine_parts, machine_part_map =         detect_specialized_machines(list(parts))

    print(f"\n  SPECIALIZED MACHINES (≤{SPECIALIZED_MACHINE_THRESHOLD} parts, "
          f"ranked first by rank_machines):")
    if specialized_machines:
        for m in sorted(specialized_machines,
                        key=lambda m: machine_part_count.get(m, 99)):
            mparts  = spec_machine_parts.get(m, [])
            cnt     = machine_part_count.get(m, "?")
            print(f"    {m:<25} Part_Count={cnt:<4} "
                  f"parts: {', '.join(mparts)}")
    else:
        print(f"    None — all machines have >3 compatible parts")

    
    sorted_active = sorted(active_parts,
                           key=lambda p: priority_scores.get(p, 0), reverse=True)

    print(f"\n  PRIMARY SCHEDULING PASS  ({len(sorted_active)} active parts)")
    print(f"  {'Part':<30} {'Score':>6} {'Tools':>5} {'Machine(s)':<30} "
          f"{'Run':>5} {'Qty':>8}  Status")
    print(f"  {'─'*95}")

    for part in sorted_active:
        inv_now  = current_inventory.get(part, 0)
        daily    = indent_daily.get(part, 0)
        monthly  = indent_monthly.get(part, 0)
        score    = priority_scores.get(part, 0)
        tools    = tools_available.get(part, 1)
        category = part_category.get(part, "Stranger")

        if monthly == 0:
            deferred.append({"Part": part, "Category": category,
                             "Reason": "Monthly indent = 0"})
            print(f"  {part:<30} {score:>6.1f} {tools:>5}  {'—':<30}  DEFERRED (no indent)")
            continue

        if not vt_compat.get(part):
            not_planned.append({
                "Part": part, "Category": category, "Score": score,
                "Daily_Indent": round(daily, 2), "Inventory_Now": round(inv_now, 0),
                "Tools": tools, "Compatible_Machines": "NONE DEFINED",
                "Reason": "Not in VT_Matrix", "Action_Needed": "Add to VT_Matrix",
            })
            print(f"  {part:<30} {score:>6.1f} {tools:>5}  {'—':<30}  ✗ NOT IN MATRIX")
            continue

        new_rows = assign_part(
            part, scenario_id, machine_hours, machine_last_part,
            current_inventory, plan, already_planned, priority_scores)

        if new_rows:
            plan.extend(new_rows)
            machines_used = [r["Machine"] for r in new_rows]
            total_qty     = sum(float(r["Production_Qty"]) for r in new_rows)
            total_run     = sum(float(r["Run_Hours"]) for r in new_rows)
            machines_str  = ", ".join(machines_used)
            flag = " [MULTI-MACHINE]" if len(new_rows) > 1 else ""
            flag += " [ZERO-INV]" if inv_now == 0 else ""
            print(f"  {part:<30} {score:>6.1f} {tools:>5}  {machines_str:<30}  "
                  f"{total_run:>5.2f}  {total_qty:>8.0f}  ✓{flag}")
        else:
            inv_days = inv_now / daily if daily > 0 else 999
            free_map = {m: round(AVAILABLE_HOURS - machine_hours.get(m, 0), 2)
                        for m in vt_compat.get(part, []) if m in machine_hours}
            not_planned.append({
                "Part":                part,
                "Category":            category,
                "Score":               score,
                "Tools":               tools,
                "Daily_Indent":        round(daily, 2),
                "Inventory_Now":       round(inv_now, 0),
                "Inv_Days_Coverage":   round(inv_days, 2),
                "Today_Target":        round(today_target_qty.get(part, 0), 0),
                "Compatible_Machines": ", ".join(vt_compat.get(part, [])),
                "Machine_Free_Hrs":    str(free_map),
                "Reason":              "No compatible machine has capacity",
                "Action_Needed":       "Review matrix or add machines",
            })
            print(f"  {part:<30} {score:>6.1f} {tools:>5}  {'—':<30}  ✗ NO CAPACITY")

    # 22H Utilization Enforcer
    micro_idle = utilization_enforcer(
        plan, machine_hours, machine_last_part,
        list(parts), already_planned,
        current_inventory, scenario_id, priority_scores)

    print(f"\n  CO Quantity Stagger  (min gap = {MIN_CO_GAP_HRS*60:.0f} min)")
    n_adj = stagger_co_by_quantity(
        plan, vt_machines, scenario_id, current_inventory)
    if n_adj == 0:
        print(f"    No adjustments needed — all CO gaps already ≥ "
              f"{MIN_CO_GAP_HRS*60:.0f} min  ✓")
    else:
        print(f"    {n_adj} quantity adjustment(s) made to stagger COs")

    
    stagger_changeovers(plan, vt_machines)

    # ── Build output views ────────────────────────────────────
    multi_machine_df     = build_multi_machine_view(plan)
    prod_vs_indent_df    = build_production_vs_indent(plan, list(parts))

    # Daily indent status
    indent_status_rows = []
    for row in plan:
        p       = row["Part"]
        planned = float(row.get("Production_Qty") or 0)
        daily   = indent_daily.get(p, 0)
        r_val   = float(row.get("Rate_Per_Hour") or rate.get(p, 0))
        inv_b   = inventory.get(p, 0)
        meets   = planned >= daily
        indent_status_rows.append({
            "Part":               p,
            "Category":           part_category.get(p, "Stranger"),
            "Machine":            row.get("Machine", "—"),
            "Role":               row.get("Role", "Primary"),
            "Priority_Score":     round(row.get("Priority_Score", 0), 2),
            "Run_Hours":          round(float(row.get("Run_Hours") or 0), 2),
            "Planned_Qty":        round(planned, 0),
            "Daily_Indent":       round(daily, 2),
            "Gap_vs_Daily":       round(daily - planned, 0),
            "Meets_Daily_Indent": "YES ✓" if meets else "NO ✗",
            "Inventory_Before":   round(inv_b, 0),
            "Total_Available":    round(inv_b + planned, 0),
            "Covers_With_Inv":    "YES ✓" if (inv_b + planned) >= daily else "NO ✗",
        })
    indent_status_df = pd.DataFrame(indent_status_rows)
    if not indent_status_df.empty:
        indent_status_df = indent_status_df.sort_values(
            ["Meets_Daily_Indent", "Gap_vs_Daily"],
            ascending=[True, False]).reset_index(drop=True)

    # Inventory health
    inv_rows = []
    for p in parts:
        inv_b    = inventory.get(p, 0)
        daily    = indent_daily.get(p, 0)
        produced = sum(float(r["Production_Qty"]) for r in plan if r["Part"] == p)
        inv_after = inv_b + produced
        days_cov  = inv_after / daily if daily > 0 else 0
        inv_rows.append({
            "Part":            p,
            "Tools":           tools_available.get(p, 1),
            "Rate_Per_Hour":   round(rate.get(p, 0), 2),
            "Monthly_Indent":  round(indent_monthly.get(p, 0), 0),
            "Daily_Indent":    round(daily, 2),
            "Inv_Before":      round(inv_b, 0),
            "Produced_Today":  round(produced, 0),
            "Inv_After_Today": round(inv_after, 0),
            "Days_Coverage":   round(days_cov, 2),
            "Status":          ("OK"       if days_cov >= TARGET_DAYS_INV
                                else "LOW"  if days_cov >= 1
                                else "CRITICAL"),
        })

    # Machine utilization
    mach_rows = []
    for m in vt_machines:
        used      = machine_hours.get(m, 0)
        parts_run = list({r["Part"] for r in plan if r["Machine"] == m})
        co_count  = sum(1 for r in plan if r["Machine"] == m
                        and r.get("Changeover") == "Yes")
        util_pct  = round(used / AVAILABLE_HOURS * 100, 1)
        mach_rows.append({
            "Machine":              m,
            "Used_Hours":           round(used, 2),
            "Unused_Hours":         round(AVAILABLE_HOURS - used, 2),
            "Utilization_%":        util_pct,
            "Status":               ("FULL"     if used >= AVAILABLE_HOURS - 0.3 else
                                     "GOOD"     if used >= AVAILABLE_HOURS * 0.98 else
                                     "PARTIAL"  if used >= AVAILABLE_HOURS * 0.85 else
                                     "UNDERUSED"),
            "Parts_Planned":        len(parts_run),
            "Changeovers":          co_count,
            "Last_Part_Run":        machine_last_part.get(m) or "—",
            "All_Parts_On_Machine": ", ".join(parts_run) if parts_run else "— idle —",
        })

    micro_df = pd.DataFrame(micro_idle)     if micro_idle else pd.DataFrame()
    plan_df  = pd.DataFrame(plan)           if plan       else pd.DataFrame()
    def_df   = pd.DataFrame(deferred)       if deferred   else pd.DataFrame()
    not_df   = pd.DataFrame(not_planned)    if not_planned else pd.DataFrame()
    mach_df  = pd.DataFrame(mach_rows)
    inv_df   = pd.DataFrame(inv_rows)

    if not plan_df.empty:
        plan_df.insert(0, "Planning_Date", str(PLANNING_DATE))
        plan_df.insert(1, "Scenario",      scenario_desc)
        plan_df.insert(2, "Working_Days",  WORKING_DAYS)

    # Summary
    multi_count = len(multi_machine_df[
        multi_machine_df.get("Role", pd.Series()) == "TOTAL"
    ]) if not multi_machine_df.empty else 0

    print(f"\n  {'='*65}")
    print(f"  SCHEDULE SUMMARY — {scenario_desc}")
    print(f"    Parts planned          : {len(already_planned)}")
    print(f"    Parts on multi-machine : {multi_count}")
    print(f"    Not planned            : {len(not_planned)}")
    print(f"    Deferred               : {len(deferred)}")
    if not mach_df.empty:
        print(f"    Avg utilization        : {mach_df['Utilization_%'].mean():.1f}%")
        print(f"    UNDERUSED machines     : "
              f"{(mach_df['Status']=='UNDERUSED').sum()}")
    print(f"  {'='*65}")

    return (plan_df, def_df, not_df, mach_df, inv_df,
            machine_last_part, horizon_df, indent_status_df,
            score_df, micro_df, multi_machine_df, prod_vs_indent_df)

# =============================================================
# SECTION 21 — PART AUDIT
# =============================================================

vt_parts = data_valid[
    data_valid["Material"].isin(vt_matrix["Part"])
]["Material"].unique()

all_vt_parts_raw = list(data["Material"].unique())
matrix_parts     = set(str(p).strip() for p in vt_matrix["Part"] if pd.notna(p))
zero_rate_set    = set(data_zero_rate["Material"].unique())

audit_rows = []
for part in all_vt_parts_raw:
    inv     = inventory.get(part, 0.0)
    r_val   = rate.get(part, None)
    monthly = indent_monthly.get(part, 0.0)
    daily   = indent_daily.get(part, 0.0)
    row_data = data[data["Material"] == part]
    ct_raw   = row_data[vt_col_cycletime].values[0] if len(row_data) else "—"
    cv_raw   = row_data[vt_col_cavity].values[0]    if len(row_data) else "—"
    tools    = tools_available.get(part, 1)

    if part in zero_rate_set or r_val is None:
        status, gate, reason = "ZERO/MISSING CYCLE TIME", "GATE 1", f"Cycle time={ct_raw}"
    elif part not in matrix_parts:
        status, gate, reason = "NOT IN VT_MATRIX", "GATE 2", "No compatible machine"
    elif monthly == 0:
        status, gate, reason = "ZERO/MISSING INDENT", "GATE 3", "Monthly indent = 0"
    elif should_skip(part)[0]:
        status, gate, reason = "SKIPPED (LOW INDENT / TRIVIAL RUN)", "GATE 4", should_skip(part)[1]
    elif daily > 0 and inv >= daily:
        status, gate, reason = "NOT REQUIRED — INV SUFFICIENT", "GATE 5", \
            f"Inventory ({inv:.0f}) ≥ daily_indent ({daily:.2f})"
    else:
        status, gate, reason = "ENTERS SCHEDULER", "—", "Passed all gates"

    audit_rows.append({
        "Part":          part,
        "Gate_Failed":   gate,
        "Reason":        reason,
        "Monthly_Indent":round(monthly, 0),
        "Daily_Indent":  round(daily, 2),
        "Inventory":     round(inv, 0),
        "Tools":         tools,
        "Rate_Per_Hour": round(r_val, 2) if r_val else "—",
        "Cycle_Time":    ct_raw,
        "Cavity":        cv_raw,
        "Status":        status,
    })

audit_df = pd.DataFrame(audit_rows)
gate_counts = audit_df["Status"].value_counts()
print(f"\n  Part audit ({len(all_vt_parts_raw)} total):")
for status, count in gate_counts.items():
    marker = "✓" if status == "ENTERS SCHEDULER" else "·"
    print(f"    {marker}  {status:<45}: {count:>4}")

# =============================================================
# SECTION 22 — RUN
# =============================================================

(vt_plan, vt_def, vt_not, vt_mach, vt_inv,
 vt_state, vt_horizon, vt_indent_status,
 vt_scores, vt_micro, vt_multi_machine,
 vt_prod_vs_indent) = schedule(vt_parts, "VT Machines")

save_machine_state(vt_state)

# =============================================================
# SECTION 23 — MACHINE-WISE PLAN
# =============================================================

def build_machine_wise_plan(plan_df):
    """
    Machine-wise quantity plan.

    We plan WHAT runs on each machine and HOW MUCH.
    The floor decides the sequence.

    One row per part per machine.
    Summary footer per machine showing totals.
    No start/end times — no sequence imposed.
    """
    if plan_df.empty:
        return pd.DataFrame()

    def sf(val, default=0.0):
        try:
            v = float(val)
            return v if not np.isnan(v) else default
        except (TypeError, ValueError):
            return default

    rows = []
    for m in vt_machines:
        machine_rows = plan_df[plan_df["Machine"] == m].copy()
        if machine_rows.empty:
            continue

        for _, pr in machine_rows.iterrows():
            p     = pr.get("Part", "—")
            co_h  = sf(pr.get("Changeover_Hrs", 0))
            run_h = sf(pr.get("Run_Hours", 0))
            r_val = sf(pr.get("Rate_Per_Hour", 0))
            rows.append({
                "Machine":            m,
                "Part":               p,
                "Category":           part_category.get(p, "Stranger"),
                "Role":               pr.get("Role", "Primary"),
                "Tools_Available":    pr.get("Tools_Available", 1),
                "Priority_Score":     round(sf(pr.get("Priority_Score", 0)), 2),
                "Rate_Per_Hour":      round(r_val, 2),
                "Run_Hours":          round(run_h, 2),
                "Changeover_Hrs":     round(co_h, 3),
                "Changeover_Needed":  pr.get("Changeover", "No") or "No",
                "Production_Qty":     round(sf(pr.get("Production_Qty", 0)), 0),
                "Daily_Indent":       round(sf(pr.get("Daily_Indent", 0)), 2),
                "Today_Target":       round(sf(pr.get("Today_Target", 0)), 0),
                "Monthly_Indent":     round(sf(pr.get("Monthly_Indent", 0)), 0),
                "Type":               pr.get("Type", "Primary") or "Primary",
                "Row_Type":           "Part",
            })

        # Summary footer per machine
        co_total  = machine_rows["Changeover_Hrs"].apply(lambda x: sf(x, 0)).sum()
        run_total = machine_rows["Run_Hours"].apply(lambda x: sf(x, 0)).sum()
        qty_total = machine_rows["Production_Qty"].apply(lambda x: sf(x, 0)).sum()
        co_count  = int(machine_rows["Changeover"].eq("Yes").sum())
        hrs_total = round(co_total + run_total, 2)

        rows.append({
            "Machine":            m,
            "Part":               f"TOTAL — {m}",
            "Category":           "—",
            "Role":               "—",
            "Tools_Available":    "—",
            "Priority_Score":     "—",
            "Rate_Per_Hour":      "—",
            "Run_Hours":          round(run_total, 2),
            "Changeover_Hrs":     round(co_total, 2),
            "Changeover_Needed":  f"{co_count} changeover(s)",
            "Production_Qty":     round(qty_total, 0),
            "Daily_Indent":       "—",
            "Today_Target":       "—",
            "Monthly_Indent":     "—",
            "Type":               (f"Total {hrs_total}h used / {AVAILABLE_HOURS}h  |  "
                                   f"Idle {round(AVAILABLE_HOURS - hrs_total, 2)}h  |  "
                                   f"Util {round(hrs_total / AVAILABLE_HOURS * 100, 1)}%"),
            "Row_Type":           "Summary",
        })
        rows.append({k: "" for k in rows[-1].keys()})

    return pd.DataFrame(rows)


def build_co_queue(plan, machines):
    """
    Tool-Changer Queue sheet.

    Shows the safe order in which changeovers should be called
    to the tool-change team. This guarantees no two COs happen
    at the same time. The floor uses this as a reference —
    not a rigid time schedule.

    Columns:
      Queue_Position | Machine | Part_Before | Part_After |
      CO_Duration_Min | Note
    """
    events = _collect_co_events(plan, machines)
    if not events:
        return pd.DataFrame()

    # Sort by natural_start — same greedy order used in the scheduler
    events.sort(key=lambda e: e["natural_start"])

    rows = []
    tool_changer_free_at = 0.0

    for pos, ev in enumerate(events, 1):
        natural_start = _recompute_natural_start(ev, plan)
        co_h          = ev["co_duration"]
        actual_start  = max(natural_start, tool_changer_free_at)
        wait_min      = round((actual_start - natural_start) * 60, 1)
        tool_changer_free_at = actual_start + co_h

        note = ""
        if wait_min > 0:
            note = (f"Machine may need to wait ~{wait_min}min for tool changer. "
                    f"Continue running previous part until team arrives.")

        rows.append({
            "Queue_Position":    pos,
            "Machine":           ev["machine"],
            "Part_Before":       ev["part_before"],
            "Part_After":        ev["part_after"],
            "CO_Duration_Min":   round(co_h * 60, 1),
            "Note":              note if note else "Tool changer available immediately",
        })

    return pd.DataFrame(rows)

# =============================================================
# SECTION 24 — EXCEL OUTPUT
# =============================================================

from openpyxl import load_workbook
from openpyxl.styles import PatternFill, Font, Alignment
from openpyxl.utils import get_column_letter

HEADER_COLORS = {
    "VT_Plan_By_Machine":      "0D6E6E",
    "VT_CO_Queue":             "375623",
    "VT_Plan":                 "1F4E79",
    "VT_Daily_Indent_Status":  "0F4C2A",
    "VT_Multi_Machine_Parts":  "4A235A",
    "VT_Production_vs_Indent": "154360",
    "VT_Priority_Scores":      "2C4770",
    "VT_Machine_Util":         "375623",
    "VT_Not_Planned":          "7B2C2C",
    "VT_Deferred":             "7F6000",
    "VT_Inventory_Health":     "4A235A",
    "VT_Indent_Horizon":       "154360",
    "VT_Part_Audit":           "1C3557",
    "VT_Micro_Idle":           "5C3D2E",
}

STATUS_FILLS = {
    "FULL":     PatternFill("solid", fgColor="C6EFCE"),
    "GOOD":     PatternFill("solid", fgColor="DDEBF7"),
    "PARTIAL":  PatternFill("solid", fgColor="FFEB9C"),
    "UNDERUSED":PatternFill("solid", fgColor="FFC7CE"),
    "OK":       PatternFill("solid", fgColor="C6EFCE"),
    "LOW":      PatternFill("solid", fgColor="FFEB9C"),
    "CRITICAL": PatternFill("solid", fgColor="FFC7CE"),
    "YES ✓":    PatternFill("solid", fgColor="C6EFCE"),
    "NO ✗":     PatternFill("solid", fgColor="FFC7CE"),
    "OVER":     PatternFill("solid", fgColor="DDEBF7"),
    "UNDER":    PatternFill("solid", fgColor="FFC7CE"),
    "MET":      PatternFill("solid", fgColor="C6EFCE"),
    "PRODUCTION NEEDED":        PatternFill("solid", fgColor="FFC7CE"),
    "ZERO INV — FORCED":        PatternFill("solid", fgColor="FFD7D7"),
    "INV SUFFICIENT":           PatternFill("solid", fgColor="C6EFCE"),
    "SKIPPED":                  PatternFill("solid", fgColor="EDEDED"),
    "NOT REQUIRED — INV SUFFICIENT":     PatternFill("solid", fgColor="DDEBF7"),
    "SKIPPED (LOW INDENT / TRIVIAL RUN)":PatternFill("solid", fgColor="EDEDED"),
    "ZERO/MISSING CYCLE TIME":           PatternFill("solid", fgColor="FFC7CE"),
    "NOT IN VT_MATRIX":                  PatternFill("solid", fgColor="FFEB9C"),
    "ZERO/MISSING INDENT":               PatternFill("solid", fgColor="FFEB9C"),
    "ENTERS SCHEDULER":                  PatternFill("solid", fgColor="C6EFCE"),
}

def style_sheet(ws, header_hex):
    for cell in ws[1]:
        cell.fill      = PatternFill("solid", fgColor=header_hex)
        cell.font      = Font(bold=True, color="FFFFFF", size=11)
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
    ws.row_dimensions[1].height = 32
    for col in ws.columns:
        col_letter = get_column_letter(col[0].column)
        max_len    = max((len(str(c.value)) for c in col if c.value is not None), default=8)
        ws.column_dimensions[col_letter].width = max(10, min(55, max_len + 3))
    headers = [cell.value for cell in ws[1]]
    for col_idx, col_name in enumerate(headers, start=1):
        if col_name and any(x in str(col_name) for x in
                            ["Status", "Indent_Status", "Meets_Daily",
                             "Covers_With", "Gap_Direction"]):
            for row in ws.iter_rows(min_row=2, min_col=col_idx, max_col=col_idx):
                for cell in row:
                    fill = STATUS_FILLS.get(str(cell.value))
                    if fill:
                        cell.fill = fill
    ws.freeze_panes = "A2"


def style_machine_wise_sheet(ws):
    header_fill  = PatternFill("solid", fgColor="1F4E79")
    summary_fill = PatternFill("solid", fgColor="0D9488")
    part_fills   = [PatternFill("solid", fgColor="EFF6FF"),
                    PatternFill("solid", fgColor="F0FDF4")]
    co_fill      = PatternFill("solid", fgColor="FEF9C3")

    for cell in ws[1]:
        cell.fill      = header_fill
        cell.font      = Font(bold=True, color="FFFFFF", size=11)
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
    ws.row_dimensions[1].height = 30

    headers      = [cell.value for cell in ws[1]]
    row_type_col = headers.index("Row_Type")  + 1 if "Row_Type"  in headers else None
    co_col       = headers.index("Changeover") + 1 if "Changeover" in headers else None
    machine_col  = headers.index("Machine")   + 1 if "Machine"   in headers else None

    machine_color_idx = 0
    current_machine   = None
    for row in ws.iter_rows(min_row=2):
        row_type = row[row_type_col-1].value if row_type_col else ""
        machine  = row[machine_col-1].value  if machine_col  else ""
        if machine and machine != current_machine:
            current_machine   = machine
            machine_color_idx = (machine_color_idx + 1) % 2
        if row_type == "Summary":
            for cell in row:
                cell.fill = summary_fill
                cell.font = Font(bold=True, color="FFFFFF", size=11)
                cell.alignment = Alignment(horizontal="center", vertical="center")
        elif row_type == "Part":
            for cell in row:
                cell.fill      = part_fills[machine_color_idx]
                cell.alignment = Alignment(vertical="center")
            if co_col and row[co_col-1].value == "Yes":
                row[co_col-1].fill = co_fill

    for col in ws.columns:
        col_letter = get_column_letter(col[0].column)
        max_len    = max((len(str(c.value)) for c in col if c.value), default=8)
        ws.column_dimensions[col_letter].width = max(10, min(45, max_len + 3))
    ws.freeze_panes = "B2"


def style_prod_vs_indent_sheet(ws):
    """Special styling for VT_Production_vs_Indent sheet."""
    style_sheet(ws, HEADER_COLORS["VT_Production_vs_Indent"])

    headers   = [c.value for c in ws[1]]
    gap_col   = headers.index("Gap_Direction") + 1 if "Gap_Direction" in headers else None
    gap_v_col = headers.index("Gap_vs_Daily")  + 1 if "Gap_vs_Daily"  in headers else None

    over_fill  = PatternFill("solid", fgColor="DDEBF7")   # blue — over-produced
    under_fill = PatternFill("solid", fgColor="FFC7CE")   # red  — under-produced
    met_fill   = PatternFill("solid", fgColor="C6EFCE")   # green — exactly met

    for row in ws.iter_rows(min_row=2):
        if gap_col:
            cell  = row[gap_col - 1]
            value = str(cell.value)
            if value == "OVER":
                cell.fill = over_fill
            elif value == "UNDER":
                cell.fill = under_fill
                for c in row:
                    c.font = Font(bold=True)
            elif value == "MET":
                cell.fill = met_fill


def style_multi_machine_sheet(ws):
    """Special styling for VT_Multi_Machine_Parts sheet."""
    style_sheet(ws, HEADER_COLORS["VT_Multi_Machine_Parts"])

    headers  = [c.value for c in ws[1]]
    role_col = headers.index("Role") + 1 if "Role" in headers else None

    total_fill   = PatternFill("solid", fgColor="0D9488")
    primary_fill = PatternFill("solid", fgColor="EFF6FF")
    expand_fill  = PatternFill("solid", fgColor="FEF9C3")

    for row in ws.iter_rows(min_row=2):
        if not role_col:
            continue
        role = str(row[role_col - 1].value)
        if role == "TOTAL":
            for cell in row:
                cell.fill = total_fill
                cell.font = Font(bold=True, color="FFFFFF", size=11)
        elif "Tool-Expansion" in role:
            for cell in row:
                cell.fill = expand_fill
        elif role == "Primary":
            for cell in row:
                cell.fill = primary_fill


# Write Excel
print(f"\nWriting output → {output_path}")

vt_mw    = build_machine_wise_plan(vt_plan)
vt_co_q  = build_co_queue(
    [{k:v for k,v in r.items()} for r in vt_plan.to_dict("records")]
    if not vt_plan.empty else [],
    vt_machines
)

sheets = {
    "VT_Plan_By_Machine":      vt_mw,
    "VT_CO_Queue":             vt_co_q,
    "VT_Plan":                 vt_plan,
    "VT_Multi_Machine_Parts":  vt_multi_machine,
    "VT_Production_vs_Indent": vt_prod_vs_indent,
    "VT_Daily_Indent_Status":  vt_indent_status,
    "VT_Priority_Scores":      vt_scores,
    "VT_Machine_Util":         vt_mach,
    "VT_Not_Planned":          vt_not,
    "VT_Deferred":             vt_def,
    "VT_Inventory_Health":     vt_inv,
    "VT_Indent_Horizon":       vt_horizon,
    "VT_Part_Audit":           audit_df,
}
if not vt_micro.empty:
    sheets["VT_Micro_Idle"] = vt_micro

with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
    for sheet_name, df in sheets.items():
        if df is not None and not df.empty:
            df.to_excel(writer, sheet_name=sheet_name, index=False)

wb = load_workbook(output_path)

if "VT_Plan_By_Machine" in wb.sheetnames:
    style_machine_wise_sheet(wb["VT_Plan_By_Machine"])
if "VT_Production_vs_Indent" in wb.sheetnames:
    style_prod_vs_indent_sheet(wb["VT_Production_vs_Indent"])
if "VT_Multi_Machine_Parts" in wb.sheetnames:
    style_multi_machine_sheet(wb["VT_Multi_Machine_Parts"])

for sheet_name, header_hex in HEADER_COLORS.items():
    if sheet_name in wb.sheetnames and sheet_name not in (
            "VT_Plan_By_Machine", "VT_Production_vs_Indent", "VT_Multi_Machine_Parts"):
        style_sheet(wb[sheet_name], header_hex)

# Colour Daily Indent Status YES/NO columns
if "VT_Daily_Indent_Status" in wb.sheetnames:
    ws_is      = wb["VT_Daily_Indent_Status"]
    headers_is = [c.value for c in ws_is[1]]
    meets_col  = (headers_is.index("Meets_Daily_Indent") + 1
                  if "Meets_Daily_Indent" in headers_is else None)
    covers_col = (headers_is.index("Covers_With_Inv") + 1
                  if "Covers_With_Inv" in headers_is else None)
    for row in ws_is.iter_rows(min_row=2):
        if meets_col:
            cell = row[meets_col - 1]
            cell.fill = (PatternFill("solid", fgColor="C6EFCE")
                         if str(cell.value) == "YES ✓"
                         else PatternFill("solid", fgColor="FFC7CE"))
        if covers_col:
            cell = row[covers_col - 1]
            cell.fill = (PatternFill("solid", fgColor="C6EFCE")
                         if str(cell.value) == "YES ✓"
                         else PatternFill("solid", fgColor="FFEB9C"))
        if meets_col and str(row[meets_col - 1].value) == "NO ✗":
            for cell in row:
                cell.font = Font(bold=True)

for name, color in HEADER_COLORS.items():
    if name in wb.sheetnames:
        wb[name].sheet_properties.tabColor = color

wb.save(output_path)
print(f"  Formatting applied  ✓")

# =============================================================
# SECTION 25 — FINAL SUMMARY
# =============================================================

print(f"\n{'='*65}")
print(f"  Smart APS V8 Complete  —  {PLANNING_DATE}")
print(f"  Tool-Aware Multi-Machine Scheduling")
print(f"{'='*65}")

for status, count in audit_df["Status"].value_counts().items():
    marker = "✓" if status == "ENTERS SCHEDULER" else "·"
    print(f"  {marker} {status:<45}: {count:>4}")

print(f"\n  Results:")
print(f"    Planned            : {len(vt_plan):>4} rows")
print(f"    Not planned        : {len(vt_not):>4}")
print(f"    Deferred           : {len(vt_def):>4}")

if not vt_multi_machine.empty:
    n_multi = vt_multi_machine[vt_multi_machine["Role"] == "TOTAL"].shape[0]
    print(f"    Multi-machine parts: {n_multi:>4}  (see VT_Multi_Machine_Parts)")

if not vt_prod_vs_indent.empty:
    over  = (vt_prod_vs_indent["Gap_Direction"] == "OVER").sum()
    under = (vt_prod_vs_indent["Gap_Direction"] == "UNDER").sum()
    met   = (vt_prod_vs_indent["Gap_Direction"] == "MET").sum()
    print(f"\n  Production vs Daily Indent:")
    print(f"    Over-produced  : {over:>4} parts  (inv building — see VT_Production_vs_Indent)")
    print(f"    Under-produced : {under:>4} parts  (machine capacity limited)")
    print(f"    Exactly met    : {met:>4} parts")

if not vt_mach.empty:
    print(f"\n  Machine utilization:")
    print(f"    Average     : {vt_mach['Utilization_%'].mean():.1f}%")
    print(f"    UNDERUSED   : {(vt_mach['Status']=='UNDERUSED').sum()} machines")

print(f"\n  Output → {output_path}")
print(f"  State  → {MACHINE_STATE_FILE}")
print(f"\n  UPDATE DAILY: PLANNING_DATE = date(2026, 3, 21)")
print(f"{'='*65}")


  Smart APS V8  —  Tool-Aware Multi-Machine Scheduling
  Planning date : 2026-03-20
  Indent month  : March 2026
  Working days  : 26  (31 days − 5 Sundays)

Loading data...
  VT parts in sheet       : 158


NameError: name 'part_color' is not defined

In [2]:
import pandas as pd
import numpy as np
import json
import os
import math
import calendar
from datetime import datetime, date

# =============================================================
#  Smart APS V8  —  Tool-Aware Multi-Machine Scheduling
#
#  What's new vs V7:
#  ─────────────────────────────────────────────────────────────
#  1. TOOL-AWARE SCHEDULING
#     Every part has a tool count (column "Tools" in VT sheet).
#     Tools control how many machines can run the part in parallel.
#
#     Phase 1 — Primary machine
#       Assign best-ranked compatible machine.
#       Run for min(available_hours, hours_to_meet_daily_indent).
#       If daily indent fully met → go to Phase 3.
#
#     Phase 2 — Tool expansion (only if Phase 1 fell short)
#       shortfall = daily_indent − qty_produced_on_machine_1
#       If tools ≥ 2 and another compatible machine has capacity:
#         Assign shortfall-only hours to machine 2.
#       If still short and tools ≥ 3 → machine 3, etc.
#       Max parallel machines = min(tools, compatible machines).
#       Second machine starts mid-shift (not from 00:00) — it only
#       fills what machine 1 couldn't.
#
#     Phase 3 — Inventory build (after daily indent fully met)
#       Extend the SAME machine(s) already running the part.
#       Cap = scenario OPD. No new tools consumed for inventory.
#
#  2. TWO NEW EXCEL OUTPUT SHEETS
#
#     VT_Multi_Machine_Parts
#       Only parts running on ≥ 2 machines.
#       Columns: Part | Machine | Run_Hours | Production_Qty |
#                Role (Primary / Tool-Expansion) | Tools_Available |
#                Machines_Used | Total_Qty_Across_Machines
#
#     VT_Production_vs_Indent
#       Every planned part.
#       Columns: Part | Machine(s) | Total_Qty_Produced |
#                Daily_Indent | Gap (+over / −under) |
#                Gap_Direction (OVER / UNDER / MET) |
#                Extra_Days_Stock (if over) |
#                Inventory_Before | Inventory_After
#
#  Planning logic (unchanged):
#    daily_indent  = monthly_indent / working_days
#    Rate          = 3600 / cycle_time  (cavities cancel)
#    working_days  = calendar days − Sundays
# =============================================================

# =============================================================
# SECTION 1 — DAILY SETTINGS
# =============================================================

PLANNING_DATE = date(2026, 3, 20)   # ← change daily
INDENT_MONTH  = date(2026, 3,  1)   # ← change when month rolls over

# =============================================================
# SECTION 2 — PARAMETERS
# =============================================================

AVAILABLE_HOURS      = 22
MIN_RUN_HOURS        = 4
TARGET_DAYS_INV      = 3
MACHINE_STATE_FILE   = "machine_state.json"

MIN_DAILY_INDENT     = 150
MIN_INDENT_HOURS     = 4.0

# =============================================================
# PRODUCTION TARGET LOGIC
# =============================================================
# Full target = TARGET_DAYS_INV (3 days buffer)
#             + SAFETY_DAYS     (2 days safety cushion)
#             = 5 days of daily indent total
#
# Why 5 days:
#   3 days = the inventory buffer we want to hold at all times
#   2 days = safety net for the reality that a part may not
#            get planned every single day. If it skips a day,
#            those 2 days cover the consumption gap.
#
# Scenario caps (maximum inventory after production):
#   S0/S1/S2 → 5 × daily_indent
#     Below full target. Produce as much as possible toward 5 days.
#     If inventory = 0 → produce up to 5 days in one run.
#     If inventory = 2 days → produce up to 3 more days to reach 5.
#
#   S3 (inventory already ≥ 5 days for all parts) → 1 × daily_indent
#     Already fully covered. Produce only today's consumption.
#     Subject to MIN_RUN_HOURS floor.
#
# The classify_scenario() thresholds are also updated:
#   Critical  = inventory < 1 day  (unchanged)
#   Healthy   = inventory ≥ 5 days (was 3 days — now matches full target)
#   Below buf = 1 ≤ inventory < 5 days
# =============================================================

SAFETY_DAYS          = 2     # extra days beyond buffer for safety cushion
FULL_TARGET_DAYS     = TARGET_DAYS_INV + SAFETY_DAYS   # = 5 days

OPD_SCENARIO_0 = float(FULL_TARGET_DAYS)   # 5.0 — critical: build to full target
OPD_SCENARIO_1 = float(FULL_TARGET_DAYS)   # 5.0 — mixed:    build to full target
OPD_SCENARIO_2 = float(FULL_TARGET_DAYS)   # 5.0 — below:    build to full target
OPD_SCENARIO_3 = 1.0                        # healthy: replace today's consumption only

W_URGENCY  = 0.55
W_CATEGORY = 0.25
W_INDENT   = 0.20

UTIL_TARGET_PCT = 98.0

# =============================================================
# SECTION 3 — FILE PATHS
# =============================================================

book_path       = "C:/Users/Ex0164/Book1.xlsx"
matrix_path     = "C:/Users/Ex0164/Important codes/compatibility_matrix.xlsx"
changeover_path = "D:/Tushar/TOOL FIX/Unique_Machines_vt.xlsx"
output_path     = f"Smart_APS_V8_Plan_{PLANNING_DATE.strftime('%Y%m%d')}.xlsx"

# =============================================================
# SECTION 4 — WORKING DAYS
# =============================================================

def compute_working_days(ref_date):
    year  = ref_date.year
    month = ref_date.month
    total = calendar.monthrange(year, month)[1]
    sundays = sum(
        1 for d in range(1, total + 1)
        if date(year, month, d).weekday() == 6
    )
    return total - sundays, total, sundays

WORKING_DAYS, TOTAL_DAYS, SUNDAY_COUNT = compute_working_days(INDENT_MONTH)

print(f"\n{'='*65}")
print(f"  Smart APS V8  —  Tool-Aware Multi-Machine Scheduling")
print(f"  Planning date : {PLANNING_DATE}")
print(f"  Indent month  : {INDENT_MONTH.strftime('%B %Y')}")
print(f"  Working days  : {WORKING_DAYS}  ({TOTAL_DAYS} days − {SUNDAY_COUNT} Sundays)")
print(f"{'='*65}\n")

# =============================================================
# SECTION 5 — LOAD DATA
# =============================================================

print("Loading data...")
vt_parts_raw = pd.read_excel(book_path,       sheet_name="VT")
vt_matrix    = pd.read_excel(matrix_path,     sheet_name="VT_Matrix")
vt_co_raw         = pd.read_excel(changeover_path, sheet_name="VT_Changeover")
vt_machine_count_raw = pd.read_excel(matrix_path, sheet_name="VT_Machine_Part_Count")

# =============================================================
# SECTION 6 — PARSE VT SHEET
# =============================================================

def find_col(df, name, sheet):
    match = next((c for c in df.columns
                  if str(c).strip().lower() == name.lower()), None)
    if match is None:
        raise ValueError(
            f"Column '{name}' not found in sheet '{sheet}'.\n"
            f"Available columns: {list(df.columns)}"
        )
    return match

vt_col_part      = find_col(vt_parts_raw, "Part",       "VT")
vt_col_cycletime = find_col(vt_parts_raw, "Cycle time", "VT")
vt_col_cavity    = find_col(vt_parts_raw, "Cavity",     "VT")
vt_col_inventory = find_col(vt_parts_raw, "Inventory",  "VT")
vt_col_indent    = find_col(vt_parts_raw, "Indent",     "VT")
vt_col_tools     = find_col(vt_parts_raw, "Tools",      "VT")
vt_col_color     = find_col(vt_parts_raw, "Color",      "VT")

data = vt_parts_raw[
    vt_parts_raw[vt_col_part].notna() &
    (vt_parts_raw[vt_col_part].astype(str).str.strip() != "")
].copy()
data = data.drop_duplicates(subset=vt_col_part).copy()
data["Material"] = data[vt_col_part].astype(str).str.strip()

# Rate = 3600 / cycle_time  (cavities cancel — full cavity operation assumed)
data["_ct"] = pd.to_numeric(data[vt_col_cycletime], errors="coerce").replace(0, np.nan)
data["Rate"] = 3600 / data["_ct"]

data_valid     = data[data["Rate"].notna()].copy()
data_zero_rate = data[data["Rate"].isna()].copy()

print(f"  VT parts in sheet       : {len(data)}")
# color_groups = {}
# for p, c in part_color.items():
#     color_groups.setdefault(c, []).append(p)
# print(f"  Distinct colours        : {len(color_groups)}")
# for col, parts in sorted(color_groups.items()):
#     print(f"    {col:<20} → {len(parts)} part(s)")
print(f"  Parts with valid rate   : {len(data_valid)}")

# =============================================================
# SECTION 7 — LOOKUP DICTIONARIES
# =============================================================

def safe_dict(df, key_col, val_col, default=0.0):
    return {
        str(k).strip(): (default if pd.isna(v) else float(v))
        for k, v in zip(df[key_col], df[val_col])
        if pd.notna(k) and str(k).strip() != ""
    }

inventory      = safe_dict(data,       "Material", vt_col_inventory)
rate           = safe_dict(data_valid, "Material", "Rate")
indent_monthly = safe_dict(data,       "Material", vt_col_indent)

# Tools available per part (default 1 if missing)
tools_available = {}
part_color      = {}   # {part: color_string}
for _, row in data.iterrows():
    p = str(row["Material"]).strip()
    v = row[vt_col_tools]
    tools_available[p] = max(1, int(float(v))) if pd.notna(v) and str(v).strip() != "" else 1
    c = row[vt_col_color]
    part_color[p] = str(c).strip().upper() if pd.notna(c) and str(c).strip() != "" else "UNKNOWN"

indent_daily = {
    p: round(qty / WORKING_DAYS, 4)
    for p, qty in indent_monthly.items()
}

color_groups = {}
for p, c in part_color.items():
    color_groups.setdefault(c, []).append(p)

print(f"  Distinct colours        : {len(color_groups)}")
for col, parts in sorted(color_groups.items()):
    print(f"    {col:<20} → {len(parts)} part(s)")


today_target_qty = {
    p: max(0.0, indent_daily.get(p, 0.0) - inventory.get(p, 0.0))
    for p in indent_monthly
}

# =============================================================
# SECTION 7A — SKIP RULES
# =============================================================

def should_skip(part):
    daily   = indent_daily.get(part, 0.0)
    monthly = indent_monthly.get(part, 0.0)
    r       = rate.get(part, 1.0)

    if daily <= MIN_DAILY_INDENT:
        return True, f"Daily indent {daily:.2f} ≤ {MIN_DAILY_INDENT} threshold"

    indent_hrs = monthly / r if r > 0 else 0.0
    if indent_hrs <= MIN_INDENT_HOURS:
        return True, f"Whole monthly indent = {indent_hrs:.2f}h ≤ {MIN_INDENT_HOURS}h threshold"

    return False, ""

# =============================================================
# SECTION 7B — CHANGEOVER TIMES
# =============================================================

def build_changeover_dict(co_df):
    co_dict = {}
    for _, row in co_df.iterrows():
        machine = str(row["Unique Machines"]).strip()
        minutes = row["Changeover time"]
        if pd.notna(minutes) and machine:
            co_dict[machine] = float(minutes) / 60.0
    return co_dict

vt_changeover          = build_changeover_dict(vt_co_raw)
DEFAULT_CHANGEOVER_HRS = 40 / 60.0

# =============================================================
# SECTION 7B2 — MACHINE PART COUNT  (from VT_Machine_Part_Count)
# -------------------------------------------------------------
# machine_part_count : {machine_name → int}
# Used in rank_machines() to prefer the least-flexible machine
# (lowest part count) for any given part assignment.
# A machine that runs only 2 parts is always preferred over one
# that runs 10 — so the constrained machine is claimed first.
# =============================================================

def build_machine_part_count(df):
    """
    Reads VT_Machine_Part_Count sheet.
    Columns: Machine | Part_Count
    Returns dict {machine_name: part_count}.
    """
    mpc = {}
    machine_col = next((c for c in df.columns
                        if str(c).strip().lower() == "machine"), None)
    count_col   = next((c for c in df.columns
                        if str(c).strip().lower() == "part_count"), None)
    if machine_col is None or count_col is None:
        print(f"  WARNING: VT_Machine_Part_Count sheet missing "
              f"'Machine' or 'Part_Count' column — using equal ranking")
        return {}
    for _, row in df.iterrows():
        m = str(row[machine_col]).strip()
        v = row[count_col]
        if m and pd.notna(v):
            try:
                mpc[m] = int(float(v))
            except (ValueError, TypeError):
                pass
    return mpc

machine_part_count = build_machine_part_count(vt_machine_count_raw)
max_part_count     = max(machine_part_count.values(), default=1) or 1

print(f"  Machine part counts loaded: {len(machine_part_count)} machines")
if machine_part_count:
    # Show most specialized machines
    specialized = sorted(machine_part_count.items(), key=lambda x: x[1])[:5]
    for m, c in specialized:
        print(f"    {m:<25} → {c} part(s)  "
              f"{'← MOST SPECIALIZED' if c <= 3 else ''}")

# =============================================================
# SECTION 7C — PART CATEGORY
# =============================================================

def build_category(df):
    cat      = {}
    part_col = next((c for c in df.columns if str(c).strip().lower() == "part"), None)
    cat_col  = next((c for c in df.columns if str(c).strip().lower() == "category"), None)
    if part_col is None:
        return cat
    for _, row in df.iterrows():
        part = row[part_col]
        if pd.isna(part) or str(part).strip() == "":
            continue
        val = "Stranger"
        if cat_col and pd.notna(row[cat_col]):
            val = str(row[cat_col]).strip().capitalize()
            if val not in ("Runner", "Repeater", "Stranger"):
                val = "Stranger"
        cat[str(part).strip()] = val
    return cat

part_category  = build_category(vt_parts_raw)
CATEGORY_SCORE = {"Runner": 100, "Repeater": 60, "Stranger": 20}

# =============================================================
# SECTION 8 — MACHINE STATE
# =============================================================

def load_machine_state():
    if os.path.exists(MACHINE_STATE_FILE):
        try:
            with open(MACHINE_STATE_FILE) as f:
                content = f.read().strip()
            if not content:
                print(f"  Machine state : file empty — treating as first run")
                os.remove(MACHINE_STATE_FILE)
                return {}
            state = json.loads(content)
            if not isinstance(state, dict):
                print(f"  Machine state : file corrupt — treating as first run")
                os.remove(MACHINE_STATE_FILE)
                return {}
            print(f"  Machine state loaded  ({len(state)} machines with history)")
            return state
        except json.JSONDecodeError as e:
            print(f"  Machine state : JSON error ({e}) — treating as first run")
            os.remove(MACHINE_STATE_FILE)
            return {}
        except Exception as e:
            print(f"  Machine state : read error ({e}) — treating as first run")
            os.remove(MACHINE_STATE_FILE)
            return {}
    print(f"  Machine state : FIRST RUN — no changeover today")
    return {}

def save_machine_state(state):
    combined = {m: p for m, p in state.items() if p is not None}
    with open(MACHINE_STATE_FILE, "w") as f:
        json.dump(combined, f, indent=2)
    print(f"\n  Machine state saved → '{MACHINE_STATE_FILE}'")

machine_state = load_machine_state()

# =============================================================
# SECTION 9 — COMPATIBILITY MATRIX
# =============================================================

def build_compatibility(matrix):
    machines = matrix.columns[1:].tolist()
    compat   = {}
    for _, row in matrix.iterrows():
        part = row["Part"]
        for m in machines:
            if row[m] == 1:
                compat.setdefault(part, []).append(m)
    return compat, machines

vt_compat, vt_machines = build_compatibility(vt_matrix)

# =============================================================
# SECTION 10 — SCENARIO CLASSIFIER
# =============================================================

def classify_scenario(parts):
    """
    Classify the day based on inventory health.

    Thresholds:
      critical = inventory < 1 day           → must produce today
      below    = 1 day ≤ inv < FULL_TARGET_DAYS (5 days)
      healthy  = inventory ≥ FULL_TARGET_DAYS (5 days)

    S0: all active parts critical
    S1: some critical, some have stock
    S2: none critical but below 5-day target
    S3: all parts at or above 5-day target → produce 1 day only
    """
    coverage = []
    for p in parts:
        inv   = inventory.get(p, 0)
        daily = indent_daily.get(p, 0)
        skip, _ = should_skip(p)
        if skip or daily == 0:
            continue
        coverage.append(inv / daily)

    if not coverage:
        return 3, "SCENARIO 3 — No active parts today"

    n        = len(coverage)
    critical = sum(1 for c in coverage if c < 1)
    low      = sum(1 for c in coverage if c < FULL_TARGET_DAYS)

    if critical == n:
        return 0, (f"SCENARIO 0 — ALL {n} parts critical (inv < 1 day) "
                   f"| target = {FULL_TARGET_DAYS} days")
    elif critical > 0:
        return 1, (f"SCENARIO 1 — {critical}/{n} parts critical "
                   f"| {n-critical} have stock | target = {FULL_TARGET_DAYS} days")
    elif low > 0:
        return 2, (f"SCENARIO 2 — {low}/{n} parts below {FULL_TARGET_DAYS}-day target "
                   f"| {n-low} already healthy")
    else:
        return 3, (f"SCENARIO 3 — All {n} parts healthy "
                   f"(≥{FULL_TARGET_DAYS} days) | produce 1 day only")

# =============================================================
# SECTION 11 — OPD CAP BY SCENARIO
# =============================================================

def opd_cap(scenario_id):
    """
    Returns the production target as a multiple of daily_indent.
    S0/S1/S2 → 5.0 (build to full 5-day target: 3 buffer + 2 safety)
    S3       → 1.0 (inventory already healthy — replace 1 day only)
    """
    return {0: OPD_SCENARIO_0, 1: OPD_SCENARIO_1,
            2: OPD_SCENARIO_2, 3: OPD_SCENARIO_3}.get(scenario_id, OPD_SCENARIO_2)

# =============================================================
# SECTION 12 — PRIORITY SCORING
# =============================================================

def compute_priority_scores(active_parts):
    rows = []
    for p in active_parts:
        inv      = inventory.get(p, 0)
        daily    = indent_daily.get(p, 0)
        cat      = part_category.get(p, "Stranger")
        days_cov = inv / daily if daily > 0 else 999.0
        gap_days = min(1.0, max(0.0, TARGET_DAYS_INV - days_cov) / TARGET_DAYS_INV)
        rows.append({"part": p, "inv": inv, "daily": daily,
                     "days_cov": days_cov, "cat": cat, "urgency_raw": gap_days})

    if not rows:
        return {}, []

    max_daily = max(r["daily"] for r in rows) or 1.0
    scores, score_rows = {}, []

    for r in rows:
        p              = r["part"]
        urgency_score  = r["urgency_raw"] * 100
        category_score = CATEGORY_SCORE.get(r["cat"], 20)
        indent_score   = (r["daily"] / max_daily) * 100
        final_score    = (W_URGENCY * urgency_score +
                          W_CATEGORY * category_score +
                          W_INDENT   * indent_score)
        scores[p] = round(final_score, 2)
        score_rows.append({
            "Part":           p,
            "Category":       r["cat"],
            "Tools":          tools_available.get(p, 1),
            "Inventory_Now":  round(r["inv"], 0),
            "Daily_Indent":   round(r["daily"], 2),
            "Days_Coverage":  round(r["days_cov"], 2),
            "Urgency_Score":  round(urgency_score, 1),
            "Category_Score": category_score,
            "Indent_Score":   round(indent_score, 1),
            "Final_Score":    round(final_score, 2),
        })

    return scores, score_rows

# =============================================================
# SECTION 13 — MACHINE RANKER
# =============================================================

def rank_machines(part, machines_to_try, machine_hours,
                  machine_last_part, inv_days):
    """
    Ranks compatible machines for a part assignment.

    PRIMARY criterion — Part_Count (from VT_Machine_Part_Count sheet):
      The machine that can run the FEWEST parts is preferred first.
      Rationale: a machine running only 2 parts is far more constrained
      than one running 15. Assigning the part to the constrained machine
      first keeps flexible machines free for parts that have no other option.

      part_count_score = part_count / max_part_count  (0..1, lower = better)

    SECONDARY criteria (tie-break within same part_count):
      • Same part ran last → no changeover (strong bonus)
      • Lower current utilization → more hours available

    Runner lock: a Runner with inv ≤ 1 day must stay on its last machine.
    """
    category    = part_category.get(part, "Stranger")
    runner_lock = (category == "Runner" and inv_days <= 1.0)

    ranked = []
    for m in machines_to_try:
        used = machine_hours.get(m, 0)
        free = round(AVAILABLE_HOURS - used, 4)
        if free < MIN_RUN_HOURS:
            continue
        last = machine_last_part.get(m)
        if runner_lock and last != part:
            continue
        co_hrs = 0.0 if (last is None or last == part) else \
                 vt_changeover.get(m, DEFAULT_CHANGEOVER_HRS)
        effective_free = round(free - co_hrs, 4)
        if effective_free < MIN_RUN_HOURS:
            continue

        # PRIMARY: prefer machine with lowest part count
        # (most specialized / least flexible machine first)
        part_count      = machine_part_count.get(m, max_part_count)
        count_score     = part_count / max_part_count   # 0..1, lower = better

        # SECONDARY: changeover, colour match, utilization
        co_penalty      = (co_hrs / AVAILABLE_HOURS) * 0.3
        util_penalty    = (used  / AVAILABLE_HOURS) * 0.2
        same_part_bonus = -0.15 if (last == part) else 0.0

        # Same-colour bonus: prefer machines where last part
        # shares the same colour — avoids the 10-min purge.
        # Applies only when a changeover is needed (last != part).
        if last is not None and last != part:
            last_color = part_color.get(last, "UNKNOWN")
            new_color  = part_color.get(part,  "UNKNOWN")
            same_color = (last_color == new_color
                          and last_color != "UNKNOWN")
            color_bonus = -0.08 if same_color else 0.0
        else:
            color_bonus = 0.0

        # Combined cost — part_count_score dominates (weight 1.0)
        cost = count_score + co_penalty + util_penalty + same_part_bonus + color_bonus

        ranked.append((m, co_hrs, effective_free, cost))

    ranked.sort(key=lambda x: x[3])
    return ranked, runner_lock

# =============================================================
# SECTION 14 — TOOL-AWARE ASSIGNMENT  (core new logic in V8)
# =============================================================
#
# assign_part() handles all three phases for a single part.
#
# MINIMUM TOOLS PRINCIPLE:
#   Always try to produce the full daily indent on ONE machine
#   first. Only add a second machine (second tool) if the primary
#   machine physically cannot produce the full shortfall in its
#   remaining hours. Only add a third if two are not enough.
#   Goal: use the fewest tools necessary.
#
#   Phase 1 — Primary machine
#     Check if primary machine can cover the FULL daily indent
#     shortfall in its available hours.
#     If yes → run it for the full amount. Done. No more tools.
#     If no  → run it for its max capacity, note the shortfall.
#
#   Phase 2 — Tool expansion (ONLY if Phase 1 left a shortfall)
#     Normal parts: maximum 2 tools total (1 primary + 1 expansion).
#     Zero-inventory (crisis) parts: all available tools may be used,
#     but only if 2 tools together still cannot cover the shortfall.
#     3+ tools is the rarest of rare cases and is logged clearly.
#     Each additional machine only ever runs enough to cover
#     what the previous machine(s) could not.
#
#   Phase 3 — Inventory build (after daily indent fully met)
#     Extend the SAME machines already running this part.
#     Cap = scenario OPD. No new tools consumed.
#
# Returns list of plan rows added, or empty list if not planned.
# =============================================================

def assign_part(part, scenario_id, machine_hours, machine_last_part,
                current_inventory, plan, already_planned,
                priority_scores):
    """
    Assigns a part using minimum-tools principle.
    Returns list of new plan rows (empty = not planned).
    """
    daily    = indent_daily.get(part, 0)
    monthly  = indent_monthly.get(part, 0)
    r_val    = rate.get(part, 1)
    inv_now  = current_inventory.get(part, 0)
    category = part_category.get(part, "Stranger")
    tools    = tools_available.get(part, 1)
    score    = priority_scores.get(part, 0)
    inv_days = inv_now / daily if daily > 0 else 999
    compatible = vt_compat.get(part, [])

    if not compatible:
        return []

    # ── Target quantity ──────────────────────────────────────
    # How much should we produce today for this part?
    #
    # S0/S1/S2 (below 5-day full target):
    #   Full target = 5 × daily_indent (3 days buffer + 2 days safety)
    #   Total shortfall = full target − current inventory
    #   We produce as much as possible toward this target.
    #   Example: inv=0, daily=500 → target=2500, produce up to 2500
    #   Example: inv=1000 (2 days), daily=500 → target=2500, produce 1500
    #
    # S3 (inventory already ≥ 5 days):
    #   Target = 1 × daily_indent (replace today's consumption only).
    #   No point producing more — buffer is fully covered.
    #   Subject to MIN_RUN_HOURS floor — machine always runs ≥ 4h.
    #
    cap_days        = opd_cap(scenario_id)            # 5.0 or 1.0
    target_inv      = cap_days * daily                # desired inventory level
    total_shortfall = max(0.0, target_inv - inv_now)  # gap to fill
    # Floor: always produce at least MIN_RUN_HOURS worth
    hrs_for_full    = total_shortfall / r_val if r_val > 0 else MIN_RUN_HOURS
    hrs_for_full    = max(MIN_RUN_HOURS, hrs_for_full)

    new_rows        = []
    produced_so_far = 0.0
    tools_used      = 0
    used_machines   = set()

    # ── PHASE 1: Primary machine — try to fit FULL indent on one machine
    ranked, runner_lock = rank_machines(
        part, compatible, machine_hours, machine_last_part, inv_days)

    if not ranked:
        return []

    m1, co1, eff1, _ = ranked[0]

    # Can machine 1 cover the full shortfall?
    # run1 = min(what machine can give, what we need for full indent)
    run1 = min(eff1, hrs_for_full)
    run1 = max(run1, MIN_RUN_HOURS)
    qty1 = round(run1 * r_val, 0)

    machine_hours[m1]       = round(machine_hours.get(m1, 0) + co1 + run1, 4)
    current_inventory[part] = round(current_inventory.get(part, 0) + qty1, 0)
    machine_last_part[m1]   = part
    produced_so_far        += qty1
    tools_used             += 1
    used_machines.add(m1)
    already_planned.add(part)

    # Did machine 1 cover the full indent?
    indent_met_on_primary = (produced_so_far >= total_shortfall - 0.5)

    new_rows.append({
        "Part":             part,
        "Category":         category,
        "Machine":          m1,
        "Run_Hours":        round(run1, 3),
        "Changeover_Hrs":   round(co1, 3),
        "Total_Hrs_Used":   round(co1 + run1, 3),
        "Rate_Per_Hour":    round(r_val, 2),
        "Production_Qty":   qty1,
        "Monthly_Indent":   round(monthly, 0),
        "Daily_Indent":     round(daily, 2),
        "Today_Target":     round(today_target_qty.get(part, 0), 0),
        "Changeover":       "No" if co1 == 0 else "Yes",
        "Type":             "Primary" + (" [ZERO-INV]" if inv_now == 0 else ""),
        "Role":             "Primary",
        "Tools_Available":  tools,
        "Tools_Used":       1,
        "Runner_Lock":      "YES" if runner_lock else "No",
        "Priority_Score":   score,
        "Phase":            1,
        "Indent_Met":       "YES" if indent_met_on_primary else "NO — shortfall remains",
        "Color":            part_color.get(part, "UNKNOWN"),
        "Stagger_Adjusted": "No",
    })

    if indent_met_on_primary:
        # Full indent covered on primary machine — minimum tools = 1
        # Skip Phase 2 entirely
        pass

    else:
        # ── PHASE 2: Tool expansion — only because primary fell short ──
        #
        # Tool usage policy:
        #   Tool 1 (primary)  — always used, covers as much as possible.
        #   Tool 2            — only if tool 1 cannot meet daily indent alone.
        #   Tool 3+           — rarest of rare. Only if inv = 0 AND tools 1+2
        #                       together still cannot cover the shortfall.

        is_critical   = (inv_now == 0)
        tool_hard_cap = tools if is_critical else min(2, tools)

        while produced_so_far < (total_shortfall - 0.5) and tools_used < tool_hard_cap:
            shortfall_now = total_shortfall - produced_so_far
            hrs_needed    = max(MIN_RUN_HOURS,
                                shortfall_now / r_val if r_val > 0 else MIN_RUN_HOURS)

            remaining_machines = [m for m in compatible if m not in used_machines]
            ranked_next, _ = rank_machines(
                part, remaining_machines, machine_hours,
                machine_last_part, inv_days)

            if not ranked_next:
                break

            mx, cox, effx, _ = ranked_next[0]

            # Run only as much as needed to cover remaining shortfall
            # Never run more than the machine has available
            run_x = min(effx, hrs_needed)
            run_x = max(run_x, MIN_RUN_HOURS)
            qty_x = round(run_x * r_val, 0)

            machine_hours[mx]       = round(machine_hours.get(mx, 0) + cox + run_x, 4)
            current_inventory[part] = round(current_inventory.get(part, 0) + qty_x, 0)
            machine_last_part[mx]   = part
            produced_so_far        += qty_x
            tools_used             += 1
            used_machines.add(mx)

            indent_met_here = (produced_so_far >= total_shortfall - 0.5)

            new_rows.append({
                "Part":             part,
                "Category":         category,
                "Machine":          mx,
                "Run_Hours":        round(run_x, 3),
                "Changeover_Hrs":   round(cox, 3),
                "Total_Hrs_Used":   round(cox + run_x, 3),
                "Rate_Per_Hour":    round(r_val, 2),
                "Production_Qty":   qty_x,
                "Monthly_Indent":   round(monthly, 0),
                "Daily_Indent":     round(daily, 2),
                "Today_Target":     round(today_target_qty.get(part, 0), 0),
                "Changeover":       "No" if cox == 0 else "Yes",
                "Type":             "Tool-Expansion",
                "Role":             f"Tool-Expansion (tool {tools_used}/{tools})",
                "Tools_Available":  tools,
                "Tools_Used":       tools_used,
                "Runner_Lock":      "No",
                "Priority_Score":   score,
                "Phase":            2,
                "Indent_Met":       "YES" if indent_met_here else "NO — shortfall remains",
                "Stagger_Adjusted": "No",
            })

            crisis_tag = " [CRISIS — 3rd+ tool]" if tools_used >= 3 else ""
            print(f"      ↳ TOOL-EXP {part:26s} tool {tools_used}/{tool_hard_cap} → "
                  f"{mx:15s}  {run_x:.2f}h  qty={qty_x:.0f}  "
                  f"shortfall was {shortfall_now:.0f}  "
                  f"{'COVERED ✓' if indent_met_here else 'still short'}"
                  f"{crisis_tag}")

    # ── PHASE 3: Inventory build on same machines ─────────────
    # Extend existing run rows for this part up to OPD cap.
    # No new machines, no new tools.
    cap_days     = opd_cap(scenario_id)
    inv_after    = current_inventory.get(part, 0)
    cap_qty      = cap_days * daily
    headroom_qty = max(0.0, cap_qty - inv_after)

    if headroom_qty > 0:
        for row in new_rows:
            if headroom_qty <= 0:
                break
            m       = row["Machine"]
            free_m  = round(AVAILABLE_HOURS - machine_hours.get(m, 0), 4)
            if free_m < 0.05:
                continue
            extend_hrs = min(free_m, headroom_qty / r_val if r_val > 0 else 0)
            if extend_hrs < 0.05:
                continue
            extra_qty = round(extend_hrs * r_val, 0)

            row["Run_Hours"]      = round(float(row["Run_Hours"]) + extend_hrs, 3)
            row["Total_Hrs_Used"] = round(float(row["Changeover_Hrs"]) + float(row["Run_Hours"]), 3)
            row["Production_Qty"] = round(float(row["Production_Qty"]) + extra_qty, 0)
            row["Type"]           = str(row["Type"]) + "+InvBuild"

            machine_hours[m]        = round(machine_hours.get(m, 0) + extend_hrs, 4)
            current_inventory[part] = round(current_inventory.get(part, 0) + extra_qty, 0)
            headroom_qty           -= extra_qty

            print(f"      ↳ INV-BUILD {part:25s} on {m:15s}  "
                  f"+{extend_hrs:.2f}h  qty+={extra_qty:.0f}")

    # Update Tools_Used on all rows for this part
    for row in new_rows:
        row["Tools_Used"] = tools_used

    return new_rows

# =============================================================
# SECTION 15 — CSP TOOL-CHANGER  (unchanged from V7)
# =============================================================

def _fmt_h(h):
    try:
        total_min = int(round(float(h) * 60))
        return f"{total_min // 60:02d}:{total_min % 60:02d}"
    except Exception:
        return "??"

def _collect_co_events(plan, machines):
    events = []
    for m in machines:
        m_rows = [r for r in plan if r["Machine"] == m]
        if len(m_rows) < 2:
            continue
        cursor = 0.0
        for i, row in enumerate(m_rows):
            co_h  = float(row.get("Changeover_Hrs") or 0.0)
            run_h = float(row.get("Run_Hours")       or 0.0)
            if co_h > 0 and i > 0:
                events.append({
                    "machine":       m,
                    "part_before":   m_rows[i-1]["Part"],
                    "part_after":    row["Part"],
                    "co_duration":   co_h,
                    "natural_start": cursor,
                    "row_before":    m_rows[i-1],
                    "row_after":     row,
                    "actual_start":  None,
                    "wait_hrs":      0.0,
                })
            cursor += co_h + run_h
    return events

def _recompute_natural_start(ev, plan):
    m, target = ev["machine"], ev["row_after"]
    cursor = 0.0
    for r in [r for r in plan if r["Machine"] == m]:
        if r is target:
            break
        cursor += float(r.get("Changeover_Hrs") or 0) + float(r.get("Run_Hours") or 0)
    return cursor

def _machine_spare(m, plan):
    used = sum(
        float(r.get("Changeover_Hrs") or 0) + float(r.get("Run_Hours") or 0)
        for r in plan if r["Machine"] == m
    )
    return max(0.0, AVAILABLE_HOURS - used)

def _extend_row_before(ev, wait_hrs, plan):
    spare     = _machine_spare(ev["machine"], plan)
    extend_by = min(wait_hrs, spare)
    if extend_by <= 0:
        return 0.0, 0
    rb    = ev["row_before"]
    r_val = rate.get(rb["Part"], 1.0)
    extra = round(extend_by * r_val, 0)
    rb["Run_Hours"]      = round(float(rb.get("Run_Hours") or 0) + extend_by, 3)
    rb["Production_Qty"] = round(float(rb.get("Production_Qty") or 0) + extra, 0)
    rb["Total_Hrs_Used"] = round(
        float(rb.get("Changeover_Hrs") or 0) + float(rb["Run_Hours"]), 3)
    rb["Stagger_Adjusted"] = f"CSP: extended +{round(extend_by*60,1)}min to fill TC wait"
    return extend_by, extra

# =============================================================
# SECTION 15B — QUANTITY-BASED CO STAGGER
# =============================================================
# Instead of scheduling times and hoping the floor follows,
# we adjust production quantities so changeovers naturally
# fall at different times — baked into the plan itself.
#
# Algorithm:
#   1. Compute finish_time for every CO event:
#      finish_time = sum of run_hours of parts BEFORE the CO
#      on that machine (i.e. when the machine will call for
#      the tool changer if it runs at planned rate).
#   2. Sort CO events by finish_time ascending.
#   3. Walk through pairs. If gap < MIN_CO_GAP_HRS:
#      a. PUSH:  add qty to the later machine's last part
#                before its CO, pushing its finish_time forward.
#                Extra qty = gap_shortfall × rate — free production.
#      b. If push would exceed OPD cap → PULL instead:
#                reduce the earlier machine's last part qty,
#                never below daily_indent.
#   4. Re-sort after each adjustment (finish_times shift).
#   5. Repeat until all gaps ≥ MIN_CO_GAP_HRS or no more
#      adjustments are possible.
#
# Result: changeovers are naturally staggered in the quantities
# themselves. Floor doesn't need to follow a time schedule —
# if they run at planned rate the COs won't clash.
# =============================================================

MIN_CO_GAP_HRS = 20 / 60.0   # 20-minute buffer AFTER a CO finishes before next can start
# Full rule: next_CO_start >= prev_CO_start + prev_CO_duration + MIN_CO_GAP_HRS
# Example: M1 CO = 40 min → next CO can start at minute 60 (40+20) earliest.


def _finish_time_of_co(ev, plan):
    """
    Finish time = cumulative run hours of all plan rows
    on this machine UP TO (but not including) the CO row.
    This is when the machine will call for the tool changer.
    """
    m      = ev["machine"]
    target = ev["row_after"]   # the row AFTER the CO
    total  = 0.0
    for row in plan:
        if row["Machine"] != m:
            continue
        if row is target:
            break
        total += float(row.get("Run_Hours") or 0)
        total += float(row.get("Changeover_Hrs") or 0)
    return round(total, 4)


def _last_row_before_co(ev, plan):
    """Return the plan row immediately before this CO event."""
    return ev["row_before"]


def stagger_co_by_quantity(plan, machines, scenario_id, current_inventory):
    """
    Adjusts production quantities to stagger changeover finish times
    so no two machines call for the tool changer within MIN_CO_GAP_HRS.
    Modifies plan rows in-place. Returns count of adjustments made.
    """
    events = _collect_co_events(plan, machines)
    if len(events) < 2:
        return 0   # nothing to stagger

    adjustments = 0
    max_passes  = len(events) * 2   # safety limit on iterations

    for _ in range(max_passes):
        # Recompute finish times from live plan each pass
        for ev in events:
            ev["_ft"] = _finish_time_of_co(ev, plan)

        # Sort by finish_time
        events.sort(key=lambda e: e["_ft"])

        # Find first pair violating the required gap.
        # required_gap = duration of the earlier CO + 20-min post-finish buffer
        # e.g. M1 CO = 40 min → next CO cannot start until minute 60.
        conflict = None
        for i in range(len(events) - 1):
            ft_early   = events[i]["_ft"]
            ft_late    = events[i+1]["_ft"]
            co_dur_e   = events[i]["co_duration"]
            required   = co_dur_e + MIN_CO_GAP_HRS
            gap        = ft_late - ft_early
            if gap < required - 0.001:
                conflict = (events[i], events[i+1], gap, required)
                break

        if conflict is None:
            break   # all gaps satisfied

        ev_early, ev_late, gap, required_gap = conflict
        shortfall_hrs = required_gap - gap

        # ── Try PUSH: add qty to the later machine's part before CO ──
        row_late  = _last_row_before_co(ev_late, plan)
        p_late    = row_late["Part"]
        r_late    = rate.get(p_late, 1)
        m_late    = ev_late["machine"]
        daily_l   = indent_daily.get(p_late, 0)
        inv_l     = current_inventory.get(p_late, 0)
        cap_qty   = opd_cap(scenario_id) * daily_l
        produced_l = float(row_late.get("Production_Qty") or 0)
        headroom  = max(0.0, cap_qty - inv_l)

        push_hrs  = shortfall_hrs
        push_qty  = round(push_hrs * r_late, 0)

        # Check free hours on that machine
        used_m    = sum(
            float(r.get("Changeover_Hrs") or 0) + float(r.get("Run_Hours") or 0)
            for r in plan if r["Machine"] == m_late
        )
        free_m    = max(0.0, AVAILABLE_HOURS - used_m)

        can_push = (
            push_qty <= headroom
            and push_hrs <= free_m + 0.001
            and r_late > 0
        )

        if can_push:
            row_late["Run_Hours"]      = round(float(row_late.get("Run_Hours") or 0) + push_hrs, 3)
            row_late["Production_Qty"] = round(produced_l + push_qty, 0)
            row_late["Total_Hrs_Used"] = round(
                float(row_late.get("Changeover_Hrs") or 0) + float(row_late["Run_Hours"]), 3)
            row_late["Stagger_Adjusted"] = (
                f"CO-stagger PUSH +{round(push_hrs*60,1)}min "
                f"gap={round(gap*60,1)}min need={round(required_gap*60,0):.0f}min "
                f"({round(ev_early['co_duration']*60,0):.0f}min CO + "
                f"{round(MIN_CO_GAP_HRS*60,0):.0f}min buffer)")
            current_inventory[p_late] = round(
                current_inventory.get(p_late, 0) + push_qty, 0)
            adjustments += 1
            print(f"    [CO-STAGGER PUSH]  {p_late:28s} on {m_late:15s}  "
                  f"+{round(push_hrs*60,1)}min  qty+={push_qty:.0f}  "
                  f"gap={round(gap*60,1)}min  "
                  f"need={round(required_gap*60,0):.0f}min "
                  f"({round(ev_early['co_duration']*60,0):.0f}min CO + "
                  f"{round(MIN_CO_GAP_HRS*60,0):.0f}min buffer)")
            continue

        # ── Try PULL: reduce qty of the earlier machine's part ──
        row_early = _last_row_before_co(ev_early, plan)
        p_early   = row_early["Part"]
        r_early   = rate.get(p_early, 1)
        m_early   = ev_early["machine"]
        daily_e   = indent_daily.get(p_early, 0)
        inv_e     = current_inventory.get(p_early, 0)
        produced_e = float(row_early.get("Production_Qty") or 0)

        # Must not pull below daily_indent coverage
        min_qty_e  = max(0.0, daily_e - inv_e)
        max_pull_qty = max(0.0, produced_e - min_qty_e)
        pull_hrs   = shortfall_hrs
        pull_qty   = round(pull_hrs * r_early, 0)

        can_pull = (
            pull_qty <= max_pull_qty
            and r_early > 0
            and produced_e - pull_qty >= MIN_RUN_HOURS * r_early
        )

        if can_pull:
            row_early["Run_Hours"]      = round(
                float(row_early.get("Run_Hours") or 0) - pull_hrs, 3)
            row_early["Production_Qty"] = round(produced_e - pull_qty, 0)
            row_early["Total_Hrs_Used"] = round(
                float(row_early.get("Changeover_Hrs") or 0) + float(row_early["Run_Hours"]), 3)
            row_early["Stagger_Adjusted"] = (
                f"CO-stagger PULL -{round(pull_hrs*60,1)}min "
                f"gap={round(gap*60,1)}min need={round(required_gap*60,0):.0f}min "
                f"({round(ev_early['co_duration']*60,0):.0f}min CO + "
                f"{round(MIN_CO_GAP_HRS*60,0):.0f}min buffer)")
            current_inventory[p_early] = round(
                current_inventory.get(p_early, 0) - pull_qty, 0)
            adjustments += 1
            print(f"    [CO-STAGGER PULL]  {p_early:28s} on {m_early:15s}  "
                  f"-{round(pull_hrs*60,1)}min  qty-={pull_qty:.0f}  "
                  f"gap={round(gap*60,1)}min  "
                  f"need={round(required_gap*60,0):.0f}min "
                  f"({round(ev_early['co_duration']*60,0):.0f}min CO + "
                  f"{round(MIN_CO_GAP_HRS*60,0):.0f}min buffer)")
            continue

        # Cannot resolve this conflict — log and move on
        print(f"    [CO-STAGGER SKIP]  {ev_early['machine']} / {ev_late['machine']}  "
              f"gap={round(gap*60,1)}min  need={round(required_gap*60,0):.0f}min  "
              f"push blocked (OPD cap) + pull blocked (daily indent) — unresolvable")
        # Artificially mark resolved to avoid infinite loop
        ev_late["_ft"] = ev_early["_ft"] + MIN_CO_GAP_HRS
        break

    return adjustments


def stagger_changeovers_serial_queue(plan, machines):
    """
    STRICT SERIAL QUEUE — guaranteed zero overlap, always.
    ═══════════════════════════════════════════════════════
    Mechanism (not optimisation):
      The tool changer is a single resource. It processes
      one changeover at a time, in order. The next CO cannot
      physically start until the previous one is finished.
      This is enforced by construction:

        tool_changer_free_at tracks when the tool changer
        becomes available again.

        For each CO event:
          actual_start = max(natural_start, tool_changer_free_at)
          tool_changer_free_at = actual_start + co_duration

      Since tool_changer_free_at only ever moves forward,
      and actual_start >= tool_changer_free_at always,
      two COs cannot overlap by construction — no solver,
      no constraint, no chance of failure.

    Ordering (greedy — earliest natural_start first):
      Serves the machine that needs its changeover soonest.
      This minimises total machine waiting time across the floor.

    Wait filling:
      If a machine must wait (actual_start > natural_start),
      the previous part's run is extended to fill the gap.
      The machine keeps producing — zero idle time.

    Why not CSP:
      CSP with getSolutions() enumerates all permutations and
      hangs the system with 20+ events. getSolution() finds one
      valid solution but cannot guarantee quality. The greedy
      serial queue gives the same no-overlap guarantee with
      O(n log n) complexity — runs in microseconds for any n.
    """
    print(f"\n  Tool-Changer Serial Queue Scheduler")
    print(f"  Guarantee: strict serial — no two COs can overlap by construction")

    events = _collect_co_events(plan, machines)

    if not events:
        print(f"  No changeovers in plan — tool changer idle  ✓")
        return

    # Sort by natural_start ascending — earliest-needed CO first
    events.sort(key=lambda e: e["natural_start"])

    print(f"  {len(events)} CO events queued across "
          f"{len({e['machine'] for e in events})} machines")
    print()
    print(f"  {'#':<4} {'Machine':<18} {'Part Before':<22} {'Part After':<22} "
          f"{'Dur':>5} {'Natural':>8} {'Actual':>8} {'Wait':>7} {'Fill':>6}")
    print(f"  {'─'*4} {'─'*18} {'─'*22} {'─'*22} "
          f"{'─'*5} {'─'*8} {'─'*8} {'─'*7} {'─'*6}")

    tool_changer_free_at = 0.0
    total_extra_pcs      = 0
    total_wait_min       = 0.0

    for idx, ev in enumerate(events, 1):
        # Recompute natural_start from live plan
        # (previous extensions may have shifted it)
        natural_start = _recompute_natural_start(ev, plan)
        co_h          = ev["co_duration"]

        # CORE GUARANTEE: actual_start cannot be before tool changer is free
        actual_start = max(natural_start, tool_changer_free_at)
        wait_hrs     = round(actual_start - natural_start, 4)

        # Advance the tool changer cursor — next CO starts no earlier than this
        tool_changer_free_at = actual_start + co_h

        # Fill machine wait by extending the previous part's run
        extra_pcs = 0
        if wait_hrs > 0.001:
            _, extra_pcs = _extend_row_before(ev, wait_hrs, plan)
            total_extra_pcs += extra_pcs
            total_wait_min  += wait_hrs * 60

        ev["actual_start"] = actual_start
        ev["wait_hrs"]     = wait_hrs

        wait_str  = f"+{round(wait_hrs*60,1)}m" if wait_hrs > 0.001 else "none"
        fill_str  = f"+{extra_pcs:.0f}" if extra_pcs > 0 else "—"

        print(f"  {idx:<4} {ev['machine']:<18} {ev['part_before']:<22} "
              f"{ev['part_after']:<22} "
              f"{round(co_h*60,1):>4.0f}m "
              f"{_fmt_h(natural_start):>8} "
              f"{_fmt_h(actual_start):>8} "
              f"{wait_str:>7} "
              f"{fill_str:>6}")

    n_waited = sum(1 for e in events if e.get("wait_hrs", 0) > 0.001)
    print(f"\n  Queue complete. Tool changer free at: {_fmt_h(tool_changer_free_at)}")
    print(f"  Events that had to wait : {n_waited} / {len(events)}")
    print(f"  Total wait time filled  : {round(total_wait_min, 1)} min "
          f"(converted to production, zero idle)")
    print(f"  Extra pieces produced   : {total_extra_pcs:,.0f}")
    print(f"  Overlap guarantee       : ABSOLUTE — serial queue, structurally impossible to overlap")


def stagger_changeovers(plan, machines):
    """Alias — calls the strict serial queue scheduler."""
    stagger_changeovers_serial_queue(plan, machines)

# =============================================================
# SECTION 16 — 22H UTILIZATION ENFORCER
# -------------------------------------------------------------
# Runs after primary scheduling. Processes every under-utilized
# machine (least-used first). For each machine with spare hours:
#
#   STEP 1 — Extend existing parts already on this machine
#     Extend up to OPD cap. Highest priority parts extended first.
#     Zero changeover cost — mould already loaded.
#
#   STEP 2 — Assign unplanned parts (not yet in any plan row)
#     MINIMUM CHANGEOVER SORT:
#       Tier 0: part that last ran on this machine → 0 CO cost
#               (zero-inventory parts in this tier go first)
#       Tier 1: all other parts needing a changeover
#               (critical/zero-inv parts go first within tier)
#     This ensures the loaded mould is always used before
#     introducing any new changeover on the machine.
#
#   STEP 3 — Re-run a planned part with a spare tool
#     Same minimum-CO sort as Step 2: same-machine last-run
#     part preferred first.
#
#   STEP 4 — Assign a skipped part (last resort)
#     Same minimum-CO sort: last-run part on this machine
#     preferred even among skipped parts.
#
#   STEP 5 — Log micro-idle
#     If still idle after all four steps: log to VT_Micro_Idle.
#
# CRITICAL PARTS OVERRIDE: a part with zero inventory always
# gets a machine even if a changeover is required. The CO-avoidance
# logic only affects ordering within the same urgency tier.
# =============================================================

def _add_part_to_machine(p, m, run_hrs, co_hrs, machine_hours,
                          machine_last_part, current_inventory,
                          already_planned, plan, priority_scores,
                          type_label, role_label):
    """Helper: add a plan row and update all tracking dicts."""
    r_val = rate.get(p, 1)
    qty   = round(run_hrs * r_val, 0)

    machine_hours[m]     = round(machine_hours.get(m, 0) + co_hrs + run_hrs, 4)
    current_inventory[p] = round(current_inventory.get(p, 0) + qty, 0)
    machine_last_part[m] = p
    already_planned.add(p)

    plan.append({
        "Part":             p,
        "Category":         part_category.get(p, "Stranger"),
        "Machine":          m,
        "Run_Hours":        round(run_hrs, 3),
        "Changeover_Hrs":   round(co_hrs, 3),
        "Total_Hrs_Used":   round(co_hrs + run_hrs, 3),
        "Rate_Per_Hour":    round(r_val, 2),
        "Production_Qty":   qty,
        "Monthly_Indent":   round(indent_monthly.get(p, 0), 0),
        "Daily_Indent":     round(indent_daily.get(p, 0), 2),
        "Today_Target":     round(today_target_qty.get(p, 0), 0),
        "Changeover":       "No" if co_hrs == 0 else "Yes",
        "Type":             type_label,
        "Role":             role_label,
        "Tools_Available":  tools_available.get(p, 1),
        "Tools_Used":       1,
        "Runner_Lock":      "No",
        "Priority_Score":   round(priority_scores.get(p, 0), 2),
        "Phase":            1,
        "Stagger_Adjusted": "No",
    })
    return qty


def _co_hrs_for(p, m, machine_last_part):
    """
    Changeover hours for adding part p to machine m.
    Same part   -> 0 hrs (mould already loaded).
    Same colour -> normal changeover time (no purge needed).
    Diff colour -> changeover time + 10 min colour purge.
    """
    last = machine_last_part.get(m)
    if last is None or last == p:
        return 0.0
    base_co    = vt_changeover.get(m, DEFAULT_CHANGEOVER_HRS)
    last_color = part_color.get(last, "UNKNOWN")
    new_color  = part_color.get(p,    "UNKNOWN")
    purge      = (COLOR_PURGE_HRS
                  if last_color != new_color
                  and last_color != "UNKNOWN"
                  and new_color  != "UNKNOWN"
                  else 0.0)
    return base_co + purge

def _run_hours_for(p, m, remaining, scenario_id, current_inventory):
    """
    Compute viable run hours for part p on machine m given remaining
    capacity. Returns (run_hrs, co_hrs) or (0, 0) if not viable.
    """
    co_hrs   = _co_hrs_for(p, m, machine_last_part_ref[m] if hasattr(_run_hours_for, '_dummy') else {})
    eff_free = round(remaining - co_hrs, 4)
    if eff_free < MIN_RUN_HOURS:
        return 0.0, 0.0
    daily_p  = indent_daily.get(p, 0)
    r_val    = rate.get(p, 1)
    inv_now  = current_inventory.get(p, 0)
    cap_qty  = opd_cap(scenario_id) * daily_p
    headroom = max(0.0, cap_qty - inv_now)
    if headroom <= 0:
        return 0.0, 0.0
    shortfall    = max(0.0, daily_p - inv_now)
    min_run      = max(MIN_RUN_HOURS, shortfall / r_val if r_val > 0 else MIN_RUN_HOURS)
    run_hrs      = min(eff_free, max(min_run, headroom / r_val if r_val > 0 else eff_free))
    run_hrs      = max(MIN_RUN_HOURS, min(run_hrs, eff_free))
    return run_hrs, co_hrs


def utilization_enforcer(plan, machine_hours, machine_last_part,
                          all_parts, already_planned,
                          current_inventory, scenario_id,
                          priority_scores):

    print(f"\n  22H UTILIZATION ENFORCER  (target ≥{UTIL_TARGET_PCT}%)")
    print(f"  Steps: 1=extend existing  2=unplanned parts  "
          f"3=re-run best planned  4=skipped parts")
    micro_idle_log = []

    # All skipped parts (for Step 4)
    all_skipped = [
        p for p in all_parts
        if should_skip(p)[0]
        and rate.get(p, 0) > 0
        and indent_monthly.get(p, 0) > 0
    ]

    # Process machines least-utilized first
    machines_by_util = sorted(vt_machines, key=lambda m: machine_hours.get(m, 0))

    for m in machines_by_util:
        remaining = round(AVAILABLE_HOURS - machine_hours.get(m, 0), 4)
        if remaining < 0.05:
            continue

        # ── STEP 1: Extend existing parts on this machine ────
        parts_on_machine = list({row["Part"] for row in plan if row["Machine"] == m})
        for p in sorted(parts_on_machine,
                        key=lambda x: priority_scores.get(x, 0), reverse=True):
            if remaining < 0.05:
                break
            daily_p  = indent_daily.get(p, 0)
            r_val    = rate.get(p, 1)
            inv_now  = current_inventory.get(p, 0)
            # Cap: in S3 (healthy) extend only up to 1×daily.
            # In S0/S1/S2 extend up to 3×daily buffer target.
            headroom = max(0.0, opd_cap(scenario_id) * daily_p - inv_now)
            ext_hrs  = min(remaining, headroom / r_val if r_val > 0 else 0)
            if ext_hrs < 0.05:
                continue
            extra_qty = round(ext_hrs * r_val, 0)
            for row in plan:
                if row["Part"] == p and row["Machine"] == m:
                    row["Run_Hours"]      = round(float(row.get("Run_Hours", 0)) + ext_hrs, 3)
                    row["Production_Qty"] = round(float(row.get("Production_Qty", 0)) + extra_qty, 0)
                    row["Total_Hrs_Used"] = round(
                        float(row.get("Changeover_Hrs", 0)) + float(row["Run_Hours"]), 3)
                    row["Type"] = str(row.get("Type", "Primary")) + "+Extended"
                    break
            machine_hours[m]     = round(machine_hours.get(m, 0) + ext_hrs, 4)
            current_inventory[p] = round(current_inventory.get(p, 0) + extra_qty, 0)
            remaining            = round(remaining - ext_hrs, 4)
            print(f"    [S1-EXTEND]  {p:28s} on {m:15s}  +{ext_hrs:.2f}h  qty+={extra_qty:.0f}")

        if remaining < MIN_RUN_HOURS:
            continue

        # ── STEP 2: Assign unplanned compatible parts ─────────
        # MINIMUM CHANGEOVER SORT ORDER:
        #   Tier 0 — part that last ran on THIS machine → 0 CO cost
        #            (zero inventory parts in this tier go first)
        #   Tier 1 — all other parts that need a changeover
        #            (sorted by priority score desc within tier,
        #             so critical/zero-inv parts still come first
        #             when a changeover is unavoidable)
        #
        # This ensures the filler pass uses the loaded mould first
        # before introducing any new changeover on this machine.
        unplanned = [
            p for p in all_parts
            if p not in already_planned
            and m in vt_compat.get(p, [])
            and not should_skip(p)[0]
            and indent_monthly.get(p, 0) > 0
            and rate.get(p, 0) > 0
            # Only include if still below the scenario cap
            and current_inventory.get(p, 0) < opd_cap(scenario_id) * indent_daily.get(p, 0)
        ]

        last_on_m = machine_last_part.get(m)   # part currently loaded on m

        def _co_sort_key(p):
            needs_co  = 0 if (last_on_m is None or last_on_m == p) else 1
            inv_now   = current_inventory.get(p, 0)
            is_critical = 1 if inv_now == 0 else 0   # critical = zero inventory
            score     = priority_scores.get(p, 0)
            # Within tier 0 (no CO): critical first, then score desc
            # Within tier 1 (CO needed): critical first, then score desc
            return (needs_co, -is_critical, -score)

        unplanned.sort(key=_co_sort_key)

        for p in unplanned:
            if remaining < MIN_RUN_HOURS:
                break
            co_hrs  = _co_hrs_for(p, m, machine_last_part)
            eff_free = round(remaining - co_hrs, 4)
            if eff_free < MIN_RUN_HOURS:
                continue
            daily_p  = indent_daily.get(p, 0)
            r_val    = rate.get(p, 1)
            inv_now  = current_inventory.get(p, 0)
            cap_qty  = opd_cap(scenario_id) * daily_p
            headroom = max(0.0, cap_qty - inv_now)
            if headroom <= 0:
                continue
            shortfall = max(0.0, daily_p - inv_now)
            min_run   = max(MIN_RUN_HOURS, shortfall / r_val if r_val > 0 else MIN_RUN_HOURS)
            run_hrs   = min(eff_free, max(min_run, headroom / r_val if r_val > 0 else eff_free))
            run_hrs   = max(MIN_RUN_HOURS, min(run_hrs, eff_free))

            co_tag = "No CO — same part loaded" if co_hrs == 0 else "CO"
            qty = _add_part_to_machine(
                p, m, run_hrs, co_hrs,
                machine_hours, machine_last_part, current_inventory,
                already_planned, plan, priority_scores,
                "Filler-Unplanned", "Primary")
            remaining = round(remaining - co_hrs - run_hrs, 4)
            print(f"    [S2-UNPLAN]  {p:28s} → {m:15s}  {run_hrs:.2f}h  "
                  f"qty={qty:.0f}  [{co_tag}]")

        if remaining < MIN_RUN_HOURS:
            continue

        # ── STEP 3: Re-run best planned part (largest indent, lowest inv)
        # Only if the part has spare tools not yet used on this machine
        planned_parts_elsewhere = [
            p for p in already_planned
            if m in vt_compat.get(p, [])
            and m not in [row["Machine"] for row in plan if row["Part"] == p]
        ]
        # Filter: must have a spare tool available
        planned_parts_elsewhere = [
            p for p in planned_parts_elsewhere
            if tools_available.get(p, 1) > len({
                row["Machine"] for row in plan if row["Part"] == p
            })
        ]
        # Sort: no-CO parts first (last ran on this machine),
        # then critical (zero inv), then priority score desc
        planned_parts_elsewhere.sort(key=lambda p: (
            0 if (machine_last_part.get(m) is None or machine_last_part.get(m) == p) else 1,
            1 if current_inventory.get(p, 0) == 0 else 0,   # critical first (negate)
            -priority_scores.get(p, 0)
        ))

        for p in planned_parts_elsewhere:
            if remaining < MIN_RUN_HOURS:
                break
            co_hrs   = _co_hrs_for(p, m, machine_last_part)
            eff_free = round(remaining - co_hrs, 4)
            if eff_free < MIN_RUN_HOURS:
                continue
            daily_p  = indent_daily.get(p, 0)
            r_val    = rate.get(p, 1)
            inv_now  = current_inventory.get(p, 0)
            cap_qty  = opd_cap(scenario_id) * daily_p
            headroom = max(0.0, cap_qty - inv_now)
            if headroom <= 0:
                continue
            shortfall = max(0.0, daily_p - inv_now)
            min_run   = max(MIN_RUN_HOURS, shortfall / r_val if r_val > 0 else MIN_RUN_HOURS)
            run_hrs   = min(eff_free, max(min_run, headroom / r_val if r_val > 0 else eff_free))
            run_hrs   = max(MIN_RUN_HOURS, min(run_hrs, eff_free))

            # Don’t add to already_planned (part is already there)
            # Just append the extra run row directly
            r_val2  = rate.get(p, 1)
            qty     = round(run_hrs * r_val2, 0)
            machine_hours[m]     = round(machine_hours.get(m, 0) + co_hrs + run_hrs, 4)
            current_inventory[p] = round(current_inventory.get(p, 0) + qty, 0)
            machine_last_part[m] = p
            remaining            = round(remaining - co_hrs - run_hrs, 4)

            tools_used_now = len({row["Machine"] for row in plan if row["Part"] == p}) + 1
            plan.append({
                "Part":             p,
                "Category":         part_category.get(p, "Stranger"),
                "Machine":          m,
                "Run_Hours":        round(run_hrs, 3),
                "Changeover_Hrs":   round(co_hrs, 3),
                "Total_Hrs_Used":   round(co_hrs + run_hrs, 3),
                "Rate_Per_Hour":    round(r_val2, 2),
                "Production_Qty":   qty,
                "Monthly_Indent":   round(indent_monthly.get(p, 0), 0),
                "Daily_Indent":     round(daily_p, 2),
                "Today_Target":     round(today_target_qty.get(p, 0), 0),
                "Changeover":       "No" if co_hrs == 0 else "Yes",
                "Type":             "Re-run (spare tool)",
                "Role":             f"Tool-Expansion (tool {tools_used_now})",
                "Tools_Available":  tools_available.get(p, 1),
                "Tools_Used":       tools_used_now,
                "Runner_Lock":      "No",
                "Priority_Score":   round(priority_scores.get(p, 0), 2),
                "Phase":            3,
                "Stagger_Adjusted": "No",
            })
            print(f"    [S3-RERUN]   {p:28s} → {m:15s}  {run_hrs:.2f}h  "
                  f"qty={qty:.0f}  tool {tools_used_now}/{tools_available.get(p,1)}")
            break   # one re-run per machine per pass

        if remaining < MIN_RUN_HOURS:
            continue

        # ── STEP 4: Assign a skipped part (last resort) ───────
        # Among compatible skipped parts: largest indent, lowest inv
        skipped_candidates = [
            p for p in all_skipped
            if m in vt_compat.get(p, [])
            and rate.get(p, 0) > 0
        ]
        skipped_candidates.sort(key=lambda p: (
            0 if (machine_last_part.get(m) is None or machine_last_part.get(m) == p) else 1,
            1 if current_inventory.get(p, 0) == 0 else 0,
            -indent_daily.get(p, 0)
        ))

        for p in skipped_candidates:
            if remaining < MIN_RUN_HOURS:
                break
            co_hrs   = _co_hrs_for(p, m, machine_last_part)
            eff_free = round(remaining - co_hrs, 4)
            if eff_free < MIN_RUN_HOURS:
                continue
            r_val   = rate.get(p, 1)
            inv_now = current_inventory.get(p, 0)
            daily_p = indent_daily.get(p, 0)
            # For skipped parts use remaining hours directly (no OPD cap)
            run_hrs = min(eff_free, max(MIN_RUN_HOURS,
                          indent_monthly.get(p, 0) / r_val if r_val > 0 else eff_free))
            run_hrs = max(MIN_RUN_HOURS, min(run_hrs, eff_free))

            qty = _add_part_to_machine(
                p, m, run_hrs, co_hrs,
                machine_hours, machine_last_part, current_inventory,
                already_planned, plan, priority_scores,
                "Filler-Skipped (last resort)", "Primary")
            remaining = round(remaining - co_hrs - run_hrs, 4)
            skip_reason = should_skip(p)[1]
            print(f"    [S4-SKIPPED] {p:28s} → {m:15s}  {run_hrs:.2f}h  "
                  f"qty={qty:.0f}  [{skip_reason[:35]}]")
            break   # one skipped part per machine per pass

        # ── STEP 5: Log micro-idle ────────────────────────────
        final_remaining = round(AVAILABLE_HOURS - machine_hours.get(m, 0), 4)
        if final_remaining >= 0.25:
            util_final = round((1 - final_remaining / AVAILABLE_HOURS) * 100, 1)
            if util_final < UTIL_TARGET_PCT:
                micro_idle_log.append({
                    "Machine":         m,
                    "Idle_Hrs":        round(final_remaining, 3),
                    "Utilization_Pct": util_final,
                    "Note":            ("No unplanned/skipped compatible part available "
                                        "and all planned parts at OPD cap"),
                })
                print(f"    [⚠ IDLE]     {m:15s}  {final_remaining:.2f}h idle "
                      f"({util_final}%) — all options exhausted")

    return micro_idle_log

# =============================================================
# SECTION 17 — NEW OUTPUT: MULTI-MACHINE PARTS VIEW
# =============================================================

def build_multi_machine_view(plan):
    """
    Returns a DataFrame showing only parts that were assigned
    to more than one machine, with one row per machine assignment.
    Includes a summary row per part showing totals.
    """
    if not plan:
        return pd.DataFrame()

    # Group plan rows by part
    from collections import defaultdict
    part_rows = defaultdict(list)
    for row in plan:
        part_rows[row["Part"]].append(row)

    # Keep only parts with multiple machine assignments
    multi = {p: rows for p, rows in part_rows.items() if len(rows) > 1}

    if not multi:
        return pd.DataFrame()

    output_rows = []
    for part, rows in sorted(multi.items(),
                              key=lambda x: -sum(r["Production_Qty"] for r in x[1])):
        daily  = indent_daily.get(part, 0)
        total_qty = sum(float(r["Production_Qty"]) for r in rows)

        for row in rows:
            qty_this   = float(row["Production_Qty"])
            run_h      = float(row["Run_Hours"])
            output_rows.append({
                "Part":                  part,
                "Category":              part_category.get(part, "Stranger"),
                "Tools_Available":       tools_available.get(part, 1),
                "Machines_Used":         len(rows),
                "Machine":               row["Machine"],
                "Role":                  row.get("Role", "Primary"),
                "Run_Hours":             round(run_h, 2),
                "Changeover_Hrs":        round(float(row.get("Changeover_Hrs", 0)), 2),
                "Production_Qty":        round(qty_this, 0),
                "Daily_Indent":          round(daily, 2),
                "Total_Qty_All_Machines":round(total_qty, 0),
                "Type":                  row.get("Type", "—"),
            })

        # Summary row for this part
        output_rows.append({
            "Part":                  f"  ↳ TOTAL — {part}",
            "Category":              "—",
            "Tools_Available":       tools_available.get(part, 1),
            "Machines_Used":         len(rows),
            "Machine":               f"{len(rows)} machines",
            "Role":                  "TOTAL",
            "Run_Hours":             round(sum(float(r["Run_Hours"]) for r in rows), 2),
            "Changeover_Hrs":        round(sum(float(r.get("Changeover_Hrs",0)) for r in rows), 2),
            "Production_Qty":        round(total_qty, 0),
            "Daily_Indent":          round(daily, 2),
            "Total_Qty_All_Machines":round(total_qty, 0),
            "Type":                  "—",
        })
        # Blank spacer
        output_rows.append({k: "" for k in output_rows[-1].keys()})

    return pd.DataFrame(output_rows)

# =============================================================
# SECTION 18 — NEW OUTPUT: PRODUCTION VS INDENT VIEW
# =============================================================

def build_production_vs_indent(plan, all_parts):
    """
    Every planned part: one row showing total production across
    all machines vs daily indent, gap direction, and extra days stock.
    """
    if not plan:
        return pd.DataFrame()

    from collections import defaultdict
    part_qty     = defaultdict(float)
    part_machines = defaultdict(list)

    for row in plan:
        p = row["Part"]
        part_qty[p]      += float(row.get("Production_Qty", 0))
        part_machines[p].append(row["Machine"])

    rows = []
    for p in sorted(part_qty.keys()):
        daily    = indent_daily.get(p, 0)
        monthly  = indent_monthly.get(p, 0)
        inv_b    = inventory.get(p, 0)
        produced = round(part_qty[p], 0)
        inv_after = round(inv_b + produced, 0)
        gap      = round(produced - daily, 0)   # +ve = over, -ve = under
        gap_dir  = "OVER" if gap > 0 else ("UNDER" if gap < 0 else "MET")

        # Extra days stock = how many days the produced qty covers
        # beyond the daily indent
        extra_days = round(gap / daily, 2) if daily > 0 and gap > 0 else 0.0

        machines_str = ", ".join(dict.fromkeys(part_machines[p]))  # unique, ordered

        rows.append({
            "Part":                  p,
            "Category":              part_category.get(p, "Stranger"),
            "Tools_Available":       tools_available.get(p, 1),
            "Machines":              machines_str,
            "Machines_Count":        len(set(part_machines[p])),
            "Total_Qty_Produced":    produced,
            "Daily_Indent":          round(daily, 2),
            "Monthly_Indent":        round(monthly, 0),
            "Gap_vs_Daily":          gap,      # +ve = over, -ve = under
            "Gap_Direction":         gap_dir,
            "Extra_Days_Stock":      extra_days,
            "Inventory_Before":      round(inv_b, 0),
            "Inventory_After":       inv_after,
            "Days_Coverage_After":   round(inv_after / daily, 2) if daily > 0 else 0,
        })

    # Sort: UNDER first (most urgent), then MET, then OVER
    order_map = {"UNDER": 0, "MET": 1, "OVER": 2}
    df = pd.DataFrame(rows)
    if not df.empty:
        df["_sort"] = df["Gap_Direction"].map(order_map)
        df = df.sort_values(["_sort", "Gap_vs_Daily"]).drop(columns=["_sort"])
        df = df.reset_index(drop=True)
    return df

# =============================================================
# SECTION 19 — INDENT HORIZON TABLE
# =============================================================

def compute_indent_horizon(parts):
    rows = []
    for p in parts:
        inv     = inventory.get(p, 0.0)
        monthly = indent_monthly.get(p, 0.0)
        daily   = indent_daily.get(p, 0.0)
        target  = today_target_qty.get(p, 0.0)
        r       = rate.get(p, 1.0)
        skip, skip_reason = should_skip(p)
        indent_hrs = monthly / r if r > 0 else 0.0

        if inv == 0.0 and monthly > 0:
            status = "ZERO INV — FORCED"
        elif skip:
            status = "SKIPPED"
        elif target > 0:
            status = "PRODUCTION NEEDED"
        elif monthly == 0:
            status = "NO INDENT"
        else:
            status = "INV SUFFICIENT"

        rows.append({
            "Part":             p,
            "Tools":            tools_available.get(p, 1),
            "Monthly_Indent":   round(monthly, 0),
            "Indent_Hrs_Total": round(indent_hrs, 2),
            "Working_Days":     WORKING_DAYS,
            "Daily_Indent":     round(daily, 2),
            "Inventory_Now":    round(inv, 0),
            "Today_Target_Qty": round(target, 0),
            "Today_Target_Hrs": round(target / r if r > 0 else 0, 2),
            "Rate_Per_Hour":    round(r, 2),
            "Indent_Status":    status,
            "Skip_Reason":      skip_reason,
        })
    return pd.DataFrame(rows)

# =============================================================
# SECTION 20 — SPECIALIZED MACHINE SUMMARY  (audit only)
# =============================================================
# detect_specialized_machines() is kept for console reporting
# only. The actual machine preference is now handled inside
# rank_machines() using the Part_Count from VT_Machine_Part_Count.
# No separate Pass 0 needed — ranking naturally sends each part
# to its most constrained compatible machine first.
# =============================================================

SPECIALIZED_MACHINE_THRESHOLD = 3   # ≤ this many parts → shown as specialized

def detect_specialized_machines(all_parts):
    """Audit helper — returns specialized machines for console display."""
    machine_parts = {}
    for m in vt_machines:
        compatible_parts = [
            p for p in all_parts
            if m in vt_compat.get(p, [])
            and not should_skip(p)[0]
            and indent_monthly.get(p, 0) > 0
        ]
        machine_parts[m] = compatible_parts

    specialized_machines = {
        m for m, mparts in machine_parts.items()
        if 0 < len(mparts) <= SPECIALIZED_MACHINE_THRESHOLD
    }
    spec_machine_parts = {m: machine_parts[m] for m in specialized_machines}
    return specialized_machines, spec_machine_parts, machine_parts


# =============================================================
# SECTION 21 — MAIN SCHEDULER
# =============================================================

def schedule(parts, label=""):

    print(f"\n{'─'*65}")
    print(f"  {label}  |  {len(parts)} parts  |  {len(vt_machines)} machines")
    print(f"{'─'*65}")

    scenario_id, scenario_desc = classify_scenario(parts)
    print(f"  {scenario_desc}")

    horizon_df = compute_indent_horizon(parts)

    # Active parts — parts that need production today.
    # A part is active if its inventory is below the scenario cap:
    #   S0/S1/S2 → below 5-day full target (3 buffer + 2 safety)
    #   S3       → below 1 day (produce daily indent to maintain)
    # Note: in S3, cap = 1×daily, so only parts below 1 day of
    # stock are active. Parts with ≥1 day are skipped today.
    cap = opd_cap(scenario_id)
    active_parts = [
        p for p in parts
        if not should_skip(p)[0]
        and indent_monthly.get(p, 0) > 0
        and inventory.get(p, 0) < cap * indent_daily.get(p, 0)
    ]

    priority_scores, score_rows = compute_priority_scores(active_parts)
    score_df = pd.DataFrame(score_rows) if score_rows else pd.DataFrame()

    machine_hours     = {m: 0.0 for m in vt_machines}
    machine_last_part = {m: machine_state.get(m) for m in vt_machines}
    current_inventory = inventory.copy()
    plan              = []
    already_planned   = set()
    not_planned       = []
    deferred          = []

    # ── Show specialized machine summary (audit only) ────────
    # The actual specialized-machine preference is built into
    # rank_machines() via Part_Count — no separate pass needed.
    specialized_machines, spec_machine_parts, machine_part_map =         detect_specialized_machines(list(parts))

    print(f"\n  SPECIALIZED MACHINES (≤{SPECIALIZED_MACHINE_THRESHOLD} parts, "
          f"ranked first by rank_machines):")
    if specialized_machines:
        for m in sorted(specialized_machines,
                        key=lambda m: machine_part_count.get(m, 99)):
            mparts  = spec_machine_parts.get(m, [])
            cnt     = machine_part_count.get(m, "?")
            print(f"    {m:<25} Part_Count={cnt:<4} "
                  f"parts: {', '.join(mparts)}")
    else:
        print(f"    None — all machines have >3 compatible parts")

    # ── PRIMARY SCHEDULING PASS ───────────────────────────────
    # All active parts sorted by priority score descending.
    # rank_machines() automatically prefers the machine with the
    # lowest Part_Count for each part — so constrained machines
    # are claimed first without a separate pass.
    sorted_active = sorted(active_parts,
                           key=lambda p: priority_scores.get(p, 0), reverse=True)

    print(f"\n  PRIMARY SCHEDULING PASS  ({len(sorted_active)} active parts)")
    print(f"  {'Part':<30} {'Score':>6} {'Tools':>5} {'Machine(s)':<30} "
          f"{'Run':>5} {'Qty':>8}  Status")
    print(f"  {'─'*95}")

    for part in sorted_active:
        inv_now  = current_inventory.get(part, 0)
        daily    = indent_daily.get(part, 0)
        monthly  = indent_monthly.get(part, 0)
        score    = priority_scores.get(part, 0)
        tools    = tools_available.get(part, 1)
        category = part_category.get(part, "Stranger")

        if monthly == 0:
            deferred.append({"Part": part, "Category": category,
                             "Reason": "Monthly indent = 0"})
            print(f"  {part:<30} {score:>6.1f} {tools:>5}  {'—':<30}  DEFERRED (no indent)")
            continue

        if not vt_compat.get(part):
            not_planned.append({
                "Part": part, "Category": category, "Score": score,
                "Daily_Indent": round(daily, 2), "Inventory_Now": round(inv_now, 0),
                "Tools": tools, "Compatible_Machines": "NONE DEFINED",
                "Reason": "Not in VT_Matrix", "Action_Needed": "Add to VT_Matrix",
            })
            print(f"  {part:<30} {score:>6.1f} {tools:>5}  {'—':<30}  ✗ NOT IN MATRIX")
            continue

        new_rows = assign_part(
            part, scenario_id, machine_hours, machine_last_part,
            current_inventory, plan, already_planned, priority_scores)

        if new_rows:
            plan.extend(new_rows)
            machines_used = [r["Machine"] for r in new_rows]
            total_qty     = sum(float(r["Production_Qty"]) for r in new_rows)
            total_run     = sum(float(r["Run_Hours"]) for r in new_rows)
            machines_str  = ", ".join(machines_used)
            flag = " [MULTI-MACHINE]" if len(new_rows) > 1 else ""
            flag += " [ZERO-INV]" if inv_now == 0 else ""
            print(f"  {part:<30} {score:>6.1f} {tools:>5}  {machines_str:<30}  "
                  f"{total_run:>5.2f}  {total_qty:>8.0f}  ✓{flag}")
        else:
            inv_days = inv_now / daily if daily > 0 else 999
            free_map = {m: round(AVAILABLE_HOURS - machine_hours.get(m, 0), 2)
                        for m in vt_compat.get(part, []) if m in machine_hours}
            not_planned.append({
                "Part":                part,
                "Category":            category,
                "Score":               score,
                "Tools":               tools,
                "Daily_Indent":        round(daily, 2),
                "Inventory_Now":       round(inv_now, 0),
                "Inv_Days_Coverage":   round(inv_days, 2),
                "Today_Target":        round(today_target_qty.get(part, 0), 0),
                "Compatible_Machines": ", ".join(vt_compat.get(part, [])),
                "Machine_Free_Hrs":    str(free_map),
                "Reason":              "No compatible machine has capacity",
                "Action_Needed":       "Review matrix or add machines",
            })
            print(f"  {part:<30} {score:>6.1f} {tools:>5}  {'—':<30}  ✗ NO CAPACITY")

    # 22H Utilization Enforcer
    micro_idle = utilization_enforcer(
        plan, machine_hours, machine_last_part,
        list(parts), already_planned,
        current_inventory, scenario_id, priority_scores)

    # ── Quantity-based CO stagger ────────────────────────────
    # Adjust quantities so changeovers fall at different times
    # naturally — baked into the plan, not imposed as a schedule.
    print(f"\n  CO Quantity Stagger  (min gap = {MIN_CO_GAP_HRS*60:.0f} min)")
    n_adj = stagger_co_by_quantity(
        plan, vt_machines, scenario_id, current_inventory)
    if n_adj == 0:
        print(f"    No adjustments needed — all CO gaps already ≥ "
              f"{MIN_CO_GAP_HRS*60:.0f} min  ✓")
    else:
        print(f"    {n_adj} quantity adjustment(s) made to stagger COs")

    # ── Serial queue (guarantee no overlap at execution) ─────
    # Belt-and-suspenders: even with quantity staggering, the
    # serial queue ensures that if the floor deviates slightly
    # from planned quantities the tool changer still never
    # gets called on two machines at once.
    stagger_changeovers(plan, vt_machines)

    # ── Build output views ────────────────────────────────────
    multi_machine_df     = build_multi_machine_view(plan)
    prod_vs_indent_df    = build_production_vs_indent(plan, list(parts))

    # Daily indent status
    indent_status_rows = []
    for row in plan:
        p       = row["Part"]
        planned = float(row.get("Production_Qty") or 0)
        daily   = indent_daily.get(p, 0)
        r_val   = float(row.get("Rate_Per_Hour") or rate.get(p, 0))
        inv_b   = inventory.get(p, 0)
        meets   = planned >= daily
        indent_status_rows.append({
            "Part":               p,
            "Category":           part_category.get(p, "Stranger"),
            "Machine":            row.get("Machine", "—"),
            "Role":               row.get("Role", "Primary"),
            "Priority_Score":     round(row.get("Priority_Score", 0), 2),
            "Run_Hours":          round(float(row.get("Run_Hours") or 0), 2),
            "Planned_Qty":        round(planned, 0),
            "Daily_Indent":       round(daily, 2),
            "Gap_vs_Daily":       round(daily - planned, 0),
            "Meets_Daily_Indent": "YES ✓" if meets else "NO ✗",
            "Inventory_Before":   round(inv_b, 0),
            "Total_Available":    round(inv_b + planned, 0),
            "Covers_With_Inv":    "YES ✓" if (inv_b + planned) >= daily else "NO ✗",
        })
    indent_status_df = pd.DataFrame(indent_status_rows)
    if not indent_status_df.empty:
        indent_status_df = indent_status_df.sort_values(
            ["Meets_Daily_Indent", "Gap_vs_Daily"],
            ascending=[True, False]).reset_index(drop=True)

    # Inventory health
    inv_rows = []
    for p in parts:
        inv_b    = inventory.get(p, 0)
        daily    = indent_daily.get(p, 0)
        produced = sum(float(r["Production_Qty"]) for r in plan if r["Part"] == p)
        inv_after = inv_b + produced
        days_cov  = inv_after / daily if daily > 0 else 0
        inv_rows.append({
            "Part":            p,
            "Tools":           tools_available.get(p, 1),
            "Rate_Per_Hour":   round(rate.get(p, 0), 2),
            "Monthly_Indent":  round(indent_monthly.get(p, 0), 0),
            "Daily_Indent":    round(daily, 2),
            "Inv_Before":      round(inv_b, 0),
            "Produced_Today":  round(produced, 0),
            "Inv_After_Today": round(inv_after, 0),
            "Days_Coverage":   round(days_cov, 2),
            "Status":          ("OK"       if days_cov >= TARGET_DAYS_INV
                                else "LOW"  if days_cov >= 1
                                else "CRITICAL"),
        })

    # Machine utilization
    mach_rows = []
    for m in vt_machines:
        used      = machine_hours.get(m, 0)
        parts_run = list({r["Part"] for r in plan if r["Machine"] == m})
        co_count  = sum(1 for r in plan if r["Machine"] == m
                        and r.get("Changeover") == "Yes")
        util_pct  = round(used / AVAILABLE_HOURS * 100, 1)
        mach_rows.append({
            "Machine":              m,
            "Used_Hours":           round(used, 2),
            "Unused_Hours":         round(AVAILABLE_HOURS - used, 2),
            "Utilization_%":        util_pct,
            "Status":               ("FULL"     if used >= AVAILABLE_HOURS - 0.3 else
                                     "GOOD"     if used >= AVAILABLE_HOURS * 0.98 else
                                     "PARTIAL"  if used >= AVAILABLE_HOURS * 0.85 else
                                     "UNDERUSED"),
            "Parts_Planned":        len(parts_run),
            "Changeovers":          co_count,
            "Last_Part_Run":        machine_last_part.get(m) or "—",
            "All_Parts_On_Machine": ", ".join(parts_run) if parts_run else "— idle —",
        })

    micro_df = pd.DataFrame(micro_idle)     if micro_idle else pd.DataFrame()
    plan_df  = pd.DataFrame(plan)           if plan       else pd.DataFrame()
    def_df   = pd.DataFrame(deferred)       if deferred   else pd.DataFrame()
    not_df   = pd.DataFrame(not_planned)    if not_planned else pd.DataFrame()
    mach_df  = pd.DataFrame(mach_rows)
    inv_df   = pd.DataFrame(inv_rows)

    if not plan_df.empty:
        plan_df.insert(0, "Planning_Date", str(PLANNING_DATE))
        plan_df.insert(1, "Scenario",      scenario_desc)
        plan_df.insert(2, "Working_Days",  WORKING_DAYS)

    # Summary
    multi_count = len(multi_machine_df[
        multi_machine_df.get("Role", pd.Series()) == "TOTAL"
    ]) if not multi_machine_df.empty else 0

    print(f"\n  {'='*65}")
    print(f"  SCHEDULE SUMMARY — {scenario_desc}")
    print(f"    Parts planned          : {len(already_planned)}")
    print(f"    Parts on multi-machine : {multi_count}")
    print(f"    Not planned            : {len(not_planned)}")
    print(f"    Deferred               : {len(deferred)}")
    if not mach_df.empty:
        print(f"    Avg utilization        : {mach_df['Utilization_%'].mean():.1f}%")
        print(f"    UNDERUSED machines     : "
              f"{(mach_df['Status']=='UNDERUSED').sum()}")
    print(f"  {'='*65}")

    return (plan_df, def_df, not_df, mach_df, inv_df,
            machine_last_part, horizon_df, indent_status_df,
            score_df, micro_df, multi_machine_df, prod_vs_indent_df)

# =============================================================
# SECTION 21 — PART AUDIT
# =============================================================

vt_parts = data_valid[
    data_valid["Material"].isin(vt_matrix["Part"])
]["Material"].unique()

all_vt_parts_raw = list(data["Material"].unique())
matrix_parts     = set(str(p).strip() for p in vt_matrix["Part"] if pd.notna(p))
zero_rate_set    = set(data_zero_rate["Material"].unique())

audit_rows = []
for part in all_vt_parts_raw:
    inv     = inventory.get(part, 0.0)
    r_val   = rate.get(part, None)
    monthly = indent_monthly.get(part, 0.0)
    daily   = indent_daily.get(part, 0.0)
    row_data = data[data["Material"] == part]
    ct_raw   = row_data[vt_col_cycletime].values[0] if len(row_data) else "—"
    cv_raw   = row_data[vt_col_cavity].values[0]    if len(row_data) else "—"
    tools    = tools_available.get(part, 1)

    if part in zero_rate_set or r_val is None:
        status, gate, reason = "ZERO/MISSING CYCLE TIME", "GATE 1", f"Cycle time={ct_raw}"
    elif part not in matrix_parts:
        status, gate, reason = "NOT IN VT_MATRIX", "GATE 2", "No compatible machine"
    elif monthly == 0:
        status, gate, reason = "ZERO/MISSING INDENT", "GATE 3", "Monthly indent = 0"
    elif should_skip(part)[0]:
        status, gate, reason = "SKIPPED (LOW INDENT / TRIVIAL RUN)", "GATE 4", should_skip(part)[1]
    elif daily > 0 and inv >= daily:
        status, gate, reason = "NOT REQUIRED — INV SUFFICIENT", "GATE 5", \
            f"Inventory ({inv:.0f}) ≥ daily_indent ({daily:.2f})"
    else:
        status, gate, reason = "ENTERS SCHEDULER", "—", "Passed all gates"

    audit_rows.append({
        "Part":          part,
        "Gate_Failed":   gate,
        "Reason":        reason,
        "Monthly_Indent":round(monthly, 0),
        "Daily_Indent":  round(daily, 2),
        "Inventory":     round(inv, 0),
        "Tools":         tools,
        "Rate_Per_Hour": round(r_val, 2) if r_val else "—",
        "Cycle_Time":    ct_raw,
        "Cavity":        cv_raw,
        "Status":        status,
    })

audit_df = pd.DataFrame(audit_rows)
gate_counts = audit_df["Status"].value_counts()
print(f"\n  Part audit ({len(all_vt_parts_raw)} total):")
for status, count in gate_counts.items():
    marker = "✓" if status == "ENTERS SCHEDULER" else "·"
    print(f"    {marker}  {status:<45}: {count:>4}")

# =============================================================
# SECTION 22 — RUN
# =============================================================

(vt_plan, vt_def, vt_not, vt_mach, vt_inv,
 vt_state, vt_horizon, vt_indent_status,
 vt_scores, vt_micro, vt_multi_machine,
 vt_prod_vs_indent) = schedule(vt_parts, "VT Machines")

save_machine_state(vt_state)

# =============================================================
# SECTION 23 — MACHINE-WISE PLAN
# =============================================================

def build_machine_wise_plan(plan_df):
    """
    Machine-wise quantity plan.

    We plan WHAT runs on each machine and HOW MUCH.
    The floor decides the sequence.

    One row per part per machine.
    Summary footer per machine showing totals.
    No start/end times — no sequence imposed.
    """
    if plan_df.empty:
        return pd.DataFrame()

    def sf(val, default=0.0):
        try:
            v = float(val)
            return v if not np.isnan(v) else default
        except (TypeError, ValueError):
            return default

    rows = []
    for m in vt_machines:
        machine_rows = plan_df[plan_df["Machine"] == m].copy()
        if machine_rows.empty:
            continue

        for _, pr in machine_rows.iterrows():
            p     = pr.get("Part", "—")
            co_h  = sf(pr.get("Changeover_Hrs", 0))
            run_h = sf(pr.get("Run_Hours", 0))
            r_val = sf(pr.get("Rate_Per_Hour", 0))
            rows.append({
                "Machine":            m,
                "Part":               p,
                "Category":           part_category.get(p, "Stranger"),
                "Role":               pr.get("Role", "Primary"),
                "Tools_Available":    pr.get("Tools_Available", 1),
                "Priority_Score":     round(sf(pr.get("Priority_Score", 0)), 2),
                "Rate_Per_Hour":      round(r_val, 2),
                "Run_Hours":          round(run_h, 2),
                "Changeover_Hrs":     round(co_h, 3),
                "Changeover_Needed":  pr.get("Changeover", "No") or "No",
                "Production_Qty":     round(sf(pr.get("Production_Qty", 0)), 0),
                "Daily_Indent":       round(sf(pr.get("Daily_Indent", 0)), 2),
                "Today_Target":       round(sf(pr.get("Today_Target", 0)), 0),
                "Monthly_Indent":     round(sf(pr.get("Monthly_Indent", 0)), 0),
                "Type":               pr.get("Type", "Primary") or "Primary",
                "Row_Type":           "Part",
            })

        # Summary footer per machine
        co_total  = machine_rows["Changeover_Hrs"].apply(lambda x: sf(x, 0)).sum()
        run_total = machine_rows["Run_Hours"].apply(lambda x: sf(x, 0)).sum()
        qty_total = machine_rows["Production_Qty"].apply(lambda x: sf(x, 0)).sum()
        co_count  = int(machine_rows["Changeover"].eq("Yes").sum())
        hrs_total = round(co_total + run_total, 2)

        rows.append({
            "Machine":            m,
            "Part":               f"TOTAL — {m}",
            "Category":           "—",
            "Role":               "—",
            "Tools_Available":    "—",
            "Priority_Score":     "—",
            "Rate_Per_Hour":      "—",
            "Run_Hours":          round(run_total, 2),
            "Changeover_Hrs":     round(co_total, 2),
            "Changeover_Needed":  f"{co_count} changeover(s)",
            "Production_Qty":     round(qty_total, 0),
            "Daily_Indent":       "—",
            "Today_Target":       "—",
            "Monthly_Indent":     "—",
            "Type":               (f"Total {hrs_total}h used / {AVAILABLE_HOURS}h  |  "
                                   f"Idle {round(AVAILABLE_HOURS - hrs_total, 2)}h  |  "
                                   f"Util {round(hrs_total / AVAILABLE_HOURS * 100, 1)}%"),
            "Row_Type":           "Summary",
        })
        rows.append({k: "" for k in rows[-1].keys()})

    return pd.DataFrame(rows)


def build_co_queue(plan, machines):
    """
    Tool-Changer Queue sheet.

    Shows the safe order in which changeovers should be called
    to the tool-change team. This guarantees no two COs happen
    at the same time. The floor uses this as a reference —
    not a rigid time schedule.

    Columns:
      Queue_Position | Machine | Part_Before | Part_After |
      CO_Duration_Min | Note
    """
    events = _collect_co_events(plan, machines)
    if not events:
        return pd.DataFrame()

    # Sort by natural_start — same greedy order used in the scheduler
    events.sort(key=lambda e: e["natural_start"])

    rows = []
    tool_changer_free_at = 0.0

    for pos, ev in enumerate(events, 1):
        natural_start = _recompute_natural_start(ev, plan)
        co_h          = ev["co_duration"]
        actual_start  = max(natural_start, tool_changer_free_at)
        wait_min      = round((actual_start - natural_start) * 60, 1)
        tool_changer_free_at = actual_start + co_h

        note = ""
        if wait_min > 0:
            note = (f"Machine may need to wait ~{wait_min}min for tool changer. "
                    f"Continue running previous part until team arrives.")

        rows.append({
            "Queue_Position":    pos,
            "Machine":           ev["machine"],
            "Part_Before":       ev["part_before"],
            "Part_After":        ev["part_after"],
            "CO_Duration_Min":   round(co_h * 60, 1),
            "Note":              note if note else "Tool changer available immediately",
        })

    return pd.DataFrame(rows)

# =============================================================
# SECTION 24 — EXCEL OUTPUT
# =============================================================

from openpyxl import load_workbook
from openpyxl.styles import PatternFill, Font, Alignment
from openpyxl.utils import get_column_letter

HEADER_COLORS = {
    "VT_Plan_By_Machine":      "0D6E6E",
    "VT_CO_Queue":             "375623",
    "VT_Plan":                 "1F4E79",
    "VT_Daily_Indent_Status":  "0F4C2A",
    "VT_Multi_Machine_Parts":  "4A235A",
    "VT_Production_vs_Indent": "154360",
    "VT_Priority_Scores":      "2C4770",
    "VT_Machine_Util":         "375623",
    "VT_Not_Planned":          "7B2C2C",
    "VT_Deferred":             "7F6000",
    "VT_Inventory_Health":     "4A235A",
    "VT_Indent_Horizon":       "154360",
    "VT_Part_Audit":           "1C3557",
    "VT_Micro_Idle":           "5C3D2E",
}

STATUS_FILLS = {
    "FULL":     PatternFill("solid", fgColor="C6EFCE"),
    "GOOD":     PatternFill("solid", fgColor="DDEBF7"),
    "PARTIAL":  PatternFill("solid", fgColor="FFEB9C"),
    "UNDERUSED":PatternFill("solid", fgColor="FFC7CE"),
    "OK":       PatternFill("solid", fgColor="C6EFCE"),
    "LOW":      PatternFill("solid", fgColor="FFEB9C"),
    "CRITICAL": PatternFill("solid", fgColor="FFC7CE"),
    "YES ✓":    PatternFill("solid", fgColor="C6EFCE"),
    "NO ✗":     PatternFill("solid", fgColor="FFC7CE"),
    "OVER":     PatternFill("solid", fgColor="DDEBF7"),
    "UNDER":    PatternFill("solid", fgColor="FFC7CE"),
    "MET":      PatternFill("solid", fgColor="C6EFCE"),
    "PRODUCTION NEEDED":        PatternFill("solid", fgColor="FFC7CE"),
    "ZERO INV — FORCED":        PatternFill("solid", fgColor="FFD7D7"),
    "INV SUFFICIENT":           PatternFill("solid", fgColor="C6EFCE"),
    "SKIPPED":                  PatternFill("solid", fgColor="EDEDED"),
    "NOT REQUIRED — INV SUFFICIENT":     PatternFill("solid", fgColor="DDEBF7"),
    "SKIPPED (LOW INDENT / TRIVIAL RUN)":PatternFill("solid", fgColor="EDEDED"),
    "ZERO/MISSING CYCLE TIME":           PatternFill("solid", fgColor="FFC7CE"),
    "NOT IN VT_MATRIX":                  PatternFill("solid", fgColor="FFEB9C"),
    "ZERO/MISSING INDENT":               PatternFill("solid", fgColor="FFEB9C"),
    "ENTERS SCHEDULER":                  PatternFill("solid", fgColor="C6EFCE"),
}

def style_sheet(ws, header_hex):
    for cell in ws[1]:
        cell.fill      = PatternFill("solid", fgColor=header_hex)
        cell.font      = Font(bold=True, color="FFFFFF", size=11)
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
    ws.row_dimensions[1].height = 32
    for col in ws.columns:
        col_letter = get_column_letter(col[0].column)
        max_len    = max((len(str(c.value)) for c in col if c.value is not None), default=8)
        ws.column_dimensions[col_letter].width = max(10, min(55, max_len + 3))
    headers = [cell.value for cell in ws[1]]
    for col_idx, col_name in enumerate(headers, start=1):
        if col_name and any(x in str(col_name) for x in
                            ["Status", "Indent_Status", "Meets_Daily",
                             "Covers_With", "Gap_Direction"]):
            for row in ws.iter_rows(min_row=2, min_col=col_idx, max_col=col_idx):
                for cell in row:
                    fill = STATUS_FILLS.get(str(cell.value))
                    if fill:
                        cell.fill = fill
    ws.freeze_panes = "A2"


def style_machine_wise_sheet(ws):
    header_fill  = PatternFill("solid", fgColor="1F4E79")
    summary_fill = PatternFill("solid", fgColor="0D9488")
    part_fills   = [PatternFill("solid", fgColor="EFF6FF"),
                    PatternFill("solid", fgColor="F0FDF4")]
    co_fill      = PatternFill("solid", fgColor="FEF9C3")

    for cell in ws[1]:
        cell.fill      = header_fill
        cell.font      = Font(bold=True, color="FFFFFF", size=11)
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
    ws.row_dimensions[1].height = 30

    headers      = [cell.value for cell in ws[1]]
    row_type_col = headers.index("Row_Type")  + 1 if "Row_Type"  in headers else None
    co_col       = headers.index("Changeover") + 1 if "Changeover" in headers else None
    machine_col  = headers.index("Machine")   + 1 if "Machine"   in headers else None

    machine_color_idx = 0
    current_machine   = None
    for row in ws.iter_rows(min_row=2):
        row_type = row[row_type_col-1].value if row_type_col else ""
        machine  = row[machine_col-1].value  if machine_col  else ""
        if machine and machine != current_machine:
            current_machine   = machine
            machine_color_idx = (machine_color_idx + 1) % 2
        if row_type == "Summary":
            for cell in row:
                cell.fill = summary_fill
                cell.font = Font(bold=True, color="FFFFFF", size=11)
                cell.alignment = Alignment(horizontal="center", vertical="center")
        elif row_type == "Part":
            for cell in row:
                cell.fill      = part_fills[machine_color_idx]
                cell.alignment = Alignment(vertical="center")
            if co_col and row[co_col-1].value == "Yes":
                row[co_col-1].fill = co_fill

    for col in ws.columns:
        col_letter = get_column_letter(col[0].column)
        max_len    = max((len(str(c.value)) for c in col if c.value), default=8)
        ws.column_dimensions[col_letter].width = max(10, min(45, max_len + 3))
    ws.freeze_panes = "B2"


def style_prod_vs_indent_sheet(ws):
    """Special styling for VT_Production_vs_Indent sheet."""
    style_sheet(ws, HEADER_COLORS["VT_Production_vs_Indent"])

    headers   = [c.value for c in ws[1]]
    gap_col   = headers.index("Gap_Direction") + 1 if "Gap_Direction" in headers else None
    gap_v_col = headers.index("Gap_vs_Daily")  + 1 if "Gap_vs_Daily"  in headers else None

    over_fill  = PatternFill("solid", fgColor="DDEBF7")   # blue — over-produced
    under_fill = PatternFill("solid", fgColor="FFC7CE")   # red  — under-produced
    met_fill   = PatternFill("solid", fgColor="C6EFCE")   # green — exactly met

    for row in ws.iter_rows(min_row=2):
        if gap_col:
            cell  = row[gap_col - 1]
            value = str(cell.value)
            if value == "OVER":
                cell.fill = over_fill
            elif value == "UNDER":
                cell.fill = under_fill
                for c in row:
                    c.font = Font(bold=True)
            elif value == "MET":
                cell.fill = met_fill


def style_multi_machine_sheet(ws):
    """Special styling for VT_Multi_Machine_Parts sheet."""
    style_sheet(ws, HEADER_COLORS["VT_Multi_Machine_Parts"])

    headers  = [c.value for c in ws[1]]
    role_col = headers.index("Role") + 1 if "Role" in headers else None

    total_fill   = PatternFill("solid", fgColor="0D9488")
    primary_fill = PatternFill("solid", fgColor="EFF6FF")
    expand_fill  = PatternFill("solid", fgColor="FEF9C3")

    for row in ws.iter_rows(min_row=2):
        if not role_col:
            continue
        role = str(row[role_col - 1].value)
        if role == "TOTAL":
            for cell in row:
                cell.fill = total_fill
                cell.font = Font(bold=True, color="FFFFFF", size=11)
        elif "Tool-Expansion" in role:
            for cell in row:
                cell.fill = expand_fill
        elif role == "Primary":
            for cell in row:
                cell.fill = primary_fill


# Write Excel
print(f"\nWriting output → {output_path}")

vt_mw    = build_machine_wise_plan(vt_plan)
vt_co_q  = build_co_queue(
    [{k:v for k,v in r.items()} for r in vt_plan.to_dict("records")]
    if not vt_plan.empty else [],
    vt_machines
)

sheets = {
    "VT_Plan_By_Machine":      vt_mw,
    "VT_CO_Queue":             vt_co_q,
    "VT_Plan":                 vt_plan,
    "VT_Multi_Machine_Parts":  vt_multi_machine,
    "VT_Production_vs_Indent": vt_prod_vs_indent,
    "VT_Daily_Indent_Status":  vt_indent_status,
    "VT_Priority_Scores":      vt_scores,
    "VT_Machine_Util":         vt_mach,
    "VT_Not_Planned":          vt_not,
    "VT_Deferred":             vt_def,
    "VT_Inventory_Health":     vt_inv,
    "VT_Indent_Horizon":       vt_horizon,
    "VT_Part_Audit":           audit_df,
}
if not vt_micro.empty:
    sheets["VT_Micro_Idle"] = vt_micro

with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
    for sheet_name, df in sheets.items():
        if df is not None and not df.empty:
            df.to_excel(writer, sheet_name=sheet_name, index=False)

wb = load_workbook(output_path)

if "VT_Plan_By_Machine" in wb.sheetnames:
    style_machine_wise_sheet(wb["VT_Plan_By_Machine"])
if "VT_Production_vs_Indent" in wb.sheetnames:
    style_prod_vs_indent_sheet(wb["VT_Production_vs_Indent"])
if "VT_Multi_Machine_Parts" in wb.sheetnames:
    style_multi_machine_sheet(wb["VT_Multi_Machine_Parts"])

for sheet_name, header_hex in HEADER_COLORS.items():
    if sheet_name in wb.sheetnames and sheet_name not in (
            "VT_Plan_By_Machine", "VT_Production_vs_Indent", "VT_Multi_Machine_Parts"):
        style_sheet(wb[sheet_name], header_hex)

# Colour Daily Indent Status YES/NO columns
if "VT_Daily_Indent_Status" in wb.sheetnames:
    ws_is      = wb["VT_Daily_Indent_Status"]
    headers_is = [c.value for c in ws_is[1]]
    meets_col  = (headers_is.index("Meets_Daily_Indent") + 1
                  if "Meets_Daily_Indent" in headers_is else None)
    covers_col = (headers_is.index("Covers_With_Inv") + 1
                  if "Covers_With_Inv" in headers_is else None)
    for row in ws_is.iter_rows(min_row=2):
        if meets_col:
            cell = row[meets_col - 1]
            cell.fill = (PatternFill("solid", fgColor="C6EFCE")
                         if str(cell.value) == "YES ✓"
                         else PatternFill("solid", fgColor="FFC7CE"))
        if covers_col:
            cell = row[covers_col - 1]
            cell.fill = (PatternFill("solid", fgColor="C6EFCE")
                         if str(cell.value) == "YES ✓"
                         else PatternFill("solid", fgColor="FFEB9C"))
        if meets_col and str(row[meets_col - 1].value) == "NO ✗":
            for cell in row:
                cell.font = Font(bold=True)

for name, color in HEADER_COLORS.items():
    if name in wb.sheetnames:
        wb[name].sheet_properties.tabColor = color

wb.save(output_path)
print(f"  Formatting applied  ✓")

# =============================================================
# SECTION 25 — FINAL SUMMARY
# =============================================================

print(f"\n{'='*65}")
print(f"  Smart APS V8 Complete  —  {PLANNING_DATE}")
print(f"  Tool-Aware Multi-Machine Scheduling")
print(f"{'='*65}")

for status, count in audit_df["Status"].value_counts().items():
    marker = "✓" if status == "ENTERS SCHEDULER" else "·"
    print(f"  {marker} {status:<45}: {count:>4}")

print(f"\n  Results:")
print(f"    Planned            : {len(vt_plan):>4} rows")
print(f"    Not planned        : {len(vt_not):>4}")
print(f"    Deferred           : {len(vt_def):>4}")

if not vt_multi_machine.empty:
    n_multi = vt_multi_machine[vt_multi_machine["Role"] == "TOTAL"].shape[0]
    print(f"    Multi-machine parts: {n_multi:>4}  (see VT_Multi_Machine_Parts)")

if not vt_prod_vs_indent.empty:
    over  = (vt_prod_vs_indent["Gap_Direction"] == "OVER").sum()
    under = (vt_prod_vs_indent["Gap_Direction"] == "UNDER").sum()
    met   = (vt_prod_vs_indent["Gap_Direction"] == "MET").sum()
    print(f"\n  Production vs Daily Indent:")
    print(f"    Over-produced  : {over:>4} parts  (inv building — see VT_Production_vs_Indent)")
    print(f"    Under-produced : {under:>4} parts  (machine capacity limited)")
    print(f"    Exactly met    : {met:>4} parts")

if not vt_mach.empty:
    print(f"\n  Machine utilization:")
    print(f"    Average     : {vt_mach['Utilization_%'].mean():.1f}%")
    print(f"    UNDERUSED   : {(vt_mach['Status']=='UNDERUSED').sum()} machines")

print(f"\n  Output → {output_path}")
print(f"  State  → {MACHINE_STATE_FILE}")
print(f"\n  UPDATE DAILY: PLANNING_DATE = date(2026, 3, 21)")
print(f"{'='*65}")


  Smart APS V8  —  Tool-Aware Multi-Machine Scheduling
  Planning date : 2026-03-20
  Indent month  : March 2026
  Working days  : 26  (31 days − 5 Sundays)

Loading data...
  VT parts in sheet       : 158
  Parts with valid rate   : 153
  Distinct colours        : 5
    BLACK                → 107 part(s)
    BLUE                 → 8 part(s)
    BROWN                → 4 part(s)
    GREEN                → 2 part(s)
    WHITE                → 37 part(s)
  Machine part counts loaded: 30 machines
    SWIFT                     → 2 part(s)  ← MOST SPECIALIZED
    TOYO 80T 1ST              → 7 part(s)  
    TOYO 80T 2ND              → 7 part(s)  
    TOYO 6TH                  → 19 part(s)  
    TOYO 9TH                  → 19 part(s)  
  Machine state loaded  (30 machines with history)

  Part audit (158 total):
    ✓  ENTERS SCHEDULER                             :   76
    ·  SKIPPED (LOW INDENT / TRIVIAL RUN)           :   76
    ·  ZERO/MISSING CYCLE TIME                      :    5
    · 

In [8]:
import pandas as pd
import numpy as np
import json
import os
import math
import calendar
from datetime import datetime, date



PLANNING_DATE = date(2026, 3, 20)   # ← change daily
INDENT_MONTH  = date(2026, 3,  1)   # ← change when month rolls over



AVAILABLE_HOURS      = 22
MIN_RUN_HOURS        = 4
MACHINE_STATE_FILE   = "machine_state.json"

MIN_DAILY_INDENT     = 150
MIN_INDENT_HOURS     = 4.0


SAFETY_DAYS  = 3    
TARGET_DAYS = 5


OPD_SCENARIO_0 = 1.5
OPD_SCENARIO_1 = 3.0
OPD_SCENARIO_2 = 4.0
OPD_SCENARIO_3 = 5.0

W_URGENCY  = 0.55
W_CATEGORY = 0.25
W_INDENT   = 0.20

UTIL_TARGET_PCT = 98.0



book_path       = "C:/Users/Ex0164/Book1.xlsx"
matrix_path     = "C:/Users/Ex0164/Important codes/compatibility_matrix.xlsx"
changeover_path = "D:/Tushar/TOOL FIX/Unique_Machines_vt.xlsx"
output_path     = f"Smart_APS_V9_Plan_{PLANNING_DATE.strftime('%Y%m%d')}color constraint.xlsx"



def compute_working_days(ref_date):
    year  = ref_date.year
    month = ref_date.month
    total = calendar.monthrange(year, month)[1]
    sundays = sum(
        1 for d in range(1, total + 1)
        if date(year, month, d).weekday() == 6
    )
    return total - sundays, total, sundays

WORKING_DAYS, TOTAL_DAYS, SUNDAY_COUNT = compute_working_days(INDENT_MONTH)

print(f"\n{'='*65}")
print(f"  Smart APS V9  —  5-Day Target Inventory System")
print(f"  Planning date : {PLANNING_DATE}")
print(f"  Indent month  : {INDENT_MONTH.strftime('%B %Y')}")
print(f"  Working days  : {WORKING_DAYS}  ({TOTAL_DAYS} days − {SUNDAY_COUNT} Sundays)")
print(f"  Safety floor  : {SAFETY_DAYS} days  |  Target ceiling : {TARGET_DAYS} days")
print(f"{'='*65}\n")


print("Loading data...")
vt_parts_raw = pd.read_excel(book_path,       sheet_name="VT")
vt_matrix    = pd.read_excel(matrix_path,     sheet_name="VT_Matrix")
vt_co_raw         = pd.read_excel(changeover_path, sheet_name="VT_Changeover")
vt_machine_count_raw = pd.read_excel(matrix_path, sheet_name="VT_Machine_Part_Count")


def find_col(df, name, sheet):
    match = next((c for c in df.columns
                  if str(c).strip().lower() == name.lower()), None)
    if match is None:
        raise ValueError(
            f"Column '{name}' not found in sheet '{sheet}'.\n"
            f"Available columns: {list(df.columns)}"
        )
    return match

vt_col_part      = find_col(vt_parts_raw, "Part",       "VT")
vt_col_cycletime = find_col(vt_parts_raw, "Cycle time", "VT")
vt_col_cavity    = find_col(vt_parts_raw, "Cavity",     "VT")
vt_col_inventory = find_col(vt_parts_raw, "Inventory",  "VT")
vt_col_indent    = find_col(vt_parts_raw, "Indent",     "VT")
vt_col_tools     = find_col(vt_parts_raw, "Tools",      "VT")

data = vt_parts_raw[
    vt_parts_raw[vt_col_part].notna() &
    (vt_parts_raw[vt_col_part].astype(str).str.strip() != "")
].copy()
data = data.drop_duplicates(subset=vt_col_part).copy()
data["Material"] = data[vt_col_part].astype(str).str.strip()

data["_ct"] = pd.to_numeric(data[vt_col_cycletime], errors="coerce").replace(0, np.nan)
data["Rate"] = 3600 / data["_ct"]

data_valid     = data[data["Rate"].notna()].copy()
data_zero_rate = data[data["Rate"].isna()].copy()

print(f"  VT parts in sheet       : {len(data)}")
print(f"  Parts with valid rate   : {len(data_valid)}")



def safe_dict(df, key_col, val_col, default=0.0):
    return {
        str(k).strip(): (default if pd.isna(v) else float(v))
        for k, v in zip(df[key_col], df[val_col])
        if pd.notna(k) and str(k).strip() != ""
    }

inventory      = safe_dict(data,       "Material", vt_col_inventory)
rate           = safe_dict(data_valid, "Material", "Rate")
indent_monthly = safe_dict(data,       "Material", vt_col_indent)

tools_available = {}
for _, row in data.iterrows():
    p = str(row["Material"]).strip()
    v = row[vt_col_tools]
    tools_available[p] = max(1, int(float(v))) if pd.notna(v) and str(v).strip() != "" else 1

indent_daily = {
    p: round(qty / WORKING_DAYS, 4)
    for p, qty in indent_monthly.items()
}

today_target_qty = {
    p: max(0.0, indent_daily.get(p, 0.0) - inventory.get(p, 0.0))
    for p in indent_monthly
}



def should_skip(part):
    daily   = indent_daily.get(part, 0.0)
    monthly = indent_monthly.get(part, 0.0)
    r       = rate.get(part, 1.0)

    if daily <= MIN_DAILY_INDENT:
        return True, f"Daily indent {daily:.2f} ≤ {MIN_DAILY_INDENT} threshold"

    indent_hrs = monthly / r if r > 0 else 0.0
    if indent_hrs <= MIN_INDENT_HOURS:
        return True, f"Whole monthly indent = {indent_hrs:.2f}h ≤ {MIN_INDENT_HOURS}h threshold"

    # V9: Gate 5 — skip only if inventory >= TARGET_DAYS × daily_indent
    # (was: inventory >= daily_indent in V8)
    inv = inventory.get(part, 0.0)
    if daily > 0 and inv >= TARGET_DAYS * daily:
        return True, (f"Inventory ({inv:.0f}) ≥ {TARGET_DAYS}-day target "
                      f"({TARGET_DAYS * daily:.0f} pcs) — at ceiling, skip today")

    return False, ""



def build_changeover_dict(co_df):
    co_dict = {}
    for _, row in co_df.iterrows():
        machine = str(row["Unique Machines"]).strip()
        minutes = row["Changeover time"]
        if pd.notna(minutes) and machine:
            co_dict[machine] = float(minutes) / 60.0
    return co_dict

vt_changeover          = build_changeover_dict(vt_co_raw)
DEFAULT_CHANGEOVER_HRS = 40 / 60.0


def build_machine_part_count(df):
    mpc = {}
    machine_col = next((c for c in df.columns
                        if str(c).strip().lower() == "machine"), None)
    count_col   = next((c for c in df.columns
                        if str(c).strip().lower() == "part_count"), None)
    if machine_col is None or count_col is None:
        print(f"  WARNING: VT_Machine_Part_Count sheet missing "
              f"'Machine' or 'Part_Count' column — using equal ranking")
        return {}
    for _, row in df.iterrows():
        m = str(row[machine_col]).strip()
        v = row[count_col]
        if m and pd.notna(v):
            try:
                mpc[m] = int(float(v))
            except (ValueError, TypeError):
                pass
    return mpc

machine_part_count = build_machine_part_count(vt_machine_count_raw)
max_part_count     = max(machine_part_count.values(), default=1) or 1

print(f"  Machine part counts loaded: {len(machine_part_count)} machines")



def build_category(df):
    cat      = {}
    part_col = next((c for c in df.columns if str(c).strip().lower() == "part"), None)
    cat_col  = next((c for c in df.columns if str(c).strip().lower() == "category"), None)
    if part_col is None:
        return cat
    for _, row in df.iterrows():
        part = row[part_col]
        if pd.isna(part) or str(part).strip() == "":
            continue
        val = "Stranger"
        if cat_col and pd.notna(row[cat_col]):
            val = str(row[cat_col]).strip().capitalize()
            if val not in ("Runner", "Repeater", "Stranger"):
                val = "Stranger"
        cat[str(part).strip()] = val
    return cat

part_category  = build_category(vt_parts_raw)
CATEGORY_SCORE = {"Runner": 100, "Repeater": 60, "Stranger": 20}



def load_machine_state():
    if os.path.exists(MACHINE_STATE_FILE):
        try:
            with open(MACHINE_STATE_FILE) as f:
                content = f.read().strip()
            if not content:
                print(f"  Machine state : file empty — treating as first run")
                os.remove(MACHINE_STATE_FILE)
                return {}
            state = json.loads(content)
            if not isinstance(state, dict):
                print(f"  Machine state : file corrupt — treating as first run")
                os.remove(MACHINE_STATE_FILE)
                return {}
            print(f"  Machine state loaded  ({len(state)} machines with history)")
            return state
        except json.JSONDecodeError as e:
            print(f"  Machine state : JSON error ({e}) — treating as first run")
            os.remove(MACHINE_STATE_FILE)
            return {}
        except Exception as e:
            print(f"  Machine state : read error ({e}) — treating as first run")
            os.remove(MACHINE_STATE_FILE)
            return {}
    print(f"  Machine state : FIRST RUN — no changeover today")
    return {}

def save_machine_state(state):
    combined = {m: p for m, p in state.items() if p is not None}
    with open(MACHINE_STATE_FILE, "w") as f:
        json.dump(combined, f, indent=2)
    print(f"\n  Machine state saved → '{MACHINE_STATE_FILE}'")

machine_state = load_machine_state()



def build_compatibility(matrix):
    machines = matrix.columns[1:].tolist()
    compat   = {}
    for _, row in matrix.iterrows():
        part = row["Part"]
        for m in machines:
            if row[m] == 1:
                compat.setdefault(part, []).append(m)
    return compat, machines

vt_compat, vt_machines = build_compatibility(vt_matrix)



def classify_scenario(parts):
    coverage = []
    for p in parts:
        inv   = inventory.get(p, 0)
        daily = indent_daily.get(p, 0)
        skip, _ = should_skip(p)
        if skip or daily == 0:
            continue
        coverage.append(inv / daily)

    if not coverage:
        return 3, "SCENARIO 3 — No active parts today"

    n        = len(coverage)
    critical = sum(1 for c in coverage if c < 1)
    # V9: low = below SAFETY_DAYS (3), not TARGET_DAYS
    low      = sum(1 for c in coverage if c < SAFETY_DAYS)

    if critical == n:
        return 0, f"SCENARIO 0 — ALL {n} active parts critical (inv < 1 day)"
    elif critical > 0:
        return 1, f"SCENARIO 1 — {critical}/{n} parts critical  |  {n-critical} have buffer"
    elif low > 0:
        return 2, (f"SCENARIO 2 — {low}/{n} parts below {SAFETY_DAYS}-day safety floor")
    else:
        return 3, (f"SCENARIO 3 — All {n} parts healthy (≥{SAFETY_DAYS} days safety floor)")



def opd_cap(scenario_id):
    """
    Returns the effective OPD cap for today.
    The scenario controls the daily build RATE.
    TARGET_DAYS (5) is the absolute ceiling — no scenario
    can push inventory beyond 5 × daily_indent.
    """
    scenario_opd = {
        0: OPD_SCENARIO_0,
        1: OPD_SCENARIO_1,
        2: OPD_SCENARIO_2,
        3: OPD_SCENARIO_3,
    }.get(scenario_id, OPD_SCENARIO_2)
    # V9: hard cap — never exceed TARGET_DAYS regardless of scenario
    return min(scenario_opd, TARGET_DAYS)



def compute_priority_scores(active_parts):
    rows = []
    for p in active_parts:
        inv      = inventory.get(p, 0)
        daily    = indent_daily.get(p, 0)
        cat      = part_category.get(p, "Stranger")
        days_cov = inv / daily if daily > 0 else 999.0
        
        gap_days = min(1.0, max(0.0, SAFETY_DAYS - days_cov) / SAFETY_DAYS)
        rows.append({"part": p, "inv": inv, "daily": daily,
                     "days_cov": days_cov, "cat": cat, "urgency_raw": gap_days})

    if not rows:
        return {}, []

    max_daily = max(r["daily"] for r in rows) or 1.0
    scores, score_rows = {}, []

    for r in rows:
        p              = r["part"]
        urgency_score  = r["urgency_raw"] * 100
        category_score = CATEGORY_SCORE.get(r["cat"], 20)
        indent_score   = (r["daily"] / max_daily) * 100
        final_score    = (W_URGENCY * urgency_score +
                          W_CATEGORY * category_score +
                          W_INDENT   * indent_score)
        scores[p] = round(final_score, 2)
        score_rows.append({
            "Part":            p,
            "Category":        r["cat"],
            "Tools":           tools_available.get(p, 1),
            "Inventory_Now":   round(r["inv"], 0),
            "Daily_Indent":    round(r["daily"], 2),
            "Days_Coverage":   round(r["days_cov"], 2),
            "Safety_Floor":    SAFETY_DAYS,
            "Target_Ceiling":  TARGET_DAYS,
            "Buffer_Status":   (
                "CRITICAL"     if r["days_cov"] < 1 else
                "BELOW_SAFETY" if r["days_cov"] < SAFETY_DAYS else
                "BUILDING"     if r["days_cov"] < TARGET_DAYS else
                "AT_TARGET"
            ),
            "Urgency_Score":   round(urgency_score, 1),
            "Category_Score":  category_score,
            "Indent_Score":    round(indent_score, 1),
            "Final_Score":     round(final_score, 2),
        })

    return scores, score_rows

def build_color(df):
    colors = {}
    part_col = next((c for c in df.columns if str(c).strip().lower() == "part"), None)
    color_col = next((c for c in df.columns if str(c).strip().lower() == "color"), None)
    if part_col is None or color_col is None:
        return colors
    for _, row in df.iterrows():
        part = str(row[part_col]).strip()
        if part and pd.notna(row[color_col]):
            colors[part] = str(row[color_col]).strip().capitalize()

    return colors

part_color = build_color(vt_parts_raw)


def displace_for_zero_inv(
    part, machine_hours, machine_last_part, current_inventory,
    plan, already_planned, priority_scores
):
    """
    Attempts to free MIN_RUN_HOURS on a compatible machine by
    reducing the run of the least-urgent planned part.
    Returns True if displacement succeeded and part was assigned.
    """
    daily    = indent_daily.get(part, 0)
    r_val    = rate.get(part, 1)
    category = part_category.get(part, "Stranger")
    score    = priority_scores.get(part, 0)
    compatible = vt_compat.get(part, [])

    if not compatible:
        return False

    # Find machines where this part could run but all hours consumed
    candidate_machines = [
        m for m in compatible
        if round(AVAILABLE_HOURS - machine_hours.get(m, 0), 4) < MIN_RUN_HOURS
    ]

    if not candidate_machines:
        return False

    # On each candidate machine, find the part with the MOST days of stock
    best_machine       = None
    best_victim_row    = None
    best_victim_days   = -1

    for m in candidate_machines:
        machine_plan_rows = [r for r in plan if r["Machine"] == m]
        for row in machine_plan_rows:
            vpart       = row["Part"]
            vdaily      = indent_daily.get(vpart, 0)
            vinv        = current_inventory.get(vpart, 0)
            vdays       = vinv / vdaily if vdaily > 0 else 999

            # Safety rule: never displace a part below safety floor
            # or itself at zero inventory
            if vdays < SAFETY_DAYS or vinv <= 0:
                continue

            # Must have enough run hours to give up MIN_RUN_HOURS
            vrun = float(row.get("Run_Hours", 0))
            vr   = rate.get(vpart, 1)
            # After reduction, victim must still produce at least MIN_RUN_HOURS
            if vrun - MIN_RUN_HOURS < MIN_RUN_HOURS:
                continue

            if vdays > best_victim_days:
                best_victim_days = vdays
                best_victim_row  = row
                best_machine     = m

    if best_machine is None or best_victim_row is None:
        return False

    # ── Perform displacement ──────────────────────────────────
    vpart    = best_victim_row["Part"]
    vr_val   = rate.get(vpart, 1)
    reduce_h = MIN_RUN_HOURS
    lost_qty = round(reduce_h * vr_val, 0)

    best_victim_row["Run_Hours"]      = round(
        float(best_victim_row["Run_Hours"]) - reduce_h, 3)
    best_victim_row["Production_Qty"] = round(
        float(best_victim_row["Production_Qty"]) - lost_qty, 0)
    best_victim_row["Total_Hrs_Used"] = round(
        float(best_victim_row.get("Changeover_Hrs", 0)) +
        float(best_victim_row["Run_Hours"]), 3)
    best_victim_row["Type"]           = str(
        best_victim_row.get("Type", "Primary")) + " [DISPLACED]"

    current_inventory[vpart]   = round(
        current_inventory.get(vpart, 0) - lost_qty, 0)
    machine_hours[best_machine] = round(
        machine_hours.get(best_machine, 0) - reduce_h, 4)

    # ── Assign zero-inv part to freed slot ────────────────────
    co_hrs = 0.0 if (
        machine_last_part.get(best_machine) is None or
        machine_last_part.get(best_machine) == part
    ) else vt_changeover.get(best_machine, DEFAULT_CHANGEOVER_HRS)

    eff_free  = round(AVAILABLE_HOURS - machine_hours.get(best_machine, 0) - co_hrs, 4)
    run_hrs   = max(MIN_RUN_HOURS, min(eff_free, MIN_RUN_HOURS))
    qty       = round(run_hrs * r_val, 0)

    machine_hours[best_machine]  = round(
        machine_hours.get(best_machine, 0) + co_hrs + run_hrs, 4)
    current_inventory[part]      = round(
        current_inventory.get(part, 0) + qty, 0)
    machine_last_part[best_machine] = part
    already_planned.add(part)

    plan.append({
        "Part":             part,
        "Category":         category,
        "Machine":          best_machine,
        "Color": part_color.get(part, "Unknown"),
        "Run_Hours":        round(run_hrs, 3),
        "Changeover_Hrs":   round(co_hrs, 3),
        "Total_Hrs_Used":   round(co_hrs + run_hrs, 3),
        "Rate_Per_Hour":    round(r_val, 2),
        "Production_Qty":   qty,
        "Monthly_Indent":   round(indent_monthly.get(part, 0), 0),
        "Daily_Indent":     round(daily, 2),
        "Today_Target":     round(today_target_qty.get(part, 0), 0),
        "Changeover":       "No" if co_hrs == 0 else "Yes",
        "Type":             "Displacement [ZERO-INV PRIORITY]",
        "Role":             "Primary",
        "Tools_Available":  tools_available.get(part, 1),
        "Tools_Used":       1,
        "Runner_Lock":      "No",
        "Priority_Score":   score,
        "Phase":            1,
        "Indent_Met":       "YES" if qty >= daily else "NO — partial",
        "Stagger_Adjusted": "No",
        "Displaced_Victim": vpart,
        "Victim_Days_Stock":round(best_victim_days, 2),
    })

    print(f"      ↳ DISPLACEMENT  {part:26s} → {best_machine:15s}  "
          f"freed from {vpart} ({best_victim_days:.1f} days stock)  "
          f"run={run_hrs:.2f}h  qty={qty:.0f}")
    return True



def rank_machines(part, machines_to_try, machine_hours,
                  machine_last_part, inv_days):
    category    = part_category.get(part, "Stranger")
    runner_lock = (category == "Runner" and inv_days <= 1.0)

    ranked = []
    for m in machines_to_try:
        used = machine_hours.get(m, 0)
        free = round(AVAILABLE_HOURS - used, 4)
        if free < MIN_RUN_HOURS:
            continue
        last = machine_last_part.get(m)
        if runner_lock and last != part:
            continue
        co_hrs = 0.0 if (last is None or last == part) else \
                 vt_changeover.get(m, DEFAULT_CHANGEOVER_HRS)

        if last and last in part_color and part in part_color:
            if part_color[last] != part_color[part]:
                co_hrs += 10.0/60.0
                
        effective_free = round(free - co_hrs, 4)
        if effective_free < MIN_RUN_HOURS:
            continue

        part_count      = machine_part_count.get(m, max_part_count)
        count_score     = part_count / max_part_count
        co_penalty      = (co_hrs / AVAILABLE_HOURS) * 0.3
        util_penalty    = (used  / AVAILABLE_HOURS) * 0.2
        same_part_bonus = -0.15 if (last == part) else 0.0
        same_color_bonus = -0.1 if last and part_color.get(last) == part_color.get(part) else 0.0
        cost = count_score + co_penalty + util_penalty + same_part_bonus + same_part_bonus

        ranked.append((m, co_hrs, effective_free, cost))

    ranked.sort(key=lambda x: x[3])
    return ranked, runner_lock



def assign_part(part, scenario_id, machine_hours, machine_last_part,
                current_inventory, plan, already_planned,
                priority_scores):
    daily    = indent_daily.get(part, 0)
    monthly  = indent_monthly.get(part, 0)
    r_val    = rate.get(part, 1)
    inv_now  = current_inventory.get(part, 0)
    category = part_category.get(part, "Stranger")
    tools    = tools_available.get(part, 1)
    score    = priority_scores.get(part, 0)
    inv_days = inv_now / daily if daily > 0 else 999
    compatible = vt_compat.get(part, [])

    if not compatible:
        return []

    total_shortfall = max(0.0, daily - inv_now)
    hrs_for_full    = total_shortfall / r_val if r_val > 0 else MIN_RUN_HOURS
    hrs_for_full    = max(MIN_RUN_HOURS, hrs_for_full)

    new_rows        = []
    produced_so_far = 0.0
    tools_used      = 0
    used_machines   = set()

    ranked, runner_lock = rank_machines(
        part, compatible, machine_hours, machine_last_part, inv_days)

    if not ranked:
        return []

    m1, co1, eff1, _ = ranked[0]

    run1 = min(eff1, hrs_for_full)
    run1 = max(run1, MIN_RUN_HOURS)
    qty1 = round(run1 * r_val, 0)

    machine_hours[m1]       = round(machine_hours.get(m1, 0) + co1 + run1, 4)
    current_inventory[part] = round(current_inventory.get(part, 0) + qty1, 0)
    machine_last_part[m1]   = part
    produced_so_far        += qty1
    tools_used             += 1
    used_machines.add(m1)
    already_planned.add(part)

    indent_met_on_primary = (produced_so_far >= total_shortfall - 0.5)

    new_rows.append({
        "Part":             part,
        "Category":         category,
        "Machine":          m1,
        "Color":            part_color.get(part, "Unknown"),
        "Run_Hours":        round(run1, 3),
        "Changeover_Hrs":   round(co1, 3),
        "Total_Hrs_Used":   round(co1 + run1, 3),
        "Rate_Per_Hour":    round(r_val, 2),
        "Production_Qty":   qty1,
        "Monthly_Indent":   round(monthly, 0),
        "Daily_Indent":     round(daily, 2),
        "Today_Target":     round(today_target_qty.get(part, 0), 0),
        "Changeover":       "No" if co1 == 0 else "Yes",
        "Type":             "Primary" + (" [ZERO-INV]" if inv_now == 0 else ""),
        "Role":             "Primary",
        "Tools_Available":  tools,
        "Tools_Used":       1,
        "Runner_Lock":      "YES" if runner_lock else "No",
        "Priority_Score":   score,
        "Phase":            1,
        "Indent_Met":       "YES" if indent_met_on_primary else "NO — shortfall remains",
        "Stagger_Adjusted": "No",
    })

    if not indent_met_on_primary:
        is_critical   = (inv_now == 0)
        tool_hard_cap = tools if is_critical else min(2, tools)

        while produced_so_far < (total_shortfall - 0.5) and tools_used < tool_hard_cap:
            shortfall_now = total_shortfall - produced_so_far
            hrs_needed    = max(MIN_RUN_HOURS,
                                shortfall_now / r_val if r_val > 0 else MIN_RUN_HOURS)

            remaining_machines = [m for m in compatible if m not in used_machines]
            ranked_next, _ = rank_machines(
                part, remaining_machines, machine_hours,
                machine_last_part, inv_days)

            if not ranked_next:
                break

            mx, cox, effx, _ = ranked_next[0]
            run_x = min(effx, hrs_needed)
            run_x = max(run_x, MIN_RUN_HOURS)
            qty_x = round(run_x * r_val, 0)

            machine_hours[mx]       = round(machine_hours.get(mx, 0) + cox + run_x, 4)
            current_inventory[part] = round(current_inventory.get(part, 0) + qty_x, 0)
            machine_last_part[mx]   = part
            produced_so_far        += qty_x
            tools_used             += 1
            used_machines.add(mx)

            indent_met_here = (produced_so_far >= total_shortfall - 0.5)

            new_rows.append({
                "Part":             part,
                "Category":         category,
                "Machine":          mx,
                "Color":            part_color.get(part, "Unknown"),
                "Run_Hours":        round(run_x, 3),
                "Changeover_Hrs":   round(cox, 3),
                "Total_Hrs_Used":   round(cox + run_x, 3),
                "Rate_Per_Hour":    round(r_val, 2),
                "Production_Qty":   qty_x,
                "Monthly_Indent":   round(monthly, 0),
                "Daily_Indent":     round(daily, 2),
                "Today_Target":     round(today_target_qty.get(part, 0), 0),
                "Changeover":       "No" if cox == 0 else "Yes",
                "Type":             "Tool-Expansion",
                "Role":             f"Tool-Expansion (tool {tools_used}/{tools})",
                "Tools_Available":  tools,
                "Tools_Used":       tools_used,
                "Runner_Lock":      "No",
                "Priority_Score":   score,
                "Phase":            2,
                "Indent_Met":       "YES" if indent_met_here else "NO — shortfall remains",
                "Stagger_Adjusted": "No",
            })

            crisis_tag = " [CRISIS — 3rd+ tool]" if tools_used >= 3 else ""
            print(f"      ↳ TOOL-EXP {part:26s} tool {tools_used}/{tool_hard_cap} → "
                  f"{mx:15s}  {run_x:.2f}h  qty={qty_x:.0f}  "
                  f"{'COVERED ✓' if indent_met_here else 'still short'}"
                  f"{crisis_tag}")

    
    cap_days     = opd_cap(scenario_id)          # already capped at TARGET_DAYS
    inv_after    = current_inventory.get(part, 0)
    cap_qty      = cap_days * daily
    headroom_qty = max(0.0, cap_qty - inv_after)

    if headroom_qty > 0:
        for row in new_rows:
            if headroom_qty <= 0:
                break
            m       = row["Machine"]
            free_m  = round(AVAILABLE_HOURS - machine_hours.get(m, 0), 4)
            if free_m < 0.05:
                continue
            extend_hrs = min(free_m, headroom_qty / r_val if r_val > 0 else 0)
            if extend_hrs < 0.05:
                continue
            extra_qty = round(extend_hrs * r_val, 0)

            row["Run_Hours"]      = round(float(row["Run_Hours"]) + extend_hrs, 3)
            row["Total_Hrs_Used"] = round(float(row["Changeover_Hrs"]) + float(row["Run_Hours"]), 3)
            row["Production_Qty"] = round(float(row["Production_Qty"]) + extra_qty, 0)
            row["Color"] = part_color.get(part, "Unknown")
            row["Type"]           = str(row["Type"]) + "+InvBuild"

            machine_hours[m]        = round(machine_hours.get(m, 0) + extend_hrs, 4)
            current_inventory[part] = round(current_inventory.get(part, 0) + extra_qty, 0)
            headroom_qty           -= extra_qty

            print(f"      ↳ INV-BUILD {part:25s} on {m:15s}  "
                  f"+{extend_hrs:.2f}h  qty+={extra_qty:.0f}  "
                  f"(target ceiling: {cap_days:.1f} days)")

    for row in new_rows:
        row["Tools_Used"] = tools_used

    return new_rows



def _fmt_h(h):
    try:
        total_min = int(round(float(h) * 60))
        return f"{total_min // 60:02d}:{total_min % 60:02d}"
    except Exception:
        return "??"

def _collect_co_events(plan, machines):
    events = []
    for m in machines:
        m_rows = [r for r in plan if r["Machine"] == m]
        if len(m_rows) < 2:
            continue
        cursor = 0.0
        for i, row in enumerate(m_rows):
            co_h  = float(row.get("Changeover_Hrs") or 0.0)
            run_h = float(row.get("Run_Hours")       or 0.0)
            if co_h > 0 and i > 0:
                events.append({
                    "machine":       m,
                    "part_before":   m_rows[i-1]["Part"],
                    "part_after":    row["Part"],
                    "co_duration":   co_h,
                    "natural_start": cursor,
                    "row_before":    m_rows[i-1],
                    "row_after":     row,
                    "actual_start":  None,
                    "wait_hrs":      0.0,
                })
            cursor += co_h + run_h
    return events

def _recompute_natural_start(ev, plan):
    m, target = ev["machine"], ev["row_after"]
    cursor = 0.0
    for r in [r for r in plan if r["Machine"] == m]:
        if r is target:
            break
        cursor += float(r.get("Changeover_Hrs") or 0) + float(r.get("Run_Hours") or 0)
    return cursor

def _machine_spare(m, plan):
    used = sum(
        float(r.get("Changeover_Hrs") or 0) + float(r.get("Run_Hours") or 0)
        for r in plan if r["Machine"] == m
    )
    return max(0.0, AVAILABLE_HOURS - used)

def _extend_row_before(ev, wait_hrs, plan):
    spare     = _machine_spare(ev["machine"], plan)
    extend_by = min(wait_hrs, spare)
    if extend_by <= 0:
        return 0.0, 0
    rb    = ev["row_before"]
    r_val = rate.get(rb["Part"], 1.0)
    extra = round(extend_by * r_val, 0)
    rb["Run_Hours"]      = round(float(rb.get("Run_Hours") or 0) + extend_by, 3)
    rb["Production_Qty"] = round(float(rb.get("Production_Qty") or 0) + extra, 0)
    rb["Total_Hrs_Used"] = round(
        float(rb.get("Changeover_Hrs") or 0) + float(rb["Run_Hours"]), 3)
    rb["Stagger_Adjusted"] = f"CSP: extended +{round(extend_by*60,1)}min to fill TC wait"
    return extend_by, extra



MIN_CO_GAP_HRS = 20 / 60.0

def _finish_time_of_co(ev, plan):
    m      = ev["machine"]
    target = ev["row_after"]
    total  = 0.0
    for row in plan:
        if row["Machine"] != m:
            continue
        if row is target:
            break
        total += float(row.get("Run_Hours") or 0)
        total += float(row.get("Changeover_Hrs") or 0)
    return round(total, 4)

def _last_row_before_co(ev, plan):
    return ev["row_before"]

def stagger_co_by_quantity(plan, machines, scenario_id, current_inventory):
    events = _collect_co_events(plan, machines)
    if len(events) < 2:
        return 0

    adjustments = 0
    max_passes  = len(events) * 2

    for _ in range(max_passes):
        for ev in events:
            ev["_ft"] = _finish_time_of_co(ev, plan)
        events.sort(key=lambda e: e["_ft"])

        conflict = None
        for i in range(len(events) - 1):
            ft_early   = events[i]["_ft"]
            ft_late    = events[i+1]["_ft"]
            co_dur_e   = events[i]["co_duration"]
            required   = co_dur_e + MIN_CO_GAP_HRS
            gap        = ft_late - ft_early
            if gap < required - 0.001:
                conflict = (events[i], events[i+1], gap, required)
                break

        if conflict is None:
            break

        ev_early, ev_late, gap, required_gap = conflict
        shortfall_hrs = required_gap - gap

        row_late  = _last_row_before_co(ev_late, plan)
        p_late    = row_late["Part"]
        r_late    = rate.get(p_late, 1)
        m_late    = ev_late["machine"]
        daily_l   = indent_daily.get(p_late, 0)
        inv_l     = current_inventory.get(p_late, 0)
        # V9: cap against TARGET_DAYS ceiling
        cap_qty   = opd_cap(scenario_id) * daily_l
        produced_l = float(row_late.get("Production_Qty") or 0)
        headroom  = max(0.0, cap_qty - inv_l)

        push_hrs  = shortfall_hrs
        push_qty  = round(push_hrs * r_late, 0)

        used_m    = sum(
            float(r.get("Changeover_Hrs") or 0) + float(r.get("Run_Hours") or 0)
            for r in plan if r["Machine"] == m_late
        )
        free_m    = max(0.0, AVAILABLE_HOURS - used_m)

        can_push = (
            push_qty <= headroom
            and push_hrs <= free_m + 0.001
            and r_late > 0
        )

        if can_push:
            row_late["Run_Hours"]      = round(float(row_late.get("Run_Hours") or 0) + push_hrs, 3)
            row_late["Production_Qty"] = round(produced_l + push_qty, 0)
            row_late["Total_Hrs_Used"] = round(
                float(row_late.get("Changeover_Hrs") or 0) + float(row_late["Run_Hours"]), 3)
            row_late["Stagger_Adjusted"] = (
                f"CO-stagger PUSH +{round(push_hrs*60,1)}min "
                f"gap={round(gap*60,1)}min need={round(required_gap*60,0):.0f}min")
            current_inventory[p_late] = round(
                current_inventory.get(p_late, 0) + push_qty, 0)
            adjustments += 1
            continue

        row_early = _last_row_before_co(ev_early, plan)
        p_early   = row_early["Part"]
        r_early   = rate.get(p_early, 1)
        m_early   = ev_early["machine"]
        daily_e   = indent_daily.get(p_early, 0)
        inv_e     = current_inventory.get(p_early, 0)
        produced_e = float(row_early.get("Production_Qty") or 0)

        min_qty_e  = max(0.0, daily_e - inv_e)
        max_pull_qty = max(0.0, produced_e - min_qty_e)
        pull_hrs   = shortfall_hrs
        pull_qty   = round(pull_hrs * r_early, 0)

        can_pull = (
            pull_qty <= max_pull_qty
            and r_early > 0
            and produced_e - pull_qty >= MIN_RUN_HOURS * r_early
        )

        if can_pull:
            row_early["Run_Hours"]      = round(
                float(row_early.get("Run_Hours") or 0) - pull_hrs, 3)
            row_early["Production_Qty"] = round(produced_e - pull_qty, 0)
            row_early["Total_Hrs_Used"] = round(
                float(row_early.get("Changeover_Hrs") or 0) + float(row_early["Run_Hours"]), 3)
            row_early["Stagger_Adjusted"] = (
                f"CO-stagger PULL -{round(pull_hrs*60,1)}min "
                f"gap={round(gap*60,1)}min need={round(required_gap*60,0):.0f}min")
            current_inventory[p_early] = round(
                current_inventory.get(p_early, 0) - pull_qty, 0)
            adjustments += 1
            continue

        print(f"    [CO-STAGGER SKIP]  {ev_early['machine']} / {ev_late['machine']}  "
              f"gap={round(gap*60,1)}min  need={round(required_gap*60,0):.0f}min  unresolvable")
        ev_late["_ft"] = ev_early["_ft"] + MIN_CO_GAP_HRS
        break

    return adjustments


def stagger_changeovers_serial_queue(plan, machines):
    print(f"\n  Tool-Changer Serial Queue Scheduler")
    print(f"  Guarantee: strict serial — no two COs can overlap by construction")

    events = _collect_co_events(plan, machines)

    if not events:
        print(f"  No changeovers in plan — tool changer idle  ✓")
        return

    events.sort(key=lambda e: e["natural_start"])

    print(f"  {len(events)} CO events queued across "
          f"{len({e['machine'] for e in events})} machines")
    print()
    print(f"  {'#':<4} {'Machine':<18} {'Part Before':<22} {'Part After':<22} "
          f"{'Dur':>5} {'Natural':>8} {'Actual':>8} {'Wait':>7} {'Fill':>6}")
    print(f"  {'─'*4} {'─'*18} {'─'*22} {'─'*22} "
          f"{'─'*5} {'─'*8} {'─'*8} {'─'*7} {'─'*6}")

    tool_changer_free_at = 0.0
    total_extra_pcs      = 0
    total_wait_min       = 0.0

    for idx, ev in enumerate(events, 1):
        natural_start = _recompute_natural_start(ev, plan)
        co_h          = ev["co_duration"]
        actual_start  = max(natural_start, tool_changer_free_at)
        wait_hrs      = round(actual_start - natural_start, 4)
        tool_changer_free_at = actual_start + co_h

        extra_pcs = 0
        if wait_hrs > 0.001:
            _, extra_pcs = _extend_row_before(ev, wait_hrs, plan)
            total_extra_pcs += extra_pcs
            total_wait_min  += wait_hrs * 60

        ev["actual_start"] = actual_start
        ev["wait_hrs"]     = wait_hrs

        wait_str  = f"+{round(wait_hrs*60,1)}m" if wait_hrs > 0.001 else "none"
        fill_str  = f"+{extra_pcs:.0f}" if extra_pcs > 0 else "—"

        print(f"  {idx:<4} {ev['machine']:<18} {ev['part_before']:<22} "
              f"{ev['part_after']:<22} "
              f"{round(co_h*60,1):>4.0f}m "
              f"{_fmt_h(natural_start):>8} "
              f"{_fmt_h(actual_start):>8} "
              f"{wait_str:>7} "
              f"{fill_str:>6}")

    n_waited = sum(1 for e in events if e.get("wait_hrs", 0) > 0.001)
    print(f"\n  Queue complete. Tool changer free at: {_fmt_h(tool_changer_free_at)}")
    print(f"  Events that had to wait : {n_waited} / {len(events)}")
    print(f"  Total wait time filled  : {round(total_wait_min, 1)} min")
    print(f"  Extra pieces produced   : {total_extra_pcs:,.0f}")
    print(f"  Overlap guarantee       : ABSOLUTE — serial queue")


def stagger_changeovers(plan, machines):
    stagger_changeovers_serial_queue(plan, machines)



def _add_part_to_machine(p, m, run_hrs, co_hrs, machine_hours,
                          machine_last_part, current_inventory,
                          already_planned, plan, priority_scores,
                          type_label, role_label):
    r_val = rate.get(p, 1)
    qty   = round(run_hrs * r_val, 0)

    machine_hours[m]     = round(machine_hours.get(m, 0) + co_hrs + run_hrs, 4)
    current_inventory[p] = round(current_inventory.get(p, 0) + qty, 0)
    machine_last_part[m] = p
    already_planned.add(p)

    plan.append({
        "Part":             p,
        "Category":         part_category.get(p, "Stranger"),
        "Machine":          m,
        "Run_Hours":        round(run_hrs, 3),
        "Changeover_Hrs":   round(co_hrs, 3),
        "Total_Hrs_Used":   round(co_hrs + run_hrs, 3),
        "Rate_Per_Hour":    round(r_val, 2),
        "Production_Qty":   qty,
        "Monthly_Indent":   round(indent_monthly.get(p, 0), 0),
        "Daily_Indent":     round(indent_daily.get(p, 0), 2),
        "Today_Target":     round(today_target_qty.get(p, 0), 0),
        "Changeover":       "No" if co_hrs == 0 else "Yes",
        "Type":             type_label,
        "Role":             role_label,
        "Tools_Available":  tools_available.get(p, 1),
        "Tools_Used":       1,
        "Runner_Lock":      "No",
        "Priority_Score":   round(priority_scores.get(p, 0), 2),
        "Phase":            1,
        "Stagger_Adjusted": "No",
    })
    return qty


def _co_hrs_for(p, m, machine_last_part):
    last = machine_last_part.get(m)
    return 0.0 if (last is None or last == p) else \
           vt_changeover.get(m, DEFAULT_CHANGEOVER_HRS)


def utilization_enforcer(plan, machine_hours, machine_last_part,
                          all_parts, already_planned,
                          current_inventory, scenario_id,
                          priority_scores):

    print(f"\n  22H UTILIZATION ENFORCER  (target ≥{UTIL_TARGET_PCT}%)")
    print(f"  V9: OPD cap = min(scenario, {TARGET_DAYS}) days  |  "
          f"Parts at {TARGET_DAYS}+ days are skipped")
    micro_idle_log = []

    all_skipped = [
        p for p in all_parts
        if should_skip(p)[0]
        and rate.get(p, 0) > 0
        and indent_monthly.get(p, 0) > 0
    ]

    machines_by_util = sorted(vt_machines, key=lambda m: machine_hours.get(m, 0))

    for m in machines_by_util:
        remaining = round(AVAILABLE_HOURS - machine_hours.get(m, 0), 4)
        if remaining < 0.05:
            continue

        # ── STEP 1: Extend existing parts on this machine ────
        parts_on_machine = list({row["Part"] for row in plan if row["Machine"] == m})
        for p in sorted(parts_on_machine,
                        key=lambda x: priority_scores.get(x, 0), reverse=True):
            if remaining < 0.05:
                break
            daily_p  = indent_daily.get(p, 0)
            r_val    = rate.get(p, 1)
            inv_now  = current_inventory.get(p, 0)
            # V9: cap at TARGET_DAYS ceiling
            headroom = max(0.0, opd_cap(scenario_id) * daily_p - inv_now)
            ext_hrs  = min(remaining, headroom / r_val if r_val > 0 else 0)
            if ext_hrs < 0.05:
                continue
            extra_qty = round(ext_hrs * r_val, 0)
            for row in plan:
                if row["Part"] == p and row["Machine"] == m:
                    row["Run_Hours"]      = round(float(row.get("Run_Hours", 0)) + ext_hrs, 3)
                    row["Production_Qty"] = round(float(row.get("Production_Qty", 0)) + extra_qty, 0)
                    row["Total_Hrs_Used"] = round(
                        float(row.get("Changeover_Hrs", 0)) + float(row["Run_Hours"]), 3)
                    row["Type"] = str(row.get("Type", "Primary")) + "+Extended"
                    break
            machine_hours[m]     = round(machine_hours.get(m, 0) + ext_hrs, 4)
            current_inventory[p] = round(current_inventory.get(p, 0) + extra_qty, 0)
            remaining            = round(remaining - ext_hrs, 4)
            print(f"    [S1-EXTEND]  {p:28s} on {m:15s}  +{ext_hrs:.2f}h  qty+={extra_qty:.0f}")

        if remaining < MIN_RUN_HOURS:
            continue

        # ── STEP 2: Assign unplanned compatible parts ─────────
        unplanned = [
            p for p in all_parts
            if p not in already_planned
            and m in vt_compat.get(p, [])
            and not should_skip(p)[0]   # V9: skip includes >= 5-day check
            and indent_monthly.get(p, 0) > 0
            and rate.get(p, 0) > 0
        ]

        last_on_m = machine_last_part.get(m)

        def _co_sort_key(p):
            needs_co    = 0 if (last_on_m is None or last_on_m == p) else 1
            inv_now     = current_inventory.get(p, 0)
            is_critical = 1 if inv_now == 0 else 0
            sc          = priority_scores.get(p, 0)
            return (needs_co, -is_critical, -sc)

        unplanned.sort(key=_co_sort_key)

        for p in unplanned:
            if remaining < MIN_RUN_HOURS:
                break
            co_hrs   = _co_hrs_for(p, m, machine_last_part)
            eff_free = round(remaining - co_hrs, 4)
            if eff_free < MIN_RUN_HOURS:
                continue
            daily_p  = indent_daily.get(p, 0)
            r_val    = rate.get(p, 1)
            inv_now  = current_inventory.get(p, 0)
            # V9: cap at TARGET_DAYS
            cap_qty  = opd_cap(scenario_id) * daily_p
            headroom = max(0.0, cap_qty - inv_now)
            if headroom <= 0:
                continue
            shortfall = max(0.0, daily_p - inv_now)
            min_run   = max(MIN_RUN_HOURS, shortfall / r_val if r_val > 0 else MIN_RUN_HOURS)
            run_hrs   = min(eff_free, max(min_run, headroom / r_val if r_val > 0 else eff_free))
            run_hrs   = max(MIN_RUN_HOURS, min(run_hrs, eff_free))

            co_tag = "No CO" if co_hrs == 0 else "CO"
            qty = _add_part_to_machine(
                p, m, run_hrs, co_hrs,
                machine_hours, machine_last_part, current_inventory,
                already_planned, plan, priority_scores,
                "Filler-Unplanned", "Primary")
            remaining = round(remaining - co_hrs - run_hrs, 4)
            print(f"    [S2-UNPLAN]  {p:28s} → {m:15s}  {run_hrs:.2f}h  "
                  f"qty={qty:.0f}  [{co_tag}]")

        if remaining < MIN_RUN_HOURS:
            continue

        # ── STEP 3: Re-run best planned part (spare tool) ─────
        planned_parts_elsewhere = [
            p for p in already_planned
            if m in vt_compat.get(p, [])
            and m not in [row["Machine"] for row in plan if row["Part"] == p]
        ]
        planned_parts_elsewhere = [
            p for p in planned_parts_elsewhere
            if tools_available.get(p, 1) > len({
                row["Machine"] for row in plan if row["Part"] == p
            })
        ]
        planned_parts_elsewhere.sort(key=lambda p: (
            0 if (machine_last_part.get(m) is None or machine_last_part.get(m) == p) else 1,
            1 if current_inventory.get(p, 0) == 0 else 0,
            -priority_scores.get(p, 0)
        ))

        for p in planned_parts_elsewhere:
            if remaining < MIN_RUN_HOURS:
                break
            co_hrs   = _co_hrs_for(p, m, machine_last_part)
            eff_free = round(remaining - co_hrs, 4)
            if eff_free < MIN_RUN_HOURS:
                continue
            daily_p  = indent_daily.get(p, 0)
            r_val    = rate.get(p, 1)
            inv_now  = current_inventory.get(p, 0)
            # V9: cap at TARGET_DAYS
            cap_qty  = opd_cap(scenario_id) * daily_p
            headroom = max(0.0, cap_qty - inv_now)
            if headroom <= 0:
                continue
            shortfall = max(0.0, daily_p - inv_now)
            min_run   = max(MIN_RUN_HOURS, shortfall / r_val if r_val > 0 else MIN_RUN_HOURS)
            run_hrs   = min(eff_free, max(min_run, headroom / r_val if r_val > 0 else eff_free))
            run_hrs   = max(MIN_RUN_HOURS, min(run_hrs, eff_free))

            r_val2  = rate.get(p, 1)
            qty     = round(run_hrs * r_val2, 0)
            machine_hours[m]     = round(machine_hours.get(m, 0) + co_hrs + run_hrs, 4)
            current_inventory[p] = round(current_inventory.get(p, 0) + qty, 0)
            machine_last_part[m] = p
            remaining            = round(remaining - co_hrs - run_hrs, 4)

            tools_used_now = len({row["Machine"] for row in plan if row["Part"] == p}) + 1
            plan.append({
                "Part":             p,
                "Category":         part_category.get(p, "Stranger"),
                "Machine":          m,
                "Run_Hours":        round(run_hrs, 3),
                "Changeover_Hrs":   round(co_hrs, 3),
                "Total_Hrs_Used":   round(co_hrs + run_hrs, 3),
                "Rate_Per_Hour":    round(r_val2, 2),
                "Production_Qty":   qty,
                "Monthly_Indent":   round(indent_monthly.get(p, 0), 0),
                "Daily_Indent":     round(daily_p, 2),
                "Today_Target":     round(today_target_qty.get(p, 0), 0),
                "Changeover":       "No" if co_hrs == 0 else "Yes",
                "Type":             "Re-run (spare tool)",
                "Role":             f"Tool-Expansion (tool {tools_used_now})",
                "Tools_Available":  tools_available.get(p, 1),
                "Tools_Used":       tools_used_now,
                "Runner_Lock":      "No",
                "Priority_Score":   round(priority_scores.get(p, 0), 2),
                "Phase":            3,
                "Stagger_Adjusted": "No",
            })
            print(f"    [S3-RERUN]   {p:28s} → {m:15s}  {run_hrs:.2f}h  "
                  f"qty={qty:.0f}  tool {tools_used_now}/{tools_available.get(p,1)}")
            break

        if remaining < MIN_RUN_HOURS:
            continue

        # ── STEP 4: Assign a skipped part (last resort) ───────
        skipped_candidates = [
            p for p in all_skipped
            if m in vt_compat.get(p, [])
            and rate.get(p, 0) > 0
        ]
        skipped_candidates.sort(key=lambda p: (
            0 if (machine_last_part.get(m) is None or machine_last_part.get(m) == p) else 1,
            1 if current_inventory.get(p, 0) == 0 else 0,
            -indent_daily.get(p, 0)
        ))

        for p in skipped_candidates:
            if remaining < MIN_RUN_HOURS:
                break
            co_hrs   = _co_hrs_for(p, m, machine_last_part)
            eff_free = round(remaining - co_hrs, 4)
            if eff_free < MIN_RUN_HOURS:
                continue
            r_val   = rate.get(p, 1)
            run_hrs = min(eff_free, max(MIN_RUN_HOURS,
                          indent_monthly.get(p, 0) / r_val if r_val > 0 else eff_free))
            run_hrs = max(MIN_RUN_HOURS, min(run_hrs, eff_free))

            qty = _add_part_to_machine(
                p, m, run_hrs, co_hrs,
                machine_hours, machine_last_part, current_inventory,
                already_planned, plan, priority_scores,
                "Filler-Skipped (last resort)", "Primary")
            remaining = round(remaining - co_hrs - run_hrs, 4)
            skip_reason = should_skip(p)[1]
            print(f"    [S4-SKIPPED] {p:28s} → {m:15s}  {run_hrs:.2f}h  "
                  f"qty={qty:.0f}  [{skip_reason[:35]}]")
            break

        # ── STEP 5: Log micro-idle ────────────────────────────
        final_remaining = round(AVAILABLE_HOURS - machine_hours.get(m, 0), 4)
        if final_remaining >= 0.25:
            util_final = round((1 - final_remaining / AVAILABLE_HOURS) * 100, 1)
            if util_final < UTIL_TARGET_PCT:
                micro_idle_log.append({
                    "Machine":         m,
                    "Idle_Hrs":        round(final_remaining, 3),
                    "Utilization_Pct": util_final,
                    "Note":            "All options exhausted / all parts at 5-day ceiling",
                })
                print(f"    [⚠ IDLE]     {m:15s}  {final_remaining:.2f}h idle "
                      f"({util_final}%) — all options exhausted")

    return micro_idle_log



def build_multi_machine_view(plan):
    if not plan:
        return pd.DataFrame()

    from collections import defaultdict
    part_rows = defaultdict(list)
    for row in plan:
        part_rows[row["Part"]].append(row)

    multi = {p: rows for p, rows in part_rows.items() if len(rows) > 1}

    if not multi:
        return pd.DataFrame()

    output_rows = []
    for part, rows in sorted(multi.items(),
                              key=lambda x: -sum(r["Production_Qty"] for r in x[1])):
        daily     = indent_daily.get(part, 0)
        total_qty = sum(float(r["Production_Qty"]) for r in rows)

        for row in rows:
            output_rows.append({
                "Part":                   part,
                "Category":               part_category.get(part, "Stranger"),
                "Tools_Available":        tools_available.get(part, 1),
                "Machines_Used":          len(rows),
                "Machine":                row["Machine"],
                "Role":                   row.get("Role", "Primary"),
                "Run_Hours":              round(float(row["Run_Hours"]), 2),
                "Changeover_Hrs":         round(float(row.get("Changeover_Hrs", 0)), 2),
                "Production_Qty":         round(float(row["Production_Qty"]), 0),
                "Daily_Indent":           round(daily, 2),
                "Total_Qty_All_Machines": round(total_qty, 0),
                "Type":                   row.get("Type", "—"),
            })

        output_rows.append({
            "Part":                   f"  ↳ TOTAL — {part}",
            "Category":               "—",
            "Tools_Available":        tools_available.get(part, 1),
            "Machines_Used":          len(rows),
            "Machine":                f"{len(rows)} machines",
            "Role":                   "TOTAL",
            "Run_Hours":              round(sum(float(r["Run_Hours"]) for r in rows), 2),
            "Changeover_Hrs":         round(sum(float(r.get("Changeover_Hrs",0)) for r in rows), 2),
            "Production_Qty":         round(total_qty, 0),
            "Daily_Indent":           round(daily, 2),
            "Total_Qty_All_Machines": round(total_qty, 0),
            "Type":                   "—",
        })
        output_rows.append({k: "" for k in output_rows[-1].keys()})

    return pd.DataFrame(output_rows)



def build_production_vs_indent(plan, all_parts):
    if not plan:
        return pd.DataFrame()

    from collections import defaultdict
    part_qty      = defaultdict(float)
    part_machines = defaultdict(list)

    for row in plan:
        p = row["Part"]
        part_qty[p]      += float(row.get("Production_Qty", 0))
        part_machines[p].append(row["Machine"])

    rows = []
    for p in sorted(part_qty.keys()):
        daily    = indent_daily.get(p, 0)
        monthly  = indent_monthly.get(p, 0)
        inv_b    = inventory.get(p, 0)
        produced = round(part_qty[p], 0)
        inv_after = round(inv_b + produced, 0)
        gap      = round(produced - daily, 0)
        gap_dir  = "OVER" if gap > 0 else ("UNDER" if gap < 0 else "MET")
        extra_days = round(gap / daily, 2) if daily > 0 and gap > 0 else 0.0
        machines_str = ", ".join(dict.fromkeys(part_machines[p]))

        rows.append({
            "Part":               p,
            "Category":           part_category.get(p, "Stranger"),
            "Tools_Available":    tools_available.get(p, 1),
            "Machines":           machines_str,
            "Machines_Count":     len(set(part_machines[p])),
            "Total_Qty_Produced": produced,
            "Daily_Indent":       round(daily, 2),
            "Monthly_Indent":     round(monthly, 0),
            "Gap_vs_Daily":       gap,
            "Gap_Direction":      gap_dir,
            "Extra_Days_Stock":   extra_days,
            "Inventory_Before":   round(inv_b, 0),
            "Inventory_After":    inv_after,
            "Days_Coverage_After":round(inv_after / daily, 2) if daily > 0 else 0,
        })

    order_map = {"UNDER": 0, "MET": 1, "OVER": 2}
    df = pd.DataFrame(rows)
    if not df.empty:
        df["_sort"] = df["Gap_Direction"].map(order_map)
        df = df.sort_values(["_sort", "Gap_vs_Daily"]).drop(columns=["_sort"])
        df = df.reset_index(drop=True)
    return df



def build_inventory_target_sheet(plan, all_parts, scenario_id):
    from collections import defaultdict
    part_produced = defaultdict(float)
    for row in plan:
        part_produced[row["Part"]] += float(row.get("Production_Qty", 0))

    rows = []
    for p in sorted(all_parts):
        daily    = indent_daily.get(p, 0)
        monthly  = indent_monthly.get(p, 0)
        inv_b    = inventory.get(p, 0)
        produced = round(part_produced.get(p, 0), 0)
        inv_after = round(inv_b + produced, 0)

        days_before = round(inv_b   / daily, 2) if daily > 0 else 0
        days_after  = round(inv_after / daily, 2) if daily > 0 else 0

        target_qty  = round(TARGET_DAYS * daily, 0)
        gap_qty     = round(target_qty - inv_after, 0)   # +ve = below target
        gap_days    = round(gap_qty / daily, 2) if daily > 0 else 0
        gap_days    = max(0, gap_days)

        cap         = opd_cap(scenario_id)
        max_prod    = round(cap * daily, 0)  # max inventory the OPD cap allows today

        # Buffer status after today's production
        if inv_after == 0:
            status = "CRITICAL"
        elif days_after < SAFETY_DAYS:
            status = "BELOW_SAFETY"
        elif days_after < TARGET_DAYS:
            status = "BUILDING"
        else:
            status = "AT_TARGET"

        # Estimated days to reach 5-day target
        # Net daily gain = (OPD cap × daily) − daily = (cap − 1) × daily
        net_gain_per_day = (cap - 1) * daily if daily > 0 else 0
        if gap_qty <= 0:
            est_days = "AT TARGET"
        elif net_gain_per_day <= 0:
            est_days = "N/A"
        else:
            est_days = str(math.ceil(gap_qty / net_gain_per_day)) + " days"

        skip, skip_reason = should_skip(p)

        rows.append({
            "Part":                   p,
            "Category":               part_category.get(p, "Stranger"),
            "Tools":                  tools_available.get(p, 1),
            "Monthly_Indent":         round(monthly, 0),
            "Daily_Indent":           round(daily, 2),
            "Target_Qty_5days":       target_qty,
            "Safety_Floor_Qty_3days": round(SAFETY_DAYS * daily, 0),
            "Inv_Before":             round(inv_b, 0),
            "Days_Coverage_Before":   days_before,
            "Produced_Today":         produced,
            "Inv_After":              inv_after,
            "Days_Coverage_After":    days_after,
            "Gap_to_Target_Qty":      max(0, gap_qty),
            "Gap_to_Target_Days":     gap_days,
            "Buffer_Status":          status,
            "OPD_Cap_Today_Days":     cap,
            "Max_Producible_Qty":     max_prod,
            "Est_Days_to_Target":     est_days,
            "Scheduled_Today":        "YES" if produced > 0 else ("SKIPPED — AT TARGET" if inv_b >= target_qty else "NO — no capacity"),
            "Skip_Reason":            skip_reason if skip else "",
        })

    # Sort: CRITICAL first, then BELOW_SAFETY, then BUILDING, then AT_TARGET
    status_order = {"CRITICAL": 0, "BELOW_SAFETY": 1, "BUILDING": 2, "AT_TARGET": 3}
    df = pd.DataFrame(rows)
    if not df.empty:
        df["_sort"] = df["Buffer_Status"].map(status_order)
        df = df.sort_values(["_sort", "Gap_to_Target_Days"], ascending=[True, False])
        df = df.drop(columns=["_sort"]).reset_index(drop=True)
    return df



def compute_indent_horizon(parts):
    rows = []
    for p in parts:
        inv     = inventory.get(p, 0.0)
        monthly = indent_monthly.get(p, 0.0)
        daily   = indent_daily.get(p, 0.0)
        target  = today_target_qty.get(p, 0.0)
        r       = rate.get(p, 1.0)
        skip, skip_reason = should_skip(p)
        indent_hrs = monthly / r if r > 0 else 0.0

        days_cov = inv / daily if daily > 0 else 0

        if inv == 0.0 and monthly > 0:
            status = "ZERO INV — FORCED"
        elif skip and inv >= TARGET_DAYS * daily:
            status = "AT 5-DAY TARGET — SKIP"
        elif skip:
            status = "SKIPPED"
        elif target > 0:
            status = "PRODUCTION NEEDED"
        elif monthly == 0:
            status = "NO INDENT"
        else:
            status = "INV SUFFICIENT"

        rows.append({
            "Part":             p,
            "Tools":            tools_available.get(p, 1),
            "Monthly_Indent":   round(monthly, 0),
            "Indent_Hrs_Total": round(indent_hrs, 2),
            "Working_Days":     WORKING_DAYS,
            "Daily_Indent":     round(daily, 2),
            "Inventory_Now":    round(inv, 0),
            "Days_Coverage":    round(days_cov, 2),
            "Safety_Floor":     SAFETY_DAYS,
            "Target_Ceiling":   TARGET_DAYS,
            "Today_Target_Qty": round(target, 0),
            "Today_Target_Hrs": round(target / r if r > 0 else 0, 2),
            "Rate_Per_Hour":    round(r, 2),
            "Indent_Status":    status,
            "Skip_Reason":      skip_reason,
        })
    return pd.DataFrame(rows)



SPECIALIZED_MACHINE_THRESHOLD = 3

def detect_specialized_machines(all_parts):
    machine_parts = {}
    for m in vt_machines:
        compatible_parts = [
            p for p in all_parts
            if m in vt_compat.get(p, [])
            and not should_skip(p)[0]
            and indent_monthly.get(p, 0) > 0
        ]
        machine_parts[m] = compatible_parts

    specialized_machines = {
        m for m, mparts in machine_parts.items()
        if 0 < len(mparts) <= SPECIALIZED_MACHINE_THRESHOLD
    }
    spec_machine_parts = {m: machine_parts[m] for m in specialized_machines}
    return specialized_machines, spec_machine_parts, machine_parts



def schedule(parts, label=""):

    print(f"\n{'─'*65}")
    print(f"  {label}  |  {len(parts)} parts  |  {len(vt_machines)} machines")
    print(f"  Safety floor: {SAFETY_DAYS} days  |  Target ceiling: {TARGET_DAYS} days")
    print(f"{'─'*65}")

    scenario_id, scenario_desc = classify_scenario(parts)
    print(f"  {scenario_desc}")
    print(f"  Effective OPD cap today: {opd_cap(scenario_id)} days "
          f"(scenario={opd_cap(scenario_id)}, ceiling={TARGET_DAYS})")

    horizon_df = compute_indent_horizon(parts)

    # V9: Active parts — skip if >= TARGET_DAYS (Gate 5 updated in should_skip)
    active_parts = [
        p for p in parts
        if not should_skip(p)[0]
        and indent_monthly.get(p, 0) > 0
    ]

    priority_scores, score_rows = compute_priority_scores(active_parts)
    score_df = pd.DataFrame(score_rows) if score_rows else pd.DataFrame()

    machine_hours     = {m: 0.0 for m in vt_machines}
    machine_last_part = {m: machine_state.get(m) for m in vt_machines}
    current_inventory = inventory.copy()
    plan              = []
    already_planned   = set()
    not_planned       = []
    deferred          = []
    displaced_log     = []

    specialized_machines, spec_machine_parts, machine_part_map = \
        detect_specialized_machines(list(parts))

    print(f"\n  SPECIALIZED MACHINES (≤{SPECIALIZED_MACHINE_THRESHOLD} parts):")
    if specialized_machines:
        for m in sorted(specialized_machines,
                        key=lambda m: machine_part_count.get(m, 99)):
            mparts = spec_machine_parts.get(m, [])
            cnt    = machine_part_count.get(m, "?")
            print(f"    {m:<25} Part_Count={cnt:<4} parts: {', '.join(mparts)}")
    else:
        print(f"    None — all machines have >3 compatible parts")

    

    zero_inv_rr = [
        p for p in active_parts
        if current_inventory.get(p, 0) == 0
        and part_category.get(p, "Stranger") in ("Runner", "Repeater")
        and vt_compat.get(p)
    ]

    if zero_inv_rr:
        print(f"\n  V9 DISPLACEMENT PRE-PASS  ({len(zero_inv_rr)} zero-inv Runner/Repeater parts)")
        print(f"  Rule: displace the part with MOST days-of-stock on a compatible machine")
        print(f"  Safety: will not displace any part below {SAFETY_DAYS}-day safety floor")
        # These are processed AFTER primary scheduling fills machines.
        # Store them for post-primary displacement.
    else:
        print(f"\n  V9 DISPLACEMENT PRE-PASS  — no zero-inv Runner/Repeater parts today  ✓")

    # ── PRIMARY SCHEDULING PASS ───────────────────────────────
    sorted_active = sorted(active_parts,
                           key=lambda p: priority_scores.get(p, 0), reverse=True)

    print(f"\n  PRIMARY SCHEDULING PASS  ({len(sorted_active)} active parts)")
    print(f"  {'Part':<30} {'Score':>6} {'Days':>5} {'Status':<15} {'Machine(s)':<28} "
          f"{'Run':>5} {'Qty':>8}")
    print(f"  {'─'*100}")

    for part in sorted_active:
        inv_now  = current_inventory.get(part, 0)
        daily    = indent_daily.get(part, 0)
        monthly  = indent_monthly.get(part, 0)
        score    = priority_scores.get(part, 0)
        tools    = tools_available.get(part, 1)
        category = part_category.get(part, "Stranger")
        days_cov = inv_now / daily if daily > 0 else 999

        # Buffer status label for console
        if inv_now == 0:
            buf_label = "CRITICAL"
        elif days_cov < SAFETY_DAYS:
            buf_label = "BELOW_SAFETY"
        elif days_cov < TARGET_DAYS:
            buf_label = "BUILDING"
        else:
            buf_label = "AT_TARGET"

        if monthly == 0:
            deferred.append({"Part": part, "Category": category,
                             "Reason": "Monthly indent = 0"})
            print(f"  {part:<30} {score:>6.1f} {days_cov:>5.1f} {'DEFERRED':<15}  —  no indent")
            continue

        if not vt_compat.get(part):
            not_planned.append({
                "Part": part, "Category": category, "Score": score,
                "Daily_Indent": round(daily, 2), "Inventory_Now": round(inv_now, 0),
                "Tools": tools, "Compatible_Machines": "NONE DEFINED",
                "Reason": "Not in VT_Matrix", "Action_Needed": "Add to VT_Matrix",
            })
            print(f"  {part:<30} {score:>6.1f} {days_cov:>5.1f} {buf_label:<15}  ✗ NOT IN MATRIX")
            continue

        new_rows = assign_part(
            part, scenario_id, machine_hours, machine_last_part,
            current_inventory, plan, already_planned, priority_scores)

        if new_rows:
            plan.extend(new_rows)
            machines_used = [r["Machine"] for r in new_rows]
            total_qty     = sum(float(r["Production_Qty"]) for r in new_rows)
            total_run     = sum(float(r["Run_Hours"]) for r in new_rows)
            machines_str  = ", ".join(machines_used)
            flag = " [MULTI]" if len(new_rows) > 1 else ""
            flag += " [ZERO-INV]" if inv_now == 0 else ""
            print(f"  {part:<30} {score:>6.1f} {days_cov:>5.1f} {buf_label:<15}  "
                  f"{machines_str:<28}  {total_run:>5.2f}  {total_qty:>8.0f}  ✓{flag}")
        else:
            not_planned.append({
                "Part":                part,
                "Category":            category,
                "Score":               score,
                "Tools":               tools,
                "Days_Coverage":       round(days_cov, 2),
                "Buffer_Status":       buf_label,
                "Daily_Indent":        round(daily, 2),
                "Inventory_Now":       round(inv_now, 0),
                "Compatible_Machines": ", ".join(vt_compat.get(part, [])),
                "Reason":              "No compatible machine has capacity",
                "Action_Needed":       "Review matrix or add machines",
            })
            print(f"  {part:<30} {score:>6.1f} {days_cov:>5.1f} {buf_label:<15}  ✗ NO CAPACITY")

    # ── V9 POST-PRIMARY DISPLACEMENT ─────────────────────────
    # Now that machines are loaded, attempt displacement for any
    # zero-inv Runner/Repeater that did not get scheduled.
    unscheduled_zero_rr = [
        p for p in zero_inv_rr
        if p not in already_planned
    ]

    if unscheduled_zero_rr:
        print(f"\n  V9 DISPLACEMENT PASS  ({len(unscheduled_zero_rr)} zero-inv R/R unscheduled)")
        for part in unscheduled_zero_rr:
            success = displace_for_zero_inv(
                part, machine_hours, machine_last_part,
                current_inventory, plan, already_planned, priority_scores)
            if success:
                displaced_log.append(part)
                # Remove from not_planned if it was added there
                not_planned[:] = [r for r in not_planned if r.get("Part") != part]
            else:
                print(f"      ↳ DISPLACEMENT FAILED  {part}  "
                      f"— no suitable victim found (all compatible machines have "
                      f"parts below {SAFETY_DAYS}-day safety floor)")
    else:
        print(f"\n  V9 DISPLACEMENT PASS  — no unscheduled zero-inv R/R parts  ✓")

    # 22H Utilization Enforcer
    micro_idle = utilization_enforcer(
        plan, machine_hours, machine_last_part,
        list(parts), already_planned,
        current_inventory, scenario_id, priority_scores)

    # CO stagger
    print(f"\n  CO Quantity Stagger  (min gap = {MIN_CO_GAP_HRS*60:.0f} min)")
    n_adj = stagger_co_by_quantity(
        plan, vt_machines, scenario_id, current_inventory)
    if n_adj == 0:
        print(f"    No adjustments needed  ✓")
    else:
        print(f"    {n_adj} quantity adjustment(s) made to stagger COs")

    stagger_changeovers(plan, vt_machines)

    # ── Build output views ────────────────────────────────────
    multi_machine_df  = build_multi_machine_view(plan)
    prod_vs_indent_df = build_production_vs_indent(plan, list(parts))

    # V9: Inventory target sheet
    inv_target_df = build_inventory_target_sheet(plan, list(parts), scenario_id)

    # Daily indent status
    indent_status_rows = []
    for row in plan:
        p       = row["Part"]
        planned = float(row.get("Production_Qty") or 0)
        daily   = indent_daily.get(p, 0)
        inv_b   = inventory.get(p, 0)
        meets   = planned >= daily
        indent_status_rows.append({
            "Part":               p,
            "Category":           part_category.get(p, "Stranger"),
            "Machine":            row.get("Machine", "—"),
            "Role":               row.get("Role", "Primary"),
            "Priority_Score":     round(row.get("Priority_Score", 0), 2),
            "Buffer_Status":      (
                "CRITICAL"     if inv_b == 0 else
                "BELOW_SAFETY" if (daily > 0 and inv_b / daily < SAFETY_DAYS) else
                "BUILDING"     if (daily > 0 and inv_b / daily < TARGET_DAYS) else
                "AT_TARGET"
            ),
            "Run_Hours":          round(float(row.get("Run_Hours") or 0), 2),
            "Planned_Qty":        round(planned, 0),
            "Daily_Indent":       round(daily, 2),
            "Gap_vs_Daily":       round(daily - planned, 0),
            "Meets_Daily_Indent": "YES ✓" if meets else "NO ✗",
            "Inventory_Before":   round(inv_b, 0),
            "Total_Available":    round(inv_b + planned, 0),
            "Covers_With_Inv":    "YES ✓" if (inv_b + planned) >= daily else "NO ✗",
        })
    indent_status_df = pd.DataFrame(indent_status_rows)
    if not indent_status_df.empty:
        indent_status_df = indent_status_df.sort_values(
            ["Meets_Daily_Indent", "Gap_vs_Daily"],
            ascending=[True, False]).reset_index(drop=True)

    # Inventory health
    inv_rows = []
    for p in parts:
        inv_b    = inventory.get(p, 0)
        daily    = indent_daily.get(p, 0)
        produced = sum(float(r["Production_Qty"]) for r in plan if r["Part"] == p)
        inv_after = inv_b + produced
        days_cov  = inv_after / daily if daily > 0 else 0
        inv_rows.append({
            "Part":            p,
            "Tools":           tools_available.get(p, 1),
            "Rate_Per_Hour":   round(rate.get(p, 0), 2),
            "Monthly_Indent":  round(indent_monthly.get(p, 0), 0),
            "Daily_Indent":    round(daily, 2),
            "Inv_Before":      round(inv_b, 0),
            "Produced_Today":  round(produced, 0),
            "Inv_After_Today": round(inv_after, 0),
            "Days_Coverage":   round(days_cov, 2),
            "Safety_Floor":    SAFETY_DAYS,
            "Target_Ceiling":  TARGET_DAYS,
            "Status":          (
                "AT_TARGET" if days_cov >= TARGET_DAYS else
                "OK"        if days_cov >= SAFETY_DAYS else
                "LOW"       if days_cov >= 1 else
                "CRITICAL"
            ),
        })

    # Machine utilization
    mach_rows = []
    for m in vt_machines:
        used      = machine_hours.get(m, 0)
        parts_run = list({r["Part"] for r in plan if r["Machine"] == m})
        co_count  = sum(1 for r in plan if r["Machine"] == m
                        and r.get("Changeover") == "Yes")
        util_pct  = round(used / AVAILABLE_HOURS * 100, 1)
        mach_rows.append({
            "Machine":              m,
            "Used_Hours":           round(used, 2),
            "Unused_Hours":         round(AVAILABLE_HOURS - used, 2),
            "Utilization_%":        util_pct,
            "Status":               ("FULL"     if used >= AVAILABLE_HOURS - 0.3 else
                                     "GOOD"     if used >= AVAILABLE_HOURS * 0.98 else
                                     "PARTIAL"  if used >= AVAILABLE_HOURS * 0.85 else
                                     "UNDERUSED"),
            "Parts_Planned":        len(parts_run),
            "Changeovers":          co_count,
            "Last_Part_Run":        machine_last_part.get(m) or "—",
            "All_Parts_On_Machine": ", ".join(parts_run) if parts_run else "— idle —",
        })

    micro_df  = pd.DataFrame(micro_idle)   if micro_idle   else pd.DataFrame()
    plan_df   = pd.DataFrame(plan)         if plan         else pd.DataFrame()
    def_df    = pd.DataFrame(deferred)     if deferred     else pd.DataFrame()
    not_df    = pd.DataFrame(not_planned)  if not_planned  else pd.DataFrame()
    mach_df   = pd.DataFrame(mach_rows)
    inv_df    = pd.DataFrame(inv_rows)

    if not plan_df.empty:
        plan_df.insert(0, "Planning_Date",   str(PLANNING_DATE))
        plan_df.insert(1, "Scenario",        scenario_desc)
        plan_df.insert(2, "Working_Days",    WORKING_DAYS)
        plan_df.insert(3, "Safety_Days",     SAFETY_DAYS)
        plan_df.insert(4, "Target_Days",     TARGET_DAYS)
        plan_df.insert(5, "OPD_Cap_Today",   opd_cap(scenario_id))

    # Summary
    n_displaced = len(displaced_log)
    print(f"\n  {'='*65}")
    print(f"  SCHEDULE SUMMARY — {scenario_desc}")
    print(f"    Parts planned          : {len(already_planned)}")
    print(f"    Displaced (zero-inv R/R): {n_displaced}")
    print(f"    Not planned            : {len(not_planned)}")
    print(f"    Deferred               : {len(deferred)}")
    if not mach_df.empty:
        print(f"    Avg utilization        : {mach_df['Utilization_%'].mean():.1f}%")
    if not inv_target_df.empty:
        at_target = (inv_target_df["Buffer_Status"] == "AT_TARGET").sum()
        building  = (inv_target_df["Buffer_Status"] == "BUILDING").sum()
        below_s   = (inv_target_df["Buffer_Status"] == "BELOW_SAFETY").sum()
        critical  = (inv_target_df["Buffer_Status"] == "CRITICAL").sum()
        print(f"\n    Inventory target status (after today):")
        print(f"      AT_TARGET   (≥5 days) : {at_target}")
        print(f"      BUILDING (3–5 days)   : {building}")
        print(f"      BELOW_SAFETY (<3 days): {below_s}")
        print(f"      CRITICAL (0 pcs)      : {critical}")
    print(f"  {'='*65}")

    return (plan_df, def_df, not_df, mach_df, inv_df,
            machine_last_part, horizon_df, indent_status_df,
            score_df, micro_df, multi_machine_df, prod_vs_indent_df,
            inv_target_df)



vt_parts = data_valid[
    data_valid["Material"].isin(vt_matrix["Part"])
]["Material"].unique()

all_vt_parts_raw = list(data["Material"].unique())
matrix_parts     = set(str(p).strip() for p in vt_matrix["Part"] if pd.notna(p))
zero_rate_set    = set(data_zero_rate["Material"].unique())

audit_rows = []
for part in all_vt_parts_raw:
    inv     = inventory.get(part, 0.0)
    r_val   = rate.get(part, None)
    monthly = indent_monthly.get(part, 0.0)
    daily   = indent_daily.get(part, 0.0)
    row_data = data[data["Material"] == part]
    ct_raw   = row_data[vt_col_cycletime].values[0] if len(row_data) else "—"
    cv_raw   = row_data[vt_col_cavity].values[0]    if len(row_data) else "—"
    tools    = tools_available.get(part, 1)
    days_cov = inv / daily if daily > 0 else 0

    if part in zero_rate_set or r_val is None:
        status, gate, reason = "ZERO/MISSING CYCLE TIME", "GATE 1", f"Cycle time={ct_raw}"
    elif part not in matrix_parts:
        status, gate, reason = "NOT IN VT_MATRIX", "GATE 2", "No compatible machine"
    elif monthly == 0:
        status, gate, reason = "ZERO/MISSING INDENT", "GATE 3", "Monthly indent = 0"
    elif daily <= MIN_DAILY_INDENT:
        status, gate, reason = "SKIPPED (LOW INDENT)", "GATE 4a", f"Daily ≤ {MIN_DAILY_INDENT}"
    elif (monthly / r_val if r_val else 0) <= MIN_INDENT_HOURS:
        status, gate, reason = "SKIPPED (TRIVIAL RUN)", "GATE 4b", f"Monthly hrs ≤ {MIN_INDENT_HOURS}h"
    elif daily > 0 and inv >= TARGET_DAYS * daily:
        # V9: Gate 5 updated
        status, gate, reason = f"AT {TARGET_DAYS}-DAY TARGET — SKIP TODAY", "GATE 5", \
            f"Inv ({inv:.0f}) ≥ {TARGET_DAYS}×daily ({TARGET_DAYS*daily:.0f})"
    else:
        status, gate, reason = "ENTERS SCHEDULER", "—", "Passed all gates"

    audit_rows.append({
        "Part":           part,
        "Gate_Failed":    gate,
        "Reason":         reason,
        "Monthly_Indent": round(monthly, 0),
        "Daily_Indent":   round(daily, 2),
        "Inventory":      round(inv, 0),
        "Days_Coverage":  round(days_cov, 2),
        "Safety_Floor":   SAFETY_DAYS,
        "Target_Ceiling": TARGET_DAYS,
        "Tools":          tools,
        "Rate_Per_Hour":  round(r_val, 2) if r_val else "—",
        "Cycle_Time":     ct_raw,
        "Cavity":         cv_raw,
        "Status":         status,
    })

audit_df = pd.DataFrame(audit_rows)
gate_counts = audit_df["Status"].value_counts()
print(f"\n  Part audit ({len(all_vt_parts_raw)} total):")
for status, count in gate_counts.items():
    marker = "✓" if status == "ENTERS SCHEDULER" else "·"
    print(f"    {marker}  {status:<50}: {count:>4}")



(vt_plan, vt_def, vt_not, vt_mach, vt_inv,
 vt_state, vt_horizon, vt_indent_status,
 vt_scores, vt_micro, vt_multi_machine,
 vt_prod_vs_indent, vt_inv_target) = schedule(vt_parts, "VT Machines")

save_machine_state(vt_state)



def build_machine_wise_plan(plan_df):
    if plan_df.empty:
        return pd.DataFrame()

    def sf(val, default=0.0):
        try:
            v = float(val)
            return v if not np.isnan(v) else default
        except (TypeError, ValueError):
            return default

    rows = []
    for m in vt_machines:
        machine_rows = plan_df[plan_df["Machine"] == m].copy()
        if machine_rows.empty:
            continue

        for _, pr in machine_rows.iterrows():
            p     = pr.get("Part", "—")
            co_h  = sf(pr.get("Changeover_Hrs", 0))
            run_h = sf(pr.get("Run_Hours", 0))
            r_val = sf(pr.get("Rate_Per_Hour", 0))
            rows.append({
                "Machine":            m,
                "Part":               p,
                "Category":           part_category.get(p, "Stranger"),
                "Role":               pr.get("Role", "Primary"),
                "Tools_Available":    pr.get("Tools_Available", 1),
                "Priority_Score":     round(sf(pr.get("Priority_Score", 0)), 2),
                "Rate_Per_Hour":      round(r_val, 2),
                "Run_Hours":          round(run_h, 2),
                "Changeover_Hrs":     round(co_h, 3),
                "Changeover_Needed":  pr.get("Changeover", "No") or "No",
                "Production_Qty":     round(sf(pr.get("Production_Qty", 0)), 0),
                "Daily_Indent":       round(sf(pr.get("Daily_Indent", 0)), 2),
                "Today_Target":       round(sf(pr.get("Today_Target", 0)), 0),
                "Monthly_Indent":     round(sf(pr.get("Monthly_Indent", 0)), 0),
                "Type":               pr.get("Type", "Primary") or "Primary",
                "Row_Type":           "Part",
            })

        co_total  = machine_rows["Changeover_Hrs"].apply(lambda x: sf(x, 0)).sum()
        run_total = machine_rows["Run_Hours"].apply(lambda x: sf(x, 0)).sum()
        qty_total = machine_rows["Production_Qty"].apply(lambda x: sf(x, 0)).sum()
        co_count  = int(machine_rows["Changeover"].eq("Yes").sum())
        hrs_total = round(co_total + run_total, 2)

        rows.append({
            "Machine":            m,
            "Part":               f"TOTAL — {m}",
            "Category":           "—",
            "Role":               "—",
            "Tools_Available":    "—",
            "Priority_Score":     "—",
            "Rate_Per_Hour":      "—",
            "Run_Hours":          round(run_total, 2),
            "Changeover_Hrs":     round(co_total, 2),
            "Changeover_Needed":  f"{co_count} changeover(s)",
            "Production_Qty":     round(qty_total, 0),
            "Daily_Indent":       "—",
            "Today_Target":       "—",
            "Monthly_Indent":     "—",
            "Type":               (f"Total {hrs_total}h / {AVAILABLE_HOURS}h  |  "
                                   f"Idle {round(AVAILABLE_HOURS - hrs_total, 2)}h  |  "
                                   f"Util {round(hrs_total / AVAILABLE_HOURS * 100, 1)}%"),
            "Row_Type":           "Summary",
        })
        rows.append({k: "" for k in rows[-1].keys()})

    return pd.DataFrame(rows)


def build_co_queue(plan, machines):
    events = _collect_co_events(plan, machines)
    if not events:
        return pd.DataFrame()

    events.sort(key=lambda e: e["natural_start"])
    rows = []
    tool_changer_free_at = 0.0

    for pos, ev in enumerate(events, 1):
        natural_start = _recompute_natural_start(ev, plan)
        co_h          = ev["co_duration"]
        actual_start  = max(natural_start, tool_changer_free_at)
        wait_min      = round((actual_start - natural_start) * 60, 1)
        tool_changer_free_at = actual_start + co_h

        note = ""
        if wait_min > 0:
            note = (f"Machine may need to wait ~{wait_min}min for tool changer. "
                    f"Continue running previous part until team arrives.")

        rows.append({
            "Queue_Position":  pos,
            "Machine":         ev["machine"],
            "Part_Before":     ev["part_before"],
            "Part_After":      ev["part_after"],
            "CO_Duration_Min": round(co_h * 60, 1),
            "Note":            note if note else "Tool changer available immediately",
        })

    return pd.DataFrame(rows)



from openpyxl import load_workbook
from openpyxl.styles import PatternFill, Font, Alignment
from openpyxl.utils import get_column_letter

HEADER_COLORS = {
    "VT_Plan_By_Machine":      "0D6E6E",
    "VT_CO_Queue":             "375623",
    "VT_Plan":                 "1F4E79",
    "VT_Daily_Indent_Status":  "0F4C2A",
    "VT_Multi_Machine_Parts":  "4A235A",
    "VT_Production_vs_Indent": "154360",
    "VT_Inventory_Target":     "1B4F72",   # NEW V9
    "VT_Priority_Scores":      "2C4770",
    "VT_Machine_Util":         "375623",
    "VT_Not_Planned":          "7B2C2C",
    "VT_Deferred":             "7F6000",
    "VT_Inventory_Health":     "4A235A",
    "VT_Indent_Horizon":       "154360",
    "VT_Part_Audit":           "1C3557",
    "VT_Micro_Idle":           "5C3D2E",
}

STATUS_FILLS = {
    "FULL":         PatternFill("solid", fgColor="C6EFCE"),
    "GOOD":         PatternFill("solid", fgColor="DDEBF7"),
    "PARTIAL":      PatternFill("solid", fgColor="FFEB9C"),
    "UNDERUSED":    PatternFill("solid", fgColor="FFC7CE"),
    "OK":           PatternFill("solid", fgColor="C6EFCE"),
    "LOW":          PatternFill("solid", fgColor="FFEB9C"),
    "CRITICAL":     PatternFill("solid", fgColor="FFC7CE"),
    "AT_TARGET":    PatternFill("solid", fgColor="C6EFCE"),
    "BUILDING":     PatternFill("solid", fgColor="DDEBF7"),
    "BELOW_SAFETY": PatternFill("solid", fgColor="FFEB9C"),
    "YES ✓":        PatternFill("solid", fgColor="C6EFCE"),
    "NO ✗":         PatternFill("solid", fgColor="FFC7CE"),
    "OVER":         PatternFill("solid", fgColor="DDEBF7"),
    "UNDER":        PatternFill("solid", fgColor="FFC7CE"),
    "MET":          PatternFill("solid", fgColor="C6EFCE"),
    "PRODUCTION NEEDED":                     PatternFill("solid", fgColor="FFC7CE"),
    "ZERO INV — FORCED":                     PatternFill("solid", fgColor="FFD7D7"),
    "INV SUFFICIENT":                        PatternFill("solid", fgColor="C6EFCE"),
    "SKIPPED":                               PatternFill("solid", fgColor="EDEDED"),
    f"AT {TARGET_DAYS}-DAY TARGET — SKIP TODAY": PatternFill("solid", fgColor="C6EFCE"),
    "NOT REQUIRED — INV SUFFICIENT":         PatternFill("solid", fgColor="DDEBF7"),
    "SKIPPED (LOW INDENT / TRIVIAL RUN)":    PatternFill("solid", fgColor="EDEDED"),
    "ZERO/MISSING CYCLE TIME":               PatternFill("solid", fgColor="FFC7CE"),
    "NOT IN VT_MATRIX":                      PatternFill("solid", fgColor="FFEB9C"),
    "ZERO/MISSING INDENT":                   PatternFill("solid", fgColor="FFEB9C"),
    "ENTERS SCHEDULER":                      PatternFill("solid", fgColor="C6EFCE"),
}

def style_sheet(ws, header_hex):
    for cell in ws[1]:
        cell.fill      = PatternFill("solid", fgColor=header_hex)
        cell.font      = Font(bold=True, color="FFFFFF", size=11)
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
    ws.row_dimensions[1].height = 32
    for col in ws.columns:
        col_letter = get_column_letter(col[0].column)
        max_len    = max((len(str(c.value)) for c in col if c.value is not None), default=8)
        ws.column_dimensions[col_letter].width = max(10, min(55, max_len + 3))
    headers = [cell.value for cell in ws[1]]
    for col_idx, col_name in enumerate(headers, start=1):
        if col_name and any(x in str(col_name) for x in
                            ["Status", "Indent_Status", "Meets_Daily",
                             "Covers_With", "Gap_Direction", "Buffer_Status",
                             "Scheduled_Today"]):
            for row in ws.iter_rows(min_row=2, min_col=col_idx, max_col=col_idx):
                for cell in row:
                    fill = STATUS_FILLS.get(str(cell.value))
                    if fill:
                        cell.fill = fill
    ws.freeze_panes = "A2"


def style_inv_target_sheet(ws):
    """
    Special styling for VT_Inventory_Target.
    Colour-codes rows by Buffer_Status:
      CRITICAL     → red
      BELOW_SAFETY → amber
      BUILDING     → blue
      AT_TARGET    → green
    Also draws a visual progress bar in the Gap_to_Target_Days column
    using cell background intensity.
    """
    style_sheet(ws, HEADER_COLORS["VT_Inventory_Target"])

    headers    = [c.value for c in ws[1]]
    status_col = headers.index("Buffer_Status") + 1 if "Buffer_Status" in headers else None

    row_fills = {
        "CRITICAL":     PatternFill("solid", fgColor="FFD7D7"),
        "BELOW_SAFETY": PatternFill("solid", fgColor="FFF2CC"),
        "BUILDING":     PatternFill("solid", fgColor="DDEEFF"),
        "AT_TARGET":    PatternFill("solid", fgColor="E2EFDA"),
    }
    row_fonts = {
        "CRITICAL":     Font(bold=True),
        "BELOW_SAFETY": Font(bold=False),
        "BUILDING":     Font(bold=False),
        "AT_TARGET":    Font(bold=False),
    }

    for row in ws.iter_rows(min_row=2):
        if not status_col:
            continue
        status = str(row[status_col - 1].value)
        fill   = row_fills.get(status)
        font   = row_fonts.get(status)
        if fill:
            for cell in row:
                if cell.fill.fill_type == "none" or cell.fill.fgColor.rgb in (
                        "00000000", "FFFFFFFF"):
                    cell.fill = fill
        if font and status == "CRITICAL":
            for cell in row:
                cell.font = font


def style_machine_wise_sheet(ws):
    header_fill  = PatternFill("solid", fgColor="1F4E79")
    summary_fill = PatternFill("solid", fgColor="0D9488")
    part_fills   = [PatternFill("solid", fgColor="EFF6FF"),
                    PatternFill("solid", fgColor="F0FDF4")]
    co_fill      = PatternFill("solid", fgColor="FEF9C3")

    for cell in ws[1]:
        cell.fill      = header_fill
        cell.font      = Font(bold=True, color="FFFFFF", size=11)
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
    ws.row_dimensions[1].height = 30

    headers      = [cell.value for cell in ws[1]]
    row_type_col = headers.index("Row_Type")  + 1 if "Row_Type"  in headers else None
    co_col       = headers.index("Changeover") + 1 if "Changeover" in headers else None
    machine_col  = headers.index("Machine")   + 1 if "Machine"   in headers else None

    machine_color_idx = 0
    current_machine   = None
    for row in ws.iter_rows(min_row=2):
        row_type = row[row_type_col-1].value if row_type_col else ""
        machine  = row[machine_col-1].value  if machine_col  else ""
        if machine and machine != current_machine:
            current_machine   = machine
            machine_color_idx = (machine_color_idx + 1) % 2
        if row_type == "Summary":
            for cell in row:
                cell.fill = summary_fill
                cell.font = Font(bold=True, color="FFFFFF", size=11)
                cell.alignment = Alignment(horizontal="center", vertical="center")
        elif row_type == "Part":
            for cell in row:
                cell.fill      = part_fills[machine_color_idx]
                cell.alignment = Alignment(vertical="center")
            if co_col and row[co_col-1].value == "Yes":
                row[co_col-1].fill = co_fill

    for col in ws.columns:
        col_letter = get_column_letter(col[0].column)
        max_len    = max((len(str(c.value)) for c in col if c.value), default=8)
        ws.column_dimensions[col_letter].width = max(10, min(45, max_len + 3))
    ws.freeze_panes = "B2"


def style_prod_vs_indent_sheet(ws):
    style_sheet(ws, HEADER_COLORS["VT_Production_vs_Indent"])
    headers   = [c.value for c in ws[1]]
    gap_col   = headers.index("Gap_Direction") + 1 if "Gap_Direction" in headers else None

    over_fill  = PatternFill("solid", fgColor="DDEBF7")
    under_fill = PatternFill("solid", fgColor="FFC7CE")
    met_fill   = PatternFill("solid", fgColor="C6EFCE")

    for row in ws.iter_rows(min_row=2):
        if gap_col:
            cell  = row[gap_col - 1]
            value = str(cell.value)
            if value == "OVER":
                cell.fill = over_fill
            elif value == "UNDER":
                cell.fill = under_fill
                for c in row:
                    c.font = Font(bold=True)
            elif value == "MET":
                cell.fill = met_fill


def style_multi_machine_sheet(ws):
    style_sheet(ws, HEADER_COLORS["VT_Multi_Machine_Parts"])
    headers  = [c.value for c in ws[1]]
    role_col = headers.index("Role") + 1 if "Role" in headers else None

    total_fill   = PatternFill("solid", fgColor="0D9488")
    primary_fill = PatternFill("solid", fgColor="EFF6FF")
    expand_fill  = PatternFill("solid", fgColor="FEF9C3")

    for row in ws.iter_rows(min_row=2):
        if not role_col:
            continue
        role = str(row[role_col - 1].value)
        if role == "TOTAL":
            for cell in row:
                cell.fill = total_fill
                cell.font = Font(bold=True, color="FFFFFF", size=11)
        elif "Tool-Expansion" in role:
            for cell in row:
                cell.fill = expand_fill
        elif role == "Primary":
            for cell in row:
                cell.fill = primary_fill


# Write Excel
print(f"\nWriting output → {output_path}")

vt_mw   = build_machine_wise_plan(vt_plan)
vt_co_q = build_co_queue(
    [{k:v for k,v in r.items()} for r in vt_plan.to_dict("records")]
    if not vt_plan.empty else [],
    vt_machines
)

sheets = {
    "VT_Plan_By_Machine":      vt_mw,
    "VT_CO_Queue":             vt_co_q,
    "VT_Plan":                 vt_plan,
    "VT_Inventory_Target":     vt_inv_target,       # NEW V9
    "VT_Multi_Machine_Parts":  vt_multi_machine,
    "VT_Production_vs_Indent": vt_prod_vs_indent,
    "VT_Daily_Indent_Status":  vt_indent_status,
    "VT_Priority_Scores":      vt_scores,
    "VT_Machine_Util":         vt_mach,
    "VT_Not_Planned":          vt_not,
    "VT_Deferred":             vt_def,
    "VT_Inventory_Health":     vt_inv,
    "VT_Indent_Horizon":       vt_horizon,
    "VT_Part_Audit":           audit_df,
}
if not vt_micro.empty:
    sheets["VT_Micro_Idle"] = vt_micro

with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
    for sheet_name, df in sheets.items():
        if df is not None and not df.empty:
            df.to_excel(writer, sheet_name=sheet_name, index=False)

wb = load_workbook(output_path)

if "VT_Plan_By_Machine"  in wb.sheetnames:
    style_machine_wise_sheet(wb["VT_Plan_By_Machine"])
if "VT_Production_vs_Indent" in wb.sheetnames:
    style_prod_vs_indent_sheet(wb["VT_Production_vs_Indent"])
if "VT_Multi_Machine_Parts" in wb.sheetnames:
    style_multi_machine_sheet(wb["VT_Multi_Machine_Parts"])
if "VT_Inventory_Target" in wb.sheetnames:
    style_inv_target_sheet(wb["VT_Inventory_Target"])    # NEW V9

for sheet_name, header_hex in HEADER_COLORS.items():
    if sheet_name in wb.sheetnames and sheet_name not in (
            "VT_Plan_By_Machine", "VT_Production_vs_Indent",
            "VT_Multi_Machine_Parts", "VT_Inventory_Target"):
        style_sheet(wb[sheet_name], header_hex)

# Colour Daily Indent Status YES/NO columns
if "VT_Daily_Indent_Status" in wb.sheetnames:
    ws_is      = wb["VT_Daily_Indent_Status"]
    headers_is = [c.value for c in ws_is[1]]
    meets_col  = (headers_is.index("Meets_Daily_Indent") + 1
                  if "Meets_Daily_Indent" in headers_is else None)
    covers_col = (headers_is.index("Covers_With_Inv") + 1
                  if "Covers_With_Inv" in headers_is else None)
    for row in ws_is.iter_rows(min_row=2):
        if meets_col:
            cell = row[meets_col - 1]
            cell.fill = (PatternFill("solid", fgColor="C6EFCE")
                         if str(cell.value) == "YES ✓"
                         else PatternFill("solid", fgColor="FFC7CE"))
        if covers_col:
            cell = row[covers_col - 1]
            cell.fill = (PatternFill("solid", fgColor="C6EFCE")
                         if str(cell.value) == "YES ✓"
                         else PatternFill("solid", fgColor="FFEB9C"))
        if meets_col and str(row[meets_col - 1].value) == "NO ✗":
            for cell in row:
                cell.font = Font(bold=True)

for name, color in HEADER_COLORS.items():
    if name in wb.sheetnames:
        wb[name].sheet_properties.tabColor = color

wb.save(output_path)
print(f"  Formatting applied  ✓")



print(f"\n{'='*65}")
print(f"  Smart APS V9 Complete  —  {PLANNING_DATE}")
print(f"  5-Day Target Inventory System")
print(f"  Safety floor: {SAFETY_DAYS} days  |  Target ceiling: {TARGET_DAYS} days")
print(f"{'='*65}")

for status, count in audit_df["Status"].value_counts().items():
    marker = "✓" if status == "ENTERS SCHEDULER" else "·"
    print(f"  {marker} {status:<52}: {count:>4}")

print(f"\n  Results:")
print(f"    Planned            : {len(vt_plan):>4} rows")
print(f"    Not planned        : {len(vt_not):>4}")
print(f"    Deferred           : {len(vt_def):>4}")

if not vt_inv_target.empty:
    at_t = (vt_inv_target["Buffer_Status"] == "AT_TARGET").sum()
    bld  = (vt_inv_target["Buffer_Status"] == "BUILDING").sum()
    bls  = (vt_inv_target["Buffer_Status"] == "BELOW_SAFETY").sum()
    crt  = (vt_inv_target["Buffer_Status"] == "CRITICAL").sum()
    print(f"\n  Inventory target status (end of day):")
    print(f"    AT_TARGET   (≥5 days) : {at_t:>4}  — skip tomorrow")
    print(f"    BUILDING  (3–5 days)  : {bld:>4}  — gradual build continues")
    print(f"    BELOW_SAFETY (<3 days): {bls:>4}  — safety buffer being consumed")
    print(f"    CRITICAL  (0 pcs)     : {crt:>4}  — displacement eligible tomorrow")

if not vt_mach.empty:
    print(f"\n  Machine utilization:")
    print(f"    Average     : {vt_mach['Utilization_%'].mean():.1f}%")
    print(f"    UNDERUSED   : {(vt_mach['Status']=='UNDERUSED').sum()} machines")

print(f"\n  Output → {output_path}")
print(f"  State  → {MACHINE_STATE_FILE}")
print(f"\n  UPDATE DAILY: PLANNING_DATE = date(2026, 3, 21)")
print(f"{'='*65}")


  Smart APS V9  —  5-Day Target Inventory System
  Planning date : 2026-03-20
  Indent month  : March 2026
  Working days  : 26  (31 days − 5 Sundays)
  Safety floor  : 3 days  |  Target ceiling : 5 days

Loading data...
  VT parts in sheet       : 158
  Parts with valid rate   : 153
  Machine part counts loaded: 30 machines
  Machine state loaded  (30 machines with history)

  Part audit (158 total):
    ✓  ENTERS SCHEDULER                                  :   76
    ·  SKIPPED (LOW INDENT)                              :   76
    ·  ZERO/MISSING CYCLE TIME                           :    5
    ·  AT 5-DAY TARGET — SKIP TODAY                      :    1

─────────────────────────────────────────────────────────────────
  VT Machines  |  153 parts  |  30 machines
  Safety floor: 3 days  |  Target ceiling: 5 days
─────────────────────────────────────────────────────────────────
  SCENARIO 0 — ALL 76 active parts critical (inv < 1 day)
  Effective OPD cap today: 1.5 days (scenario=1.5, ce

In [10]:
plan_df = pd.DataFrame(plan)
plan_df.to_excel(output_path, sheet_name="VT_Plan_By_Machine", index=False)


NameError: name 'plan' is not defined